# İleri Yapay Zeka (YZ313) – WarmUp Ödev 2
## FedFST'nin Google Colab'da çalıştırılması ve POWER (KDD'25) / FedSPA (CVPR'25) ile karşılaştırılması

| | |
|---|---|
| **Ad Soyad** | *(doldurunuz)* |
| **Öğrenci No** | *(doldurunuz)* |
| **Ders** | İleri Yapay Zeka (YZ313) |

**Özet.** Bu notebook, federe çizge sürekli öğrenmesi (Federated Graph Continual Learning, FGCL) için önerilen FedFST'nin resmi kodunu ([YunQi572/FedFST](https://github.com/YunQi572/FedFST), commit `d83fac0`) Colab'da çalıştırır ve sonuçlarını POWER ([Zhu vd., KDD'25](https://arxiv.org/abs/2411.18919)) ve FedSPA ([Tan vd., CVPR'25](https://github.com/OakleyTan/FedSPA)) ile karşılaştırır. Üç yöntem de beş veri setinde (Cora, CiteSeer, Actor, Squirrel, Minesweeper) aynı Louvain bölümlemesini (3 istemci), aynı sınıf-artımlı görevleri ve aynı eğitim/doğrulama/test maskelerini kullanır. Değerlendirme tek ve ortak bir fonksiyonla yapılır: ortalama doğruluk (AA ↑) ve ortalama unutma (AF ↓). Repoların olduğu gibi çalışmayan kısımları en küçük yamalarla düzeltildi. FedFST'nin yavaş Python döngüleri, sonuçları bit düzeyinde değiştirmeden vektörize edildi. FedSPA'nın DGL'e bağlı mesaj geçirme kısmı saf PyTorch'a taşındı.

> **Çalıştırma:** `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`, ardından `Çalışma zamanı → Tümünü çalıştır`. Drive erişim izni istenir. Sonuçlar, log'lar ve veri cache'i `MyDrive/FedFST_Odev2/` altına yazılır. Bağlantı koparsa notebook'u yeniden çalıştırmak yeterli: biten koşular atlanır ve deney kaldığı yerden devam eder. GPU kotası yoksa her şey otomatik olarak CPU'da çalışır. Hızlı bir deneme için Bölüm 8'deki CONFIG hücresinde `SMOKE = True` yapın.

**İçindekiler**
1. FedFST'nin çalışma mantığı
2. Karşılaştırılan yöntemler: POWER ve FedSPA
3. Kurulum
4. Kaynak kodların indirilmesi
5. Yamalar ve harness dosyaları
6. Eşdeğerlik testleri ve sağlamalar
7. Veri setleri
8. Deneyler
9. Sonuçlar
10. Tartışma
11. Kaynakça

## 1. FedFST'nin çalışma mantığı

### 1.1 Problem: federe çizge sürekli öğrenmesi (FGCL) ve felaket unutma
Bir çizge, Louvain topluluk bölümlemesiyle **K = 3** istemciye dağıtılır. Sınıflar görevlere ayrılır (burada görev başına 2 sınıf, Minesweeper'da 1) ve görevler sırayla gelir. Buna *sınıf-artımlı* senaryo denir. Her görevde istemciler yalnızca o görevin düğümleriyle yerel eğitim yapar, sunucu da modelleri birleştirir. Her görevin sonunda global modelin o ana kadar görülen **tüm** sınıfları ayırt etmesi beklenir. Model yeni sınıflara uyum sağlarken eski sınıflardaki doğruluğunu kaybeder; buna **felaket unutma** (catastrophic forgetting) denir.

Değerlendirme bir doğruluk matrisi $A$ ile yapılır: $A_{t,i}$, görev $t$'nin eğitimi bittiğinde global modelin görev $i$'nin test düğümlerindeki doğruluğudur (istemciler arasında test düğümü sayısıyla ağırlıklı). $T$ görev için FedFST'nin `utils.AA` ve `utils.AF` fonksiyonları şunları hesaplar:

$$\mathrm{AA}=\frac{1}{T}\sum_{i=1}^{T}A_{T,i}\ (\uparrow),\qquad \mathrm{AF}=\frac{1}{T-1}\sum_{i=1}^{T-1}\left(A_{i,i}-A_{T,i}\right)\ (\downarrow)$$

POWER'ın kodundaki `utils.AA` / `utils.AF` aynı tanımları kullanır; POWER'ın teknik raporundaki grafiklerde bunlar **AM** ve **FM** olarak geçer. Tek görevli bir akışta unutma ölçülemez (`AF` bu durumda −1 döndürür).

### 1.2 Spektral bakış: yüksek ve düşük frekans
Kombinatoryal Laplasyen $L = D - A$ ile bir düğüm sinyali $x$ için $x^\top L x=\sum_{(i,j)\in E}(x_i-x_j)^2$ (yönsüz çizgede) komşular arasındaki farkları toplar. Rayleigh oranı $x^\top L x / x^\top x$ küçükse sinyal komşular arasında **pürüzsüzdür**. Bu düşük frekanstır: komşular benzerdir (homofili) ve sınıflar arası ortak, genel anlam bu bileşende taşınır. Oran büyükse sinyal kenarlar boyunca **hızla değişir**. Bu yüksek frekanstır: komşular arasında tutarsızlık (heterofili) ve ayırt edici ayrıntı vardır. FedFST'nin README'si unutmayı bu iki bileşen üzerinden tanımlar ([README](https://github.com/YunQi572/FedFST)):
* **Yüksek frekanslı tutarsızlık unutması:** yeni görevlerle ortaya çıkan düğüm tutarsızlıkları mesaj geçirmeyi bozar ve ayırt edici bilgiyi siler.
* **Düşük frekanslı tutarlılık unutması:** yeni görevlere aşırı uyum, global anlamsal tutarlılığı sulandırır.

README'ye göre FedFST iki bileşenden oluşur: **HHKR** (Historical High-Frequency Knowledge Restoration) geçmişteki yüksek frekanslı tutarsızlık bilgisini yeniden kurup korur; **HLST** (Historical Low-Frequency Semantic Transfer) düşük frekanslı tutarlılığı spektral damıtmayla korur.

Kodda yüksek frekans ölçüsü `utils.get_Shigh` fonksiyonunda hesaplanır. Önce rastgele seçilen özniteliklerin %20'si için $x^\top L x / x^\top x$ oranlarının ortalaması alınır ($S^{x}_{high}$). Ardından komşuluk matrisinin her sütunu bir sinyal gibi ele alınıp aynı oranın ortalaması alınır ($S^{A}_{high}$). Sonuç $S_{high}=0.5\,S^{x}_{high}+0.5\,S^{A}_{high}$ olur.

### 1.3 Bileşenlerin koda karşılığı
Kodda "HHKR" ve "HLST" adları geçmiyor. Aşağıdaki eşleştirme, README'deki tanımlarla fonksiyonların yaptığı işi karşılaştırarak yaptığım bir okuma. Bu okumayı destekleyen bir ipucu var: KD kaybı kodda `Low Freq Loss` olarak adlandırılıyor (`algorithm/Ours.py`, `KD_train`). Hiperparametreler `args.py`'deki varsayılanlardır.

| Adım | Kod | Kodda gördüğüm kadarıyla ne yapıyor |
|---|---|---|
| Yerel eğitim | `OursClient.train` | Global ağırlıklarla başlar; yalnızca mevcut görevin eğitim düğümlerinde çapraz entropiyle 3 epoch eğitir (her round yeni Adam, lr 0.005, wd 5e-4). Yerel tarafta unutmaya karşı ek bir kayıp terimi yok. |
| Birleştirme | `OursServer.aggregate` | FedAvg; istemci ağırlıkları görevdeki düğüm sayısıyla orantılı. |
| HHKR-1: koşullu jeneratör | `OursClient.feature_gen_train`, `utils.Generator` | Gürültü ve sınıf gömmesinden düğüm özniteliği üreten bir MLP. Kaybın iki terimi var: (i) bir önceki görevin global modeli (`last_global_model`) üretilen düğümleri koşul sınıfına atamalı (CE); (ii) üretilen özniteliğin softmax dağılımı, aynı sınıftan rastgele seçilen **gerçek bir eski-görev düğümünün** dağılımına KL ile yaklaşmalı (ağırlık 100). İstemci başına 2 jeneratör round'u × 200 epoch çalışır; sunucu jeneratörleri öğrenilmiş düğüm sayısıyla ağırlıklı ortalar (`feature_gen_aggregate`). |
| HHKR-2: homofili eşleme | `get_learned_Homophily_Level`, `utils.reduce_homophilic_edges` / `reduce_heterophilic_edges` | Üretilen düğümler rastgele bir çizgeyle bağlanır (düğüm başına 2 kenar). Çizgenin homofilisi, istemcinin eski görevlerdeki homofili oranına (eğitim düğümleri arası kenarlarda) ±0.05 yaklaşana kadar her adımda homofilik ya da heterofilik kenarların %5'i silinir (en çok 60 adım). |
| HHKR-3: sunucuda $S_{high}$ eşleme | `send_feature_gen` (`get_Shigh`), `OursServer.synthesis_data` | İstemciler eski-görev alt-çizgilerinin $S_{high}$ değerini gönderir; sunucu bunların ağırlıklı ortalamasını alır. Global jeneratörle eski sınıfların her birinden 300 düğüm üretir ve rastgele çizgeden kenar silerek sentetik çizgenin $S_{high}$'ını bu değere ±0.05 yaklaştırır. Böylece sentetik tekrar çizgesi geçmişin yüksek frekans (tutarsızlık) düzeyini taşır. |
| HLST: düşük frekanslı damıtma | `OursServer.KD_train`, `utils.get_khop_feature`, `utils.edge_distribution_low` | Öğrenci global model, öğretmen önceki görevin global modeli. Kayıp = 0.5·CE (sentetik etiketler) + 1·KL. KL, iki modelin softmax çıktılarının **2 adımlı komşu ortalamasıyla yumuşatılmış** hâlleri (alçak geçiren filtre) arasında, kenar uçları üzerinden hesaplanır. 200 epoch, Adam lr 0.002. |
| Model seçimi | `evaluate_on_clients`, `evaluate_for_kd` | Her KD epoch'unda istemcilerin **doğrulama** maskeleriyle AA−AF hesaplanır. Eski görevlerin doğruluğu mevcut görevi geçince eğitim erken durur ve son iki kayıttan iyisi seçilir; erken durmazsa en iyi kayıt kullanılır. Test maskesi seçimde kullanılmıyor. |

### 1.4 Akış şeması
```
Görev t = 0, 1, …, T−1
│
├─ 10 round ─┬─ istemci k: GAT'ı global ağırlıklarla başlat, yalnızca görev t verisiyle 3 epoch eğit
│            └─ sunucu: FedAvg (görevdeki düğüm sayısıyla ağırlıklı)
│
├─ t > 0 ise ─┬─ 2 jeneratör round'u
│             │    istemci k: koşullu jeneratör, CE(önceki global model) + 100·KL(gerçek eski düğüm)
│             │               sentetik çizgenin homofilisi ← eski görevlerin homofilisi       [HHKR]
│             │    istemci k → sunucu: jeneratör ağırlıkları + eski alt-çizginin S_high değeri
│             │    sunucu: jeneratörleri ağırlıklı ortala
│             ├─ synthesis_data: eski sınıflardan 300'er düğüm üret,
│             │                  sentetik çizgenin S_high'ı ← global S_high                    [HHKR]
│             └─ KD_train: global model ← 0.5·CE + KL(2-adım yumuşatılmış öğrenci ‖ öğretmen),
│                          doğrulama AA−AF ile model seçimi / erken durma                   [HLST]
│
└─ önceki global model ← global model;  değerlendirme: A[t, 0..t]
```

### 1.5 Kodu okurken dikkat ettiğim noktalar
* **Geçmiş veriye erişim.** Görev $t$'de istemciler önceki görevlerin gerçek verisini kullanıyor: jeneratörün KL terimi gerçek düğüm özniteliklerini, homofili ve $S_{high}$ hesapları da etiketleri ve kenarları kullanıyor (`get_learned_graph`, `get_subgraph`, `get_learned_Homophily_Level`). KD'deki model seçimi de önceki görevlerin doğrulama etiketlerine bakıyor. Yani istemciler geçmiş görevlerin yerel verisini saklıyor. POWER ise sınıf başına yalnızca 1 düğüm saklıyor (`num_samples_per_class = 1`). Karşılaştırmayı yorumlarken bu fark akılda tutulmalı.
* **KD kaybının biçimi.** `KD_train`'de değişken adı `student_log_probs` olsa da değer `F.softmax(...)`, yani olasılık. `edge_distribution_low` bu yumuşatılmış olasılıkları `nn.KLDivLoss(log_target=True)`'ya log-olasılık gibi veriyor.
* **Tekrarlanabilirlik.** `main.py`, istemcilerin jeneratör eğitimini `ThreadPoolExecutor` ile eşzamanlı çalıştırıyor. Üç iş parçacığı aynı küresel RNG'yi paylaştığı için aynı seed'le bile sonuç değişiyor (Bölüm 6.3'te iki ayrı koşu). Harness bu adımı aynı sırayla ardışık çalıştırıyor.

## 2. Karşılaştırılan yöntemler

### 2.1 POWER (Zhu, Hu ve Wu, KDD'25)
POWER, FGCL problemini ilk tanımlayan çalışma. Makale iki zorluğu adlandırıyor ([arXiv:2411.18919](https://arxiv.org/abs/2411.18919)):
* **Yerel çizge unutması (LGF):** istemci modeli yeni göreve uyum sağlarken eski bilgiyi kaybeder.
* **Global uzmanlık çatışması (GEC):** istemcilerin tutarsız uzmanlıkları sunucuda birleştirilirken çatışır ve global model hem eski hem yeni görevlerde kötüleşir.

LGF için yerel-global kapsamı en geniş deneyim düğümleri saklanıp tekrar oynatılır. GEC için sunucuda **sözde prototip yeniden oluşturma** ve **yörünge-farkında bilgi aktarımı** kullanılır. Resmi kodda ([Zenodo](https://zenodo.org/records/15487655): `flcore/Ours.py`, `utils.py`) gördüğüm karşılıklar şunlar:

| Bileşen | Kod | Ne yapıyor |
|---|---|---|
| Deneyim düğümleri (LGF) | `OursClient.task_done` → `utils.update_converage_max` | Görev bitince her sınıftan 1 düğüm seçer (kapsama maksimizasyonu). Uzaklıklar, yerel ve global modelin gömmelerinin α = 0.5 karışımında hesaplanır; yarıçap, ortalama uzaklığın 0.1 katıdır. |
| Yerel eğitim | `OursClient.execute` | Kayıp = β·CE(mevcut görev) + (1−β)·CE(tampondaki düğümler, kenarsız). β'nın varsayılanı 0.01. |
| Prototip gradyanı | `OursClient.encode_gradient` | Her görevin ilk round'unda her sınıfın öznitelik ortalaması (prototip), tüm istemcilerde ortak ve hiç eğitilmeyen 4 katmanlı bir MLP'den (`GEModel`) geçirilir. CE kaybının bu MLP'nin parametrelerine göre gradyanı sunucuya gönderilir. |
| Sözde prototip (GEC) | `OursServer.execute`, `gradient2label` | Sunucu, sınıf etiketini son katman bias gradyanının en küçük elemanından çıkarır. Rastgele bir girdiyi, gradyanı gelen gradyanla eşleşene kadar L-BFGS ile optimize eder (300 iterasyon, lr 1.0). Bu, DLG'ye benzer bir gradyan ters çevirmedir. Sonuç, sınıfın prototip tamponuna eklenir. |
| Yörünge-farkında aktarım (GEC) | `OursServer.execute` (son kısım) | FedAvg'den sonra global model, tüm sınıfların sözde prototipleri üzerinde her istemcinin yerel modelinin çıktılarına (L1) yaklaştırılır (3 epoch, Adam lr 0.01). İstemci-sınıf ağırlıkları sınıf örnek sayılarından hesaplanır; geçmiş görevlerin katkısı `decay = 0.3` ile azaltılır. |

Kodda gördüğüm ve karşılaştırmayı etkileyen ayrıntılar:
* **β = 0.01.** `config.py`'de "explored" başlığı altında. Resmi eğitim betiği (`optuna_search.py`) β, `decay`, `num_recon_nodes` ve `num_epoch_g` için 300 denemelik bir Optuna araması yapıyor. Teknik raporda da β "keşif" hiperparametreleri arasında (Ek A.3). Ödev gereği arama yapmadım; kodun varsayılanlarını kullandım.
* **Görev sırası.** POWER'ın veri katmanı (`data.py`, `class_incremental`) görevlerin sırasını **her istemci için ayrı ayrı karıştırıyor** (`np.random.shuffle(tasks)`). Yani orijinal senaryoda istemciler aynı anda farklı sınıfları öğreniyor. Ortak protokolde ise, FedFST'deki gibi, tüm istemciler aynı sırayı izliyor.
* **Değerlendirme.** `utils.start()` varsayılan olarak (`global_eval = False`) her istemcinin **yerel** modelini kendi verisinde değerlendiriyor. Ortak protokolde global model değerlendiriliyor.
* **KNN çizgesi.** Teknik rapor KNN parametresini 1 olarak veriyor (Ek A.3). Kodda `construct_knn_graph` tanımlı ama kullanılmıyor; sunucu sözde prototipler için kenarsız bir çizge (`construct_self_loop_graph`) kuruyor.
* **Veri setleri ve bağımlılıklar.** POWER'ın veri katmanı (`data.py`) bölme oranlarını Cora, CiteSeer, Computers, Physics, Actor, ogbn-arxiv, Flickr, Squirrel ve Roman-empire için tanımlıyor; Minesweeper için tanım yok, bu yüzden orijinal betik Minesweeper'da çalışmaz. Ortak cache kullanıldığı için bu kısıt harness'te geçerli değil. Kod derlenmiş bir eklenti (torch_scatter, torch_sparse, DGL) kullanmıyor; PyTorch Geometric'e, veri katmanı için de OpenFGL'e bağlı. `config.py`'de `--gpuid 0` ve yer tutucu bir `--root` var; `optuna_search.py` ise cihazı CPU'ya zorluyor (`args.disable_cuda = True`). Harness'te cihaz otomatik seçiliyor (GPU varsa GPU).
* **`edge_masking`.** `task_data`, görev çizgesini `utils.edge_masking` ile kurmaya çalışıyor. Fonksiyon boolean bir maske üzerinde `for node in handled` ile döndüğü için görevler arası kenarları silmiyor ve self-loop'ları yalnızca 0 ve 1 numaralı düğümlere ekliyor (Bölüm 6.4'te küçük bir örnekle gösteriliyor). Harness bu fonksiyonu kullanmıyor; tüm yöntemlere aynı görev alt-çizgisini veriyor.

### 2.2 FedSPA (Tan vd., CVPR'25)
FedSPA, standart federe çizge öğrenmesinde (FGL) **homofili heterojenliği** sorununa yöneliyor. İstemcilerin yerel alt-çizgilerinin homofili düzeyleri farklı olduğunda global GNN'in genellemesi bozuluyor ([README](https://github.com/OakleyTan/FedSPA)). Yöntemin iki bileşeni var: **SFPD** (Subgraph Feature Propagation Decoupling) istemcilerin ortak bir homofili düzeyinde iş birliği yapmasını sağlıyor; **HBDA** (Homophily Bias-Driven Aggregation) birleştirmede homofili yanlılığı düşük istemcileri öne çıkarıyor.

**FedSPA bir sürekli öğrenme (continual learning) yöntemi değil; unutmaya karşı bir mekanizması yok.** Bu çalışmada, POWER makalesinin FedAvg ve FedGTA gibi FGL yöntemlerini FGCL'ye uyarladığı biçimde kullanıldı: görevler sırayla geliyor ve aynı global model her görevde FedSPA ile eğitiliyor ("FL/FGL fine-tuning"; [POWER teknik raporu](https://zenodo.org/records/15487655), Ek A.1).

| Bileşen | Kod (`Code/` altında) | Ne yapıyor |
|---|---|---|
| SFPD (omurga) | `benchmark001/config.py`: `ACM`, `GraphConvolution`, `mp_func_spa`, `agg_func_spa` | 2 katmanlı bir model; her katmanda üç kanal var: alçak geçiren ($\tilde D^{-1}(A+I)$), yüksek geçiren ($I-\tilde D^{-1}(A+I)$) ve MLP. Sınıf adı `ACM`; yapı, ACM-GCN'in (Luan vd., 2022) kanallarına benziyor. Komşu mesajları eğitim etiketlerine göre üçe ayrılır: iki uç da etiketli ve **aynı** sınıftan (homofilik), etiketli ve **farklı** sınıftan (heterofilik), en az bir ucu etiketsiz (bilinmeyen). Her grup için ayrı dikkat vektörleri öğrenilir. Bilinmeyen gruptaki mesajlar, öğrenilen bir kapıyla (`fc_balance`, sigmoid) heterofilik ("fr") ve homofilik ("be") dikkat vektörlerinin karışımıyla puanlanır. |
| HBDA | `flgo/algorithm/fedspa.py`: `Server.aggregate`, `fish_cal`, `calculate_weights`; `fedbase.py`: `calculate_shigh` | "fr" ve "be" dikkat parametreleri, Fisher bilgisi (log-olabilirlik gradyanının karesi) istemci ortalamasına yakın olan istemcilere daha büyük ağırlık verilerek birleştirilir: $w_k \propto e^{-\tilde d_k}$, burada $\tilde d_k$ min-max ölçeklenmiş farktır. Diğer tüm parametreler aynı formülle, istemcinin öznitelik yüksek frekans oranına ($S_{high}$, `calculate_shigh`) göre ağırlıklandırılır. |
| Komşuluk matrisleri | `flgo/experiment/logger/simple_logger.py` (`log_once`) | `adj_low` ve `adj_high`, eğitimin başında logger'ın içinde bir kez hesaplanıyor. |

Kodda gördüğüm ayrıntılar:
1. HBDA'nın homofili ağırlıkları, sunucunun hesapladığı "istemci $S_{high}$'ının ağırlıklı ortalamadan sapması" (`server.radio_diff`) ile değil, istemcinin **ham** $|S_{high}|$ değeriyle hesaplanıyor (`fedspa.py`: `self.diffs.append(abs(client.radio_diff))`).
2. Değerlendirmede `model.eval()` hiç çağrılmıyor; yani orijinal değerlendirme dropout açıkken yapılıyor.
3. `GraphConvolution.att_struc_low` ilklenmemiş bellekle (`torch.FloatTensor`) oluşturuluyor. Yerel denemelerde bu bellek zaman zaman NaN içerdi; birleştirmedeki `has_nan()` kontrolü o modeli dışarıda bıraktı ve `ZeroDivisionError` / `ValueError` hatalarına yol açtı (Bölüm 6.2).
4. Denge dalındaki altıncı dikkat terimi `neigh_unk_high` yerine `neigh_hete_high`'ı kullanıyor (`attention7`). Port'ta kodda olduğu gibi korundu.

## 3. Kurulum
Colab'daki hazır torch'a dokunulmuyor (yeniden kurma ya da sürüm düşürme yok). Yalnızca eksik olan saf-Python paketleri pip ile kuruluyor: `torch_geometric`, `python-louvain` (FedFST `from community import community_louvain` kullanıyor; PyPI'daki `community` adlı paket başka bir şey), `ogb` ve `gdown` (FedFST'nin veri yükleyicisi bunları import ediyor). Önceden eğitilmiş bir model gerekmiyor; tüm GNN'ler sıfırdan eğitiliyor.

### 3.1 Ortam bilgisi ve Google Drive

In [ ]:
import os, sys, re, json, time, shutil, platform, subprocess, importlib, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message=".*pkg_resources is deprecated.*")   # ogb'nin bağımlılığından
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/fedfst_odev2")                    # kodlar (oturum boyunca)
    WORK = Path("/content/drive/MyDrive/FedFST_Odev2")      # sonuçlar, log'lar, cache (kalıcı)
else:  # yerel test: Colab'a özgü adımlar atlanır
    BASE = Path(os.environ.get("FGCL_BASE", "fedfst_odev2_local")).resolve()
    WORK = Path(os.environ.get("FGCL_WORK", str(BASE / "work"))).resolve()
SRC = BASE / "src"
for p in (BASE, SRC, WORK, WORK / "data", WORK / "cache"):
    p.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
print("Colab:", IN_COLAB, "| kod klasörü:", BASE, "| kalıcı klasör:", WORK)

import torch
print("Python", platform.python_version(), "| torch", torch.__version__, "| torch'un CUDA sürümü:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    if shutil.which("nvidia-smi"):
        print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                              "--format=csv"], capture_output=True, text=True).stdout)
else:
    print("GPU yok: tüm koşular otomatik olarak CPU'da çalışacak.")

### 3.2 Bağımlılıklar

In [ ]:
def importable(module):
    try:
        importlib.import_module(module)
        return True
    except Exception:
        return False

missing = [pkg for mod, pkg in [("torch_geometric", "torch_geometric"),
                                ("community.community_louvain", "python-louvain"),
                                ("ogb", "ogb"), ("gdown", "gdown")] if not importable(mod)]
if missing:
    print("kuruluyor:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    importlib.invalidate_caches()
else:
    print("gerekli paketler zaten kurulu")

from importlib.metadata import version, PackageNotFoundError
for pkg in ["torch", "torch_geometric", "python-louvain", "ogb", "gdown", "networkx", "numpy", "scipy",
            "scikit-learn", "pandas", "matplotlib"]:
    try:
        print(f"{pkg:16s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:16s} -")

## 4. Kaynak kodların indirilmesi
### 4.1 FedFST ve FedSPA (sabit commit'ler)

In [ ]:
def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()

REPOS = {"FedFST": ("https://github.com/YunQi572/FedFST", "d83fac0"),
         "FedSPA": ("https://github.com/OakleyTan/FedSPA", "d2b3ddb")}
for name, (url, commit) in REPOS.items():
    repo = SRC / name
    if not (repo / ".git").exists():
        git("clone", "-q", url, str(repo))
    if not git("rev-parse", "HEAD", cwd=repo).startswith(commit):
        git("checkout", "-q", commit, cwd=repo)
    print(f"{name:7s} {git('log', '-1', '--format=%h  %ad  %s', '--date=short', cwd=repo)}")
FEDFST = SRC / "FedFST"

### 4.2 POWER (Zenodo)
POWER'ın GitHub reposu açılmıyor; resmi kod Zenodo'da ([10.5281/zenodo.15487654](https://doi.org/10.5281/zenodo.15487654), `zyl24/FCGL_POWER-official.zip`, md5 `567faa826ad62a3d8610682ee2e8474f`). Zip, Drive'daki çalışma klasörüne indirilir. Zenodo'ya erişilemezse zip'i elle indirip `MyDrive/FedFST_Odev2/FCGL_POWER-official.zip` olarak koymak yeterli; hücre önce oraya bakar.

In [ ]:
import hashlib, zipfile, urllib.request

POWER_URL = "https://zenodo.org/records/15487655/files/zyl24/FCGL_POWER-official.zip?download=1"
POWER_MD5 = "567faa826ad62a3d8610682ee2e8474f"
POWER_ZIP = WORK / "FCGL_POWER-official.zip"
POWER_DIR = SRC / "POWER"

def md5sum(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not (POWER_DIR / "flcore" / "Ours.py").exists():
    if not (POWER_ZIP.exists() and md5sum(POWER_ZIP) == POWER_MD5):
        print("Zenodo'dan indiriliyor ...")
        part = POWER_ZIP.with_suffix(".part")
        if shutil.which("wget"):
            subprocess.run(["wget", "-q", "-O", str(part), POWER_URL])
        else:
            try:
                urllib.request.urlretrieve(POWER_URL, part)
            except Exception as exc:
                print("indirme hatası:", exc)
        if part.exists() and part.stat().st_size > 0 and md5sum(part) == POWER_MD5:
            part.replace(POWER_ZIP)
    assert POWER_ZIP.exists() and md5sum(POWER_ZIP) == POWER_MD5, (
        f"POWER zip'i indirilemedi ya da md5 tutmuyor. Zenodo'dan elle indirip {POWER_ZIP} olarak koyun.")
    with zipfile.ZipFile(POWER_ZIP) as z:
        z.extractall(SRC)
    (SRC / "zyl24-FCGL_POWER-78e30b6").rename(POWER_DIR)
if POWER_ZIP.exists():
    print("POWER zip md5:", md5sum(POWER_ZIP))
print("POWER dosyaları:", sorted(p.name for p in POWER_DIR.iterdir()))

## 5. Yamalar ve harness dosyaları
Değişiklikler en küçük tutuldu ve her biri ayrı bir hücrede. Yamalar **idempotenttir**: yeni metin bir `[Odev2-Px]` işareti taşır; işaret dosyada zaten varsa yama atlanır, yani hücreler iki kez çalıştırılınca dosyalar bozulmaz. Yama motoru (`fgcl_bench/patching.py`) orijinal metni satır satır arar. Orijinal dosyalarda boş satırlarda girinti boşlukları olduğu için satır sonu boşluklarını yok sayar. Tam olarak bir eşleşme bulamazsa hata verir.

In [ ]:
for d in ["fgcl_bench", "fgcl_bench/shims/openfgl/utils"]:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile fgcl_bench/patching.py
"""Idempotent metin yamaları.

apply_patch(), dosyada `old` bloğunu satır satır arar (satır sonu boşlukları yok
sayılır, çünkü orijinal dosyalarda boş satırlarda girinti boşlukları var) ve
`new` ile değiştirir. `new` bloğu dosyada zaten varsa hiçbir şey yapmaz; bu
yüzden hücre iki kez çalıştırılınca dosya bozulmaz.
"""
import difflib
from pathlib import Path


def _norm(block):
    return [line.rstrip() for line in block.strip("\n").split("\n")]


def _find(file_lines, block_lines):
    n = len(block_lines)
    return [i for i in range(len(file_lines) - n + 1) if file_lines[i:i + n] == block_lines]


def create_files(root, pid, files, content):
    created = []
    for rel in files:
        path = Path(root) / rel
        if not path.exists():
            path.write_text(content, encoding="utf-8")
            created.append(rel)
    print(f"{pid}: " + (f"oluşturuldu: {', '.join(created)}" if created else "dosyalar zaten var, atlandı."))
    return bool(created)


def apply_patch(root, relpath, pid, old, new, show_diff=True, max_diff_lines=40):
    path = Path(root) / relpath
    text = path.read_text(encoding="utf-8")
    lines = text.split("\n")
    norm_lines = [line.rstrip() for line in lines]
    if _find(norm_lines, _norm(new)):
        print(f"{pid}: {relpath} zaten yamalı, atlandı.")
        return False
    hits = _find(norm_lines, _norm(old))
    if len(hits) != 1:
        raise RuntimeError(f"{pid}: {relpath} içinde beklenen orijinal metin {len(hits)} kez bulundu "
                           f"(1 bekleniyordu). Repo beklenen commit'te mi?")
    start = hits[0]
    old_len = len(_norm(old))
    new_lines = new.strip("\n").split("\n")
    patched = lines[:start] + new_lines + lines[start + old_len:]
    path.write_text("\n".join(patched), encoding="utf-8")
    removed, added = old_len, len(new_lines)
    print(f"{pid}: {relpath} yamalandı (satır {start + 1}: -{removed} / +{added}).")
    if show_diff:
        diff = list(difflib.unified_diff([l.rstrip() for l in lines[start:start + old_len]],
                                         new_lines, lineterm="", n=0))[2:]
        for line in diff[:max_diff_lines]:
            print("   " + line)
        if len(diff) > max_diff_lines:
            print(f"   ... ({len(diff) - max_diff_lines} satır daha)")
    return True


In [ ]:
from fgcl_bench.patching import apply_patch, create_files

### 5.1 FedFST yamaları
**P0–P4**, repoyu çalışır hâle getirir; **P5a–P5f**, ölçülen darboğazlardaki Python döngülerini aynı matematik ve aynı RNG kullanımıyla vektörize eder. `--device_id` ve diğer argümanlar için yama gerekmiyor: koşucu `--device_id 0` veriyor ve `type=bool` tanımlı `--use_gpu` / `--shuffle_flag` argümanlarını komut satırından vermiyor (argparse `"False"` dizesini de True okur).

**P0 – datasets/, algorithm/, models/ klasörlerine boş __init__.py ekle**

*Neden:* FedFST bu klasörleri `from datasets.dataset_loader import ...` gibi import ediyor ama klasörlerde `__init__.py` yok (namespace package). Colab'da Hugging Face `datasets` paketi önceden kurulu; Python normal bir paketi namespace package'a tercih ettiği için `datasets` adı HF paketine gidiyor ve import `ModuleNotFoundError: No module named 'datasets.dataset_loader'` ile çöküyor (HF datasets kurulu bir ortamda yeniden ürettim). Boş `__init__.py` dosyaları kodun davranışını değiştirmez.

In [ ]:
create_files(FEDFST, "P0", ['datasets/__init__.py', 'algorithm/__init__.py', 'models/__init__.py'],
             '# [Odev2-P0] Klasörü normal bir Python paketi yapar (bkz. notebook, yama P0).\n')

**P1 – `datasets/dataset_loader.py`: torch_sparse import'unu kaldır**

*Neden:* `from torch_sparse import SparseTensor` satırı ModuleNotFoundError veriyor. SparseTensor dosyada hiç kullanılmıyor; torch_sparse derlenmiş bir eklenti ve Colab'ın güncel torch sürümü için wheel'i yok.

In [ ]:
apply_patch(FEDFST, "datasets/dataset_loader.py", "P1",
old=r'''
from torch_sparse import SparseTensor
''',
new=r'''
# [Odev2-P1] "from torch_sparse import SparseTensor" kaldırıldı: SparseTensor kullanılmıyor.
''')

**P2 – `algorithm/Ours.py`: Var olmayan iki fonksiyonun import'unu kaldır**

*Neden:* Ours.py 12. satır utils'ten `edge_distribution_low1` ve `edge_distribution_low2` import ediyor; bu fonksiyonlar utils.py'de yok (ImportError) ve Ours.py'de hiç kullanılmıyor.

In [ ]:
apply_patch(FEDFST, "algorithm/Ours.py", "P2",
old=r'''
from utils import Generator, edge_distribution_low, get_khop_feature, edge_distribution_low1, edge_distribution_low2, get_Shigh, reduce_homophilic_edges, reduce_heterophilic_edges, get_Homophily_Level
''',
new=r'''
# [Odev2-P2] edge_distribution_low1/2 import'tan çıkarıldı (utils.py'de yoklar, kullanılmıyorlar).
from utils import Generator, edge_distribution_low, get_khop_feature, get_Shigh, reduce_homophilic_edges, reduce_heterophilic_edges, get_Homophily_Level
''')

**P3a – `args.py`: Eksik iki argümanı ekle**

*Neden:* main.py 26–27. satırlar `args.kd_ce_weight` ve `args.gen_ce_weight` yazdırıyor ama args.py'de tanımlı değiller (AttributeError). Varsayılanlar Ours.py'deki getattr varsayılanlarıyla aynı (254. satır: 0.5, 799. satır: 1.0); davranış değişmez.

In [ ]:
apply_patch(FEDFST, "args.py", "P3a",
old=r'''
parser.add_argument('--gen_kl_weight', type=float, default=100)
''',
new=r'''
parser.add_argument('--gen_kl_weight', type=float, default=100)
# [Odev2-P3a] main.py bu iki argümanı yazdırıyor ama tanımlı değillerdi. Varsayılanlar
# Ours.py'deki getattr varsayılanlarıyla aynı (satır 254 ve 799); davranış değişmez.
parser.add_argument('--kd_ce_weight', type=float, default=0.5)
parser.add_argument('--gen_ce_weight', type=float, default=1.0)
''')

**P3b – `main.py`: Tanımsız argümanı yazdıran print'i düzelt**

*Neden:* main.py 28. satır tanımsız `args.S_high_D_weight`'i yazdırıyor (AttributeError). Bu değer hiçbir yerde kullanılmıyor; print, gerçekte kullanılan `S_high_A_weight`'i yazdıracak şekilde düzeltildi.

In [ ]:
apply_patch(FEDFST, "main.py", "P3b",
old=r'''
    print(f"S_high_x_weight={args.S_high_x_weight}, S_high_D_weight={args.S_high_D_weight}")
''',
new=r'''
    print(f"S_high_x_weight={args.S_high_x_weight}, S_high_A_weight={args.S_high_A_weight}")  # [Odev2-P3b]
''')

**P4a – `datasets/dataset_loader.py`: Minesweeper ve Squirrel'i kabul edilen veri setlerine ekle (assert)**

*Neden:* `load_dataset` yalnızca belirli isimleri kabul ediyor; minesweeper ve squirrel listede yok.

In [ ]:
apply_patch(FEDFST, "datasets/dataset_loader.py", "P4a",
old=r'''
    assert dataset_name in ('cora', 'citeseer', 'pubmed', 'ogbn-arxiv',
                            'computers', 'physics', 'roman_empire', 'year', 'actor', 'cs'), 'Invalid dataset'
''',
new=r'''
    assert dataset_name in ('cora', 'citeseer', 'pubmed', 'ogbn-arxiv',
                            'computers', 'physics', 'roman_empire', 'year', 'actor', 'cs',
                            'minesweeper', 'squirrel'), 'Invalid dataset'  # [Odev2-P4a]
''')

**P4b – `datasets/dataset_loader.py`: Minesweeper ve Squirrel yükleme dallarını ekle**

*Neden:* İki veri seti PyG ile yükleniyor: `HeterophilousGraphDataset(root, name='Minesweeper')` ve `WikipediaNetwork(root, name='squirrel')` (geom-gcn ön işlemeli sürüm, PyG varsayılanı). İki sınıf da dosyada zaten import edilmiş.

In [ ]:
apply_patch(FEDFST, "datasets/dataset_loader.py", "P4b",
old=r'''
    elif dataset_name in ['actor']:
        dataset = Actor(root=root_dir)
        data = dataset[0]
''',
new=r'''
    elif dataset_name in ['actor']:
        dataset = Actor(root=root_dir)
        data = dataset[0]

    # [Odev2-P4b] Minesweeper ve Squirrel (iki sınıf da bu dosyada zaten import edilmiş).
    elif dataset_name in ['minesweeper']:
        dataset = HeterophilousGraphDataset(root=root_dir, name='Minesweeper')
        data = dataset[0]

    elif dataset_name in ['squirrel']:
        dataset = WikipediaNetwork(root=root_dir, name='squirrel')
        data = dataset[0]
''')

**P5a – `utils.py`: reduce_homophilic_edges / reduce_heterophilic_edges: kenar döngüsünü vektörize et**

*Neden:* İki fonksiyon kenar kenar `.item()` ile dönüyor; jeneratör eğitiminde her epoch'ta 60 kez çağrılıyor. Yeni sürüm aynı kenarları aynı sırada seçiyor ve `torch.randperm`'i aynı generator'dan (varsayılan CPU generator'ı), aynı n ile, aynı sırada çağırıyor; çıktı ve RNG durumu birebir aynı (eşdeğerlik testi aşağıda).

In [ ]:
apply_patch(FEDFST, "utils.py", "P5a",
old=r'''
def reduce_homophilic_edges(data, edge_index, reduction_ratio=0.1):
    labels = data.y

    homophilic_edges = []
    heterophilic_edges = []

    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i].item(), edge_index[1, i].item()
        if labels[src] == labels[dst]:
            homophilic_edges.append(i)
        else:
            heterophilic_edges.append(i)

    num_to_remove = int(len(homophilic_edges) * reduction_ratio)
    if num_to_remove > 0:
        remove_indices = torch.randperm(len(homophilic_edges))[:num_to_remove]
        edges_to_remove = [homophilic_edges[i] for i in remove_indices]

        keep_mask = torch.ones(edge_index.shape[1], dtype=torch.bool)
        keep_mask[edges_to_remove] = False
        new_edge_index = edge_index[:, keep_mask]
    else:
        new_edge_index = edge_index

    return new_edge_index

def reduce_heterophilic_edges(data, edge_index, reduction_ratio=0.1):
    labels = data.y

    heterophilic_edges = []

    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i].item(), edge_index[1, i].item()
        if labels[src] != labels[dst]:
            heterophilic_edges.append(i)

    num_to_remove = int(len(heterophilic_edges) * reduction_ratio)
    if num_to_remove > 0:
        remove_indices = torch.randperm(len(heterophilic_edges))[:num_to_remove]
        edges_to_remove = [heterophilic_edges[i] for i in remove_indices]

        keep_mask = torch.ones(edge_index.shape[1], dtype=torch.bool)
        keep_mask[edges_to_remove] = False
        new_edge_index = edge_index[:, keep_mask]
    else:
        new_edge_index = edge_index

    return new_edge_index
''',
new=r'''
# [Odev2-P5a] Kenar döngüsü vektörize edildi. Seçilen kenarlar ve sıraları aynı; torch.randperm
# aynı generator'dan (varsayılan CPU generator'ı), aynı n ile çağrılıyor -> çıktı ve RNG durumu aynı.
def _reduce_edges_by_label_relation(data, edge_index, reduction_ratio, homophilic):
    labels = data.y
    ei = edge_index.to(labels.device)
    same = labels[ei[0]] == labels[ei[1]]
    selected = same if homophilic else ~same
    candidate_edges = torch.nonzero(selected, as_tuple=False).view(-1).cpu()

    num_to_remove = int(candidate_edges.numel() * reduction_ratio)
    if num_to_remove > 0:
        remove_indices = torch.randperm(candidate_edges.numel())[:num_to_remove]
        edges_to_remove = candidate_edges[remove_indices]

        keep_mask = torch.ones(edge_index.shape[1], dtype=torch.bool)
        keep_mask[edges_to_remove] = False
        new_edge_index = edge_index[:, keep_mask]
    else:
        new_edge_index = edge_index

    return new_edge_index

def reduce_homophilic_edges(data, edge_index, reduction_ratio=0.1):
    return _reduce_edges_by_label_relation(data, edge_index, reduction_ratio, homophilic=True)

def reduce_heterophilic_edges(data, edge_index, reduction_ratio=0.1):
    return _reduce_edges_by_label_relation(data, edge_index, reduction_ratio, homophilic=False)
''')

**P5b – `algorithm/Ours.py`: get_learned_Homophily_Level: kenar döngüsünü vektörize et**

*Neden:* Önceki görevlerin eğitim düğümleri arasındaki kenarlarda homofili oranını kenar kenar `.item()` ile sayıyor. Yeni sürüm aynı iki sayacı (toplam kenar, aynı etiketli kenar) tensör işlemleriyle sayıp aynı bölmeyi yapıyor. RNG kullanılmıyor.

In [ ]:
apply_patch(FEDFST, "algorithm/Ours.py", "P5b",
old=r'''
    def get_learned_Homophily_Level(self, task_id):
        total_edge_nums = 0
        same_label_edges = 0

        for task_i in range(task_id):
            task = self.tasks[task_i]
            graph = task["local_data"]
            train_mask = task["train_mask"]

            edge_index = graph.edge_index
            node_labels = graph.y

            for edge_idx in range(edge_index.size(1)):
                src_node = edge_index[0, edge_idx].item()
                dst_node = edge_index[1, edge_idx].item()

                if train_mask[src_node] and train_mask[dst_node]:
                    total_edge_nums += 1

                    if node_labels[src_node] == node_labels[dst_node]:
                        same_label_edges += 1

        if total_edge_nums > 0:
            homophily_ratio = same_label_edges / total_edge_nums
        else:
            homophily_ratio = 0.0

        return homophily_ratio
''',
new=r'''
    def get_learned_Homophily_Level(self, task_id):
        # [Odev2-P5b] Kenar döngüsü vektörize edildi; iki sayaç ve sonuç aynı, RNG kullanılmıyor.
        total_edge_nums = 0
        same_label_edges = 0

        for task_i in range(task_id):
            task = self.tasks[task_i]
            graph = task["local_data"]
            train_mask = task["train_mask"]

            edge_index = graph.edge_index
            node_labels = graph.y

            src_nodes, dst_nodes = edge_index[0], edge_index[1]
            train_mask = train_mask.to(edge_index.device)
            both_train = train_mask[src_nodes] & train_mask[dst_nodes]
            same_label = node_labels[src_nodes] == node_labels[dst_nodes]
            total_edge_nums += int(both_train.sum().item())
            same_label_edges += int((both_train & same_label).sum().item())

        if total_edge_nums > 0:
            homophily_ratio = same_label_edges / total_edge_nums
        else:
            homophily_ratio = 0.0

        return homophily_ratio
''')

**P5c – `algorithm/Ours.py`: feature_gen_train: düğüm düğüm KL döngüsü için vektörize yardımcı fonksiyonu ekle**

*Neden:* Yardımcı fonksiyon, P5d'de döngünün yerine çağrılıyor. Örnekler aynı sırada işleniyor; aynı etiketli ardışık örnekler için tek bir `torch.randint(0, n_c, (k,))` çağrısı, varsayılan CPU generator'ından k ayrı `torch.randint(0, n_c, (1,))` çağrısıyla aynı sayıları çekiyor (testte doğrulandı).

In [ ]:
apply_patch(FEDFST, "algorithm/Ours.py", "P5c",
old=r'''
class OursServer(BaseServer):
''',
new=r'''
def _gen_kl_loss_vectorized(all_generated_features, all_generated_labels, subgraph, device):
    # [Odev2-P5c] OursClient.feature_gen_train'deki örnek örnek KL döngüsünün vektörize hali.
    # Örnekler aynı sırada işlenir. Aynı etiketli ardışık örnekler için tek bir torch.randint
    # çağrısı, varsayılan CPU generator'ından tek tek çağrılarla aynı sayıları aynı sırada çeker.
    labels_cpu = all_generated_labels.detach().cpu()
    num_generated = labels_cpu.numel()
    rows_list, real_list = [], []
    if num_generated > 0:
        change = torch.nonzero(labels_cpu[1:] != labels_cpu[:-1], as_tuple=False).view(-1) + 1
        bounds = [0] + change.tolist() + [num_generated]
        for start, end in zip(bounds[:-1], bounds[1:]):
            gen_label = all_generated_labels[start]
            same_class_mask = (subgraph.y == gen_label)
            if same_class_mask.sum() > 0:
                same_class_indices = torch.where(same_class_mask)[0]
                random_idx = torch.randint(0, len(same_class_indices), (end - start,))
                real_list.append(same_class_indices[random_idx.to(same_class_indices.device)])
                rows_list.append(torch.arange(start, end, device=all_generated_features.device))
    if rows_list:
        rows = torch.cat(rows_list)
        real_idx = torch.cat(real_list)
        kl_per_sample = F.kl_div(
            F.log_softmax(all_generated_features[rows], dim = 1),
            F.softmax(subgraph.x[real_idx], dim = 1),
            reduction = 'none'
        ).sum(dim = 1)
        return kl_per_sample.mean()
    return torch.tensor(0.0, device=device)


class OursServer(BaseServer):
''')

**P5d – `algorithm/Ours.py`: feature_gen_train: KL döngüsünü vektörize yardımcıyla değiştir**

*Neden:* Her epoch'ta üretilen her düğüm için (Cora'da 400–800 düğüm) ayrı ayrı rastgele gerçek düğüm seçip KL hesaplayan döngü, P5c'deki fonksiyona çağrıyla değiştirildi. Kayıp formülü aynı: KL(softmax(gerçek) || softmax(üretilen)) özniteliklerde toplanıp örneklerde ortalanıyor.

In [ ]:
apply_patch(FEDFST, "algorithm/Ours.py", "P5d",
old=r'''
            kl_losses = []
            for i, gen_label in enumerate(all_generated_labels):
                same_class_mask = (subgraph.y == gen_label)
                if same_class_mask.sum() > 0:
                    same_class_indices = torch.where(same_class_mask)[0]
                    random_idx = torch.randint(0, len(same_class_indices), (1,)).item()
                    real_feature = subgraph.x[same_class_indices[random_idx]]

                    kl_loss = F.kl_div(
                        F.log_softmax(all_generated_features[i], dim = 0),
                        F.softmax(real_feature, dim = 0),
                        reduction = 'sum'
                    )

                    kl_losses.append(kl_loss)

            if kl_losses:
                loss_kl = torch.stack(kl_losses).mean()
            else:
                loss_kl = torch.tensor(0.0, device=self.device)
''',
new=r'''
            # [Odev2-P5d] Örnek örnek KL döngüsü yerine eşdeğer vektörize hesap (bkz. _gen_kl_loss_vectorized).
            loss_kl = _gen_kl_loss_vectorized(all_generated_features, all_generated_labels, subgraph, self.device)
''')

**P5e – `datasets/partition.py`: get_subgraph_by_node: kenar döngüsünü vektörize et**

*Neden:* Görev alt-grafikleri (class_to_task), öğrenilmiş alt-graf (get_learned_graph) ve her round'daki get_subgraph bu fonksiyonu çağırıyor; kenar kenar `.item()` kullanıyor. Yeni sürüm aynı kenarları aynı sırada, aynı dtype ve bellek düzeniyle (transpoze görünüm) döndürüyor. Düğüm listesinde tekrar varsa orijinal döngü aynen çalışıyor. RNG kullanılmıyor.

In [ ]:
apply_patch(FEDFST, "datasets/partition.py", "P5e",
old=r'''
def get_subgraph_by_node(dataset, node_list, flag):
    node_id_set = set(node_list)
''',
new=r'''
def get_subgraph_by_node(dataset, node_list, flag):
    # [Odev2-P5e] Vektörize sürüm: aynı kenarlar, aynı sıra, aynı dtype ve bellek düzeni.
    node_ids = [int(v) for v in node_list]
    if len(set(node_ids)) != len(node_ids):
        return _get_subgraph_by_node_loop(dataset, node_list, flag)
    local_id_to_global_id = list(node_list)
    edge_index = dataset.edge_index.cpu()
    num_slots = int(dataset.x.shape[0])
    if node_ids:
        num_slots = max(num_slots, max(node_ids) + 1)
    if edge_index.numel() > 0:
        num_slots = max(num_slots, int(edge_index.max()) + 1)
    global_id_to_local_id = torch.full((num_slots,), -1, dtype=torch.long)
    if node_ids:
        global_id_to_local_id[torch.tensor(node_ids, dtype=torch.long)] = torch.arange(len(node_ids), dtype=torch.long)
    keep = (global_id_to_local_id[edge_index[0]] >= 0) & (global_id_to_local_id[edge_index[1]] >= 0) if edge_index.numel() > 0 else torch.zeros(0, dtype=torch.bool)
    if bool(keep.any()):
        # Orijinal kod torch.tensor(list_of_pairs).t() döndürüyordu: (2, E) şeklinde transpoze görünüm.
        global_edge_list = edge_index[:, keep].t().contiguous().t()
        local_edge_index = global_id_to_local_id[global_edge_list.t()].contiguous().t()
    else:
        local_edge_index = torch.empty((2, 0), dtype=torch.int64)
        global_edge_list = torch.tensor([]).t()
    if flag:
        local_subgraph = Data(x=dataset.x[node_list], edge_index=local_edge_index, y=dataset.y[node_list])
    else:
        local_subgraph = Data(x=dataset.x, edge_index=global_edge_list, y=dataset.y)
    local_subgraph.global_map = local_id_to_global_id

    return local_subgraph


def _get_subgraph_by_node_loop(dataset, node_list, flag):
    # Orijinal (döngülü) sürüm; yalnızca düğüm listesinde tekrar varsa kullanılır.
    node_id_set = set(node_list)
''')

**P5f – `datasets/partition.py`: louvain_partitioner: istemci alt-grafiği çıkarımını vektörize et**

*Neden:* Louvain bölümlemesi (python-louvain, numpy RNG) aynen korunuyor. Yalnızca her istemcinin kenarlarını kenar kenar `.item()` ile seçen döngü vektörize edildi; aynı kenarlar aynı sırada ve aynı yeni numaralarla seçiliyor. Düğüm listesinde tekrar varsa orijinal döngü çalışıyor.

In [ ]:
apply_patch(FEDFST, "datasets/partition.py", "P5f",
old=r'''
    clients_data = []
    for nodes in clients_nodes:
        node_map = {node: i for i, node in enumerate(nodes)}
        sub_edge_index = []
        for i in range(data.edge_index.size(1)):
            if data.edge_index[0, i].item() in node_map and data.edge_index[1, i].item() in node_map:
                sub_edge_index.append([node_map[data.edge_index[0, i].item()], node_map[data.edge_index[1, i].item()]])
        sub_edge_index = torch.tensor(sub_edge_index, dtype=torch.long).t().contiguous()
''',
new=r'''
    clients_data = []
    edge_src_all = data.edge_index[0]
    edge_dst_all = data.edge_index[1]
    for nodes in clients_nodes:
        # [Odev2-P5f] Kenar seçimi vektörize edildi: aynı kenarlar, aynı sıra, aynı yeni numaralar.
        node_ids = [int(v) for v in nodes]
        if len(set(node_ids)) == len(node_ids):
            node_map_t = torch.full((data.num_nodes,), -1, dtype=torch.long)
            node_map_t[torch.tensor(node_ids, dtype=torch.long)] = torch.arange(len(node_ids), dtype=torch.long)
            keep = (node_map_t[edge_src_all] >= 0) & (node_map_t[edge_dst_all] >= 0)
            if bool(keep.any()):
                sub_edge_index = torch.stack([node_map_t[edge_src_all[keep]], node_map_t[edge_dst_all[keep]]], dim=0).contiguous()
            else:
                sub_edge_index = torch.tensor([], dtype=torch.long).t().contiguous()
        else:
            node_map = {node: i for i, node in enumerate(nodes)}
            sub_edge_index = []
            for i in range(data.edge_index.size(1)):
                if data.edge_index[0, i].item() in node_map and data.edge_index[1, i].item() in node_map:
                    sub_edge_index.append([node_map[data.edge_index[0, i].item()], node_map[data.edge_index[1, i].item()]])
            sub_edge_index = torch.tensor(sub_edge_index, dtype=torch.long).t().contiguous()
''')

### 5.2 POWER ve FedSPA
**POWER:** ödevde tercih edilen yol seçildi: resmi istemci ve sunucu sınıfları (`flcore/Ours.py`) olduğu gibi import edilip harness'e bir adaptörle bağlandı. Bu yol mümkündü, çünkü POWER'ın algoritma kodu veri katmanından ayrık; veriyi yalnızca `self.data["data"]`, `self.data["task"]` ve `task_data()` üzerinden görüyor. Bu yüzden veri sözlüğü ortak cache'ten kuruldu ve bir alt sınıfta yalnızca `task_data` değiştirildi. Eğitim döngüsü POWER'ın `utils.start()` döngüsünün aynısı. Algoritma dosyalarına (`flcore/Ours.py`, `model.py`, `utils.py`) dokunulmuyor. `flcore/` klasörü de FedFST'nin klasörleri gibi `__init__.py`'siz bir namespace package olduğu için aynı önlem (P0) burada da alınıyor. POWER'ın OpenFGL'den kullandığı tek fonksiyon (`idx_to_mask_tensor`) OpenFGL'den aynen kopyalanarak `fgcl_bench/shims/` altına konuyor. Veri arayüzü `fgcl_bench/runner_power.py`'deki adaptörde (Bölüm 5.4).

**FedSPA:** repoya yama yapılmıyor. Yöntem, gömülü flgo kopyasından ve DGL'den bağımsız olarak `fgcl_bench/fedspa_port.py`'ye taşındı (Bölüm 5.4 ve 6.2).

In [ ]:
create_files(POWER_DIR, "POWER-P0", ["flcore/__init__.py"],
             "# [Odev2-POWER-P0] flcore'u normal bir Python paketi yapar (bkz. FedFST P0).\n")

### 5.3 Harness dosyaları
Ortak deney düzeneği `fgcl_bench/` paketinde. Her koşu ayrı bir süreçte çalışır (`python -m fgcl_bench.run --method {fedfst,power,fedspa} --dataset ... --seed ...`) ve sonucunu bir JSON dosyasına yazar.

| Dosya | Görevi |
|---|---|
| `fgcl_bench/__init__.py` | paket işareti |
| `fgcl_bench/common.py` | veri seti tanımları, seed (FedFST'nin set_seed'iyle aynı), görev hash'i, atomik JSON yazımı |
| `fgcl_bench/data.py` | ham dosyaların indirilmesi; FedFST fonksiyonlarıyla bölümleme ve görevlerin üretilip diske (cache) yazılması |
| `fgcl_bench/evaluate.py` | ortak değerlendirme: doğruluk matrisi, AA ve AF (FedFST'nin utils.AA / utils.AF fonksiyonlarıyla) |
| `fgcl_bench/runner_fedfst.py` | FedFST: main.py'nin eğitim döngüsü, resmi (yamalı) Ours.py sınıflarıyla |
| `fgcl_bench/runner_power.py` | POWER: resmi flcore/Ours.py + veri arayüzü adaptörü |
| `fgcl_bench/fedspa_port.py` | FedSPA: SFPD + HBDA'nın orijinal koddan port'u (DGL'siz) |
| `fgcl_bench/runner_fedspa.py` | FedSPA: görev akışında ardışık eğitim (FGL ince ayar baseline'ı) |
| `fgcl_bench/run.py` | tek komutla koşu: python -m fgcl_bench.run --method ... --dataset ... --seed ... |
| `fgcl_bench/equivalence.py` | vektörizasyon eşdeğerlik testi (orijinal kod git geçmişinden okunur) |
| `fgcl_bench/run_main_sequential.py` | sağlama: orijinal main.py'yi tek iş parçacıklı jeneratör eğitimiyle çalıştırır |
| `fgcl_bench/fedspa_dgl_check.py` | FedSPA port'u ile orijinal DGL'li kodun karşılaştırılması (DGL gerektirir; Colab'da çalıştırılmaz) |
| `fgcl_bench/shims/openfgl/__init__.py` | POWER için OpenFGL'den tek fonksiyon |
| `fgcl_bench/shims/openfgl/utils/basic_utils.py` | idx_to_mask_tensor (OpenFGL'den aynen) |

In [ ]:
%%writefile fgcl_bench/__init__.py
"""FGCL karşılaştırma harness'i (FedFST / POWER / FedSPA)."""


In [ ]:
%%writefile fgcl_bench/common.py
"""Harness'in ortak parçaları: veri seti tanımları, yollar, seed, hash, JSON."""
import hashlib
import importlib.util
import json
import os
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

# Veri seti -> FedFST'deki adı ve görev başına sınıf sayısı (Minesweeper: 1, bkz. notebook).
DATASETS = {
    "cora":        dict(title="Cora",        fedfst_name="cora",        per_task=2),
    "citeseer":    dict(title="CiteSeer",    fedfst_name="citeseer",    per_task=2),
    "actor":       dict(title="Actor",       fedfst_name="actor",       per_task=2),
    "squirrel":    dict(title="Squirrel",    fedfst_name="squirrel",    per_task=2),
    "minesweeper": dict(title="Minesweeper", fedfst_name="minesweeper", per_task=1),
}
METHODS = {"fedfst": "FedFST", "power": "POWER", "fedspa": "FedSPA"}
NUM_CLIENTS = 3


def run_name(dataset, method, seed):
    return f"{dataset}__{method}__seed{seed}"


def set_seed(seed):
    """FedFST'nin utils.set_seed'i ile aynı (POWER'ın seed_everything'i de aynı içerikte)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False
    torch.set_num_threads(1)
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'


def pick_device(prefer="auto"):
    if prefer == "cpu" or not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device("cuda:0")


def device_label(device):
    if device.type == "cuda":
        return f"cuda:0 ({torch.cuda.get_device_name(0)})"
    return "cpu"


def load_module_from_file(alias, path):
    """Bir .py dosyasını farklı bir modül adıyla yükler (ör. FedFST utils.py -> fedfst_utils),
    böylece POWER'ın kendi `utils` modülüyle çakışmaz."""
    spec = importlib.util.spec_from_file_location(alias, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[alias] = module
    spec.loader.exec_module(module)
    return module


def _tensor_bytes(t):
    t = t.detach().cpu().contiguous()
    return str(t.dtype).encode() + str(tuple(t.shape)).encode() + t.numpy().tobytes()


def task_view_hash(views):
    """views[c][t] = dict(train, valid, test, nodes, edges, y): düğümler istemci-içi numaralarla,
    train/valid/test/nodes sıralı LongTensor, edges [2, E] (sıra korunur), y = nodes'un etiketleri.
    Üç yöntem de beslediği veriden bu yapıyı kurup hash'i cache'teki ile karşılaştırır."""
    h = hashlib.sha256()
    for c, tasks in enumerate(views):
        for t, v in enumerate(tasks):
            h.update(f"client{c}/task{t}".encode())
            for key in ("train", "valid", "test", "nodes", "edges", "y"):
                h.update(key.encode())
                h.update(_tensor_bytes(v[key]))
    return h.hexdigest()


def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False)
    os.replace(tmp, path)


def env_info(device):
    import torch_geometric
    return {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torch_geometric": torch_geometric.__version__,
        "numpy": np.__version__,
        "device": device_label(device),
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    }


In [ ]:
%%writefile fgcl_bench/data.py
"""Veri katmanı: ham dosyalar, FedFST fonksiyonlarıyla bölümleme/görevler ve diske cache.

Bölümleme ve görevler, FedFST'nin main.py'sindeki sırayla ve FedFST'nin kendi
fonksiyonlarıyla üretilir:
    set_seed(seed) -> load_dataset -> louvain_partitioner (python-louvain, numpy RNG)
    -> sınıf sırası: random.seed(seed); random.shuffle (load_server_clients'taki gibi)
    -> her istemci için class_to_task (sınıf başına 0.6/0.2/0.2, numpy RNG)
main.py'de bu adımların arasında yalnızca model ağırlıkları başlatılır (torch RNG);
numpy/random akışını etkilemedikleri için maskeler main.py'dekiyle aynıdır. FedFST
koşucusu (runner_fedfst.py) bu sırayı modellerle birlikte yeniden çalıştırır ve
maskelerin cache'tekiyle aynı olduğunu hash ile doğrular.

Komut satırı:  python -m fgcl_bench.data --fedfst <dir> --work_dir <dir> --dataset cora --seed 24
"""
import argparse
import random
import sys
import time
import urllib.request
from pathlib import Path

import torch

from .common import DATASETS, NUM_CLIENTS, task_view_hash

GH = "https://raw.githubusercontent.com"
GEOM = f"{GH}/graphdml-uiuc-jlu/geom-gcn"
GEOM_SQUIRREL = f"{GEOM}/f1fc0d14b3b019c562737240d06ec83b07d16a8f"  # PyG'nin WikipediaNetwork'te sabitlediği commit


def raw_files(dataset):
    """(kök altındaki göreli yol, URL) çiftleri; PyG'nin beklediği klasör düzeninde."""
    if dataset in ("cora", "citeseer"):
        return [(f"{dataset}/raw/ind.{dataset}.{s}", f"{GH}/kimiyoung/planetoid/master/data/ind.{dataset}.{s}")
                for s in ("x", "tx", "allx", "y", "ty", "ally", "graph", "test.index")]
    if dataset == "actor":
        files = [(f"raw/{f}", f"{GEOM}/master/new_data/film/{f}")
                 for f in ("out1_node_feature_label.txt", "out1_graph_edges.txt")]
        return files + [(f"raw/film_split_0.6_0.2_{i}.npz", f"{GEOM}/master/splits/film_split_0.6_0.2_{i}.npz")
                        for i in range(10)]
    if dataset == "squirrel":
        files = [(f"squirrel/geom_gcn/raw/{f}", f"{GEOM_SQUIRREL}/new_data/squirrel/{f}")
                 for f in ("out1_node_feature_label.txt", "out1_graph_edges.txt")]
        return files + [(f"squirrel/geom_gcn/raw/squirrel_split_0.6_0.2_{i}.npz",
                         f"{GEOM_SQUIRREL}/splits/squirrel_split_0.6_0.2_{i}.npz") for i in range(10)]
    if dataset == "minesweeper":
        return [("minesweeper/raw/minesweeper.npz",
                 f"{GH}/yandex-research/heterophilous-graphs/main/data/minesweeper.npz")]
    raise KeyError(dataset)


def prefetch_raw(data_root, dataset, retries=4):
    """Eksik ham dosyaları indirir. PyG de aynı dosyaları indirebilir; burada önceden indirip
    PyG'nin indirme adımını atlamasını sağlıyoruz (github.com/.../raw yönlendirmesi yerine
    doğrudan raw.githubusercontent.com). İndirme başarısız olursa PyG kendi indirmesini dener."""
    data_root = Path(data_root)
    for rel, url in raw_files(dataset):
        dest = data_root / rel
        if dest.exists() and dest.stat().st_size > 0:
            continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        for attempt in range(retries):
            try:
                tmp = dest.with_suffix(dest.suffix + ".part")
                urllib.request.urlretrieve(url, tmp)
                tmp.replace(dest)
                break
            except Exception as exc:  # ağ hatası: kısa bekleyip tekrar dene
                if attempt == retries - 1:
                    print(f"[uyarı] {url} indirilemedi ({exc}); PyG kendi indirmesini deneyecek.")
                else:
                    time.sleep(2 ** (attempt + 1))


def cache_path(work_dir, dataset, seed):
    return Path(work_dir) / "cache" / f"{dataset}_seed{seed}.pt"


def views_from_cache(cache):
    views = []
    for cl in cache["clients"]:
        tv = []
        for task in cl["tasks"]:
            nodes = torch.sort(task["nodes"])[0]
            tv.append(dict(train=task["train_mask"].nonzero().view(-1),
                           valid=task["valid_mask"].nonzero().view(-1),
                           test=task["test_mask"].nonzero().view(-1),
                           nodes=nodes, edges=task["edge_index"], y=cl["y"][nodes]))
        views.append(tv)
    return views


def build_cache(fedfst_dir, data_root, dataset, seed):
    sys.path.insert(0, str(fedfst_dir))
    from utils import set_seed                                     # FedFST
    from datasets.dataset_loader import load_dataset, class_to_task  # FedFST (yamalı)
    from datasets.partition import louvain_partitioner             # FedFST (yamalı)
    from torch_geometric.utils import homophily

    info = DATASETS[dataset]
    k = info["per_task"]
    t0 = time.time()
    set_seed(seed)
    data, in_dim, out_dim = load_dataset(str(data_root), info["fedfst_name"], k, None)
    y_original = data.y.clone()
    stats = dict(nodes=int(data.num_nodes), edges=int(data.edge_index.size(1)), features=int(data.x.size(1)),
                 classes=int(data.y.max()) + 1,
                 edge_homophily=float(homophily(data.edge_index, data.y, method="edge")))

    clients_data = louvain_partitioner(data, NUM_CLIENTS)
    classes_num = data.y.max().item() + 1
    classes_order = list(range(classes_num))
    random.seed(seed)                      # algorithm/Ours.py: load_server_clients (shuffle_flag=True)
    random.shuffle(classes_order)

    clients = []
    for cid, cdata in enumerate(clients_data):
        tasks = class_to_task(data=cdata, per_task_class_num=k, train_prop=0.6, valid_prop=0.2,
                              test_prop=0.2, shuffle_flag=True, classes_order=classes_order)
        ctasks = []
        for task in tasks:
            ctasks.append(dict(train_mask=task["train_mask"].clone(), valid_mask=task["valid_mask"].clone(),
                               test_mask=task["test_mask"].clone(),
                               nodes=torch.tensor([int(v) for v in task["local_data"].global_map], dtype=torch.long),
                               edge_index=task["local_data"].edge_index.contiguous().clone()))
        clients.append(dict(x=cdata.x.clone(), y=cdata.y.clone(), edge_index=cdata.edge_index.contiguous().clone(),
                            tasks=ctasks))

    num_tasks = classes_num // k
    task_classes = [[classes_order[t * k + i] for i in range(k)] for t in range(num_tasks)]
    dropped = [classes_order[j] for j in range(num_tasks * k, classes_num)]
    stats["clients"] = [dict(nodes=int(cl["x"].size(0)), edges=int(cl["edge_index"].size(1)),
                             tasks=[dict(nodes=int(t["nodes"].numel()), train=int(t["train_mask"].sum()),
                                         valid=int(t["valid_mask"].sum()), test=int(t["test_mask"].sum()),
                                         edges=int(t["edge_index"].size(1))) for t in cl["tasks"]])
                        for cl in clients]
    cache = dict(meta=dict(dataset=dataset, seed=seed, num_clients=NUM_CLIENTS, per_task=k, num_tasks=num_tasks,
                           num_classes=classes_num, in_dim=int(in_dim), out_dim=int(out_dim),
                           classes_order=classes_order, task_classes=task_classes, dropped_classes=dropped,
                           stats=stats, build_seconds=round(time.time() - t0, 2)),
                 clients=clients, y_original=y_original)
    cache["hash"] = task_view_hash(views_from_cache(cache))
    return cache


def load_cache(work_dir, dataset, seed):
    return torch.load(cache_path(work_dir, dataset, seed), weights_only=False)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--fedfst", required=True)
    ap.add_argument("--work_dir", required=True)
    ap.add_argument("--dataset", required=True, choices=list(DATASETS))
    ap.add_argument("--seed", type=int, default=24)
    args = ap.parse_args(argv)
    data_root = Path(args.work_dir) / "data"
    out = cache_path(args.work_dir, args.dataset, args.seed)
    if out.exists():
        cache = torch.load(out, weights_only=False)
        print(f"cache mevcut: {out} (hash {cache['hash'][:12]})")
        return 0
    prefetch_raw(data_root, args.dataset)
    cache = build_cache(args.fedfst, data_root, args.dataset, args.seed)
    out.parent.mkdir(parents=True, exist_ok=True)
    tmp = out.with_suffix(".pt.tmp")
    torch.save(cache, tmp)
    tmp.replace(out)
    m = cache["meta"]
    print(f"cache yazıldı: {out} | görev={m['num_tasks']} atılan sınıf={m['dropped_classes']} "
          f"| istemci düğümleri={[c['nodes'] for c in m['stats']['clients']]} | hash {cache['hash'][:12]} "
          f"| {m['build_seconds']} sn")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile fgcl_bench/evaluate.py
"""Ortak değerlendirme: her görevden sonra global modelin o ana kadarki tüm görevlerdeki test
doğruluğu -> doğruluk matrisi -> AA ve AF.

Aritmetik, FedFST main.py'deki değerlendirme bloğunun aynısıdır:
    ACC[t, e] += client_acc * n_test   (client_acc = doğru / n_test * 100, Python float)
    ACC[t, e] /= toplam n_test
AA ve AF, FedFST'nin utils.AA / utils.AF fonksiyonlarıyla hesaplanır (dosya `fedfst_utils`
adıyla yüklenir; POWER'ın kendi `utils` modülüyle çakışmasın diye).
"""
from pathlib import Path

import torch

from .common import load_module_from_file


class Evaluator:
    def __init__(self, cache, fedfst_dir):
        self.fedfst_utils = load_module_from_file("fedfst_utils", Path(fedfst_dir) / "utils.py")
        self.num_tasks = cache["meta"]["num_tasks"]
        self.acc = torch.zeros(size=(self.num_tasks, self.num_tasks))
        # test düğümleri (istemci-içi numara, sıralı) ve etiketleri
        self.test_idx = [[t["test_mask"].nonzero().view(-1) for t in cl["tasks"]] for cl in cache["clients"]]
        self.test_y = [[cl["y"][idx] for idx in tests] for cl, tests in zip(cache["clients"], self.test_idx)]
        self.test_count = [[t["test_mask"].sum() for t in cl["tasks"]] for cl in cache["clients"]]

    def evaluate_after_task(self, task_id, predict_fn):
        """predict_fn(client_id, eval_task_id) -> tahmin edilen etiketler; sırası self.test_idx[c][e] ile aynı."""
        for eval_task_id in range(task_id + 1):
            total_nodes_num = 0
            for client_id in range(len(self.test_idx)):
                nodes_num = self.test_count[client_id][eval_task_id]
                if int(nodes_num) == 0:
                    continue
                pred = predict_fn(client_id, eval_task_id).detach().cpu()
                correct = pred.eq(self.test_y[client_id][eval_task_id]).sum().item()
                client_acc = (correct / nodes_num.item()) * 100
                self.acc[task_id, eval_task_id] += client_acc * nodes_num
                total_nodes_num += nodes_num
            self.acc[task_id, eval_task_id] /= total_nodes_num
            print(f"Task {task_id} trained, Task {eval_task_id} global acc: {self.acc[task_id, eval_task_id]:.2f}")
        aa = self.fedfst_utils.AA(self.acc, T=task_id + 1)
        af = self.fedfst_utils.AF(self.acc, T=task_id + 1)
        aa, af = float(aa), float(af)
        print(f"Task {task_id} - AA: {aa:.4f}, AF: {af:.4f}")
        return aa, af

    def final(self):
        aa = float(self.fedfst_utils.AA(self.acc, T=self.num_tasks))
        af = float(self.fedfst_utils.AF(self.acc, T=self.num_tasks))
        return aa, af

    def matrix(self):
        return [[round(float(v), 6) for v in row] for row in self.acc]


In [ ]:
%%writefile fgcl_bench/runner_fedfst.py
"""FedFST koşucusu: main.py'nin eğitim döngüsü, resmi (yamalı) Ours.py sınıflarıyla.

main.py'ye göre yalnızca üç fark var (eğitim mantığı aynı):
  1. Veri/bölümleme main.py'deki sırayla üretilir ve maskelerin ortak cache'le aynı olduğu
     hash ile doğrulanır.
  2. main.py istemcilerin jeneratör eğitimini ThreadPoolExecutor ile eşzamanlı çalıştırıyor.
     Üç iş parçacığı aynı küresel RNG'yi paylaştığı için sonuç aynı seed ile bile
     tekrarlanamıyor. Burada aynı adımlar aynı sırayla (istemci 0, 1, 2) ardışık çalışır.
  3. Değerlendirme ortak Evaluator ile yapılır. Karşılaştırma için FedFST'nin kendi
     değerlendirmesi de (client.evaluate, global_flag=True) aynı yerde çalıştırılıp
     sonuçları JSON'a "acc_matrix_native" olarak yazılır.
Kayıp grafikleri (plot_all_losses) algoritmanın parçası olmadığı için çizilmez.
"""
import sys
import time

import torch

from .common import DATASETS, task_view_hash


def fedfst_args(cfg):
    from args import parser  # FedFST'nin args.py'si
    info = DATASETS[cfg.dataset]
    argv = ["--dataset_name", info["fedfst_name"], "--seed", str(cfg.seed), "--device_id", "0",
            "--per_task_class_num", str(info["per_task"]), "--clients_num", "3",
            "--rounds", str(cfg.rounds), "--local_epochs", str(cfg.local_epochs),
            "--dataset_dir", str(cfg.data_root), "--save_dir", str(cfg.tmp_dir)]
    if cfg.smoke:
        argv += ["--gen_epochs", "2", "--kd_epochs", "3"]
    return parser.parse_args(argv)


def views_from_fedfst_clients(clients):
    views = []
    for client in clients:
        tv = []
        for task in client.tasks:
            nodes = torch.sort(torch.tensor([int(v) for v in task["local_data"].global_map], dtype=torch.long))[0]
            tv.append(dict(train=task["train_mask"].nonzero().view(-1), valid=task["valid_mask"].nonzero().view(-1),
                           test=task["test_mask"].nonzero().view(-1), nodes=nodes,
                           edges=task["local_data"].edge_index.detach().cpu().contiguous(),
                           y=client.data.y.detach().cpu()[nodes]))
        views.append(tv)
    return views


def run(cfg, cache, evaluator):
    sys.path.insert(0, str(cfg.fedfst_dir))
    from algorithm.Ours import load_server_clients
    from datasets.partition import louvain_partitioner
    from datasets.dataset_loader import load_dataset
    from utils import set_seed

    args = fedfst_args(cfg)
    # ---- main.py, satır 23-58 (aynı sıra) ----
    print(f"Seed: {args.seed}, Dataset: {args.dataset_name}, Clients: {args.clients_num}, Model: {args.model}")
    print(f"gen_lr={args.gen_lr}, kd_lr={args.kd_lr}, kd_khop={args.kd_khop}, gen_epochs={args.gen_epochs}, kd_epochs={args.kd_epochs}, lr={args.lr}")
    print(f"gen_rounds={args.gen_rounds}, tolerance={args.tolerance}, max_iterations={args.max_iterations}, gen_num_nodes={args.gen_num_nodes}, per_task_class_num={args.per_task_class_num}")
    print(f"kd_ce_weight={args.kd_ce_weight}, kd_low_weight={args.kd_low_weight}, temperature={args.kd_temperature}")
    print(f"gen_ce_weight={args.gen_ce_weight}, gen_kl_weight={args.gen_kl_weight}")
    print(f"S_high_x_weight={args.S_high_x_weight}, S_high_A_weight={args.S_high_A_weight}")

    set_seed(args.seed)

    if args.use_gpu:
        device = torch.device(f"cuda:{args.device_id}" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device("cpu")
    print(f"Using device: {device}")

    data, input_dim, out_dim = load_dataset(args.dataset_dir, args.dataset_name, args.per_task_class_num, args)
    args.input_dim = input_dim
    args.output_dim = out_dim
    print(f"Nodes: {data.num_nodes}, Edges: {data.num_edges}, Features: {tuple(data.x.shape)}, Classes: {data.y.max().item() + 1}")

    clients_data = louvain_partitioner(data, args.clients_num)
    args.input_dim = input_dim
    args.out_dim = out_dim
    classes_num = data.y.max().item() + 1
    server, clients, message_pool = load_server_clients(args, clients_data, device, classes_num)

    tasks_num = (data.y.max().item() + 1) // args.per_task_class_num
    print(f"Total number of tasks: {tasks_num}")

    used_hash = task_view_hash(views_from_fedfst_clients(clients))
    assert used_hash == cache["hash"], "FedFST'nin ürettiği görev maskeleri ortak cache ile aynı değil!"
    print(f"[harness] görev maskeleri cache ile aynı (hash {used_hash[:12]})")
    assert tasks_num == cache["meta"]["num_tasks"]

    native_acc = torch.zeros(size=(tasks_num, tasks_num)).to(device)
    task_seconds = []

    def predict(client_id, eval_task_id):
        model = server.global_model
        was_training = model.training
        model.eval()
        data_t = clients[client_id].tasks[eval_task_id]["local_data"].to(device)
        with torch.no_grad():
            _, out = model(data_t)
        model.train(was_training)
        _, pred = out.max(dim=1)
        return pred[evaluator.test_idx[client_id][eval_task_id].to(pred.device)]

    for task_id in range(tasks_num):
        t_task = time.time()
        print(f"\n========== Task {task_id}/{tasks_num} ==========")

        for round in range(args.rounds):
            print(f"GNN Round {round}")
            print(f"Training clients for task {task_id}...")
            client_losses = []
            for client_id, client in enumerate(clients):
                loss = client.train(task_id)
                client_losses.append(loss)
                print(f"Client {client_id} loss: {loss:.4f}")

            avg_client_loss = sum(client_losses) / len(client_losses)
            print(f"Average client loss: {avg_client_loss:.4f}")

            for client_id, client in enumerate(clients):
                client.send_message(task_id)

            server.clients_nodes_num = [server.message_pool[f"client_{client_id}"]["nodes_num"]
                                        for client_id in range(args.clients_num)]
            server.clients_learned_nodes_num = [server.message_pool[f"client_{client_id}"]["learned_nodes_num"]
                                                for client_id in range(args.clients_num)]

            print("Server aggregating client models...")
            server.aggregate()
            server.send_message()

        if task_id != 0:
            server.feature_gen_init(task_id = task_id)
            for client_id, client in enumerate(clients):
                client.feature_gen_init(task_id = task_id)

            for round in range(args.gen_rounds):
                print(f"Generator Round {round}")
                # main.py: ThreadPoolExecutor ile eşzamanlı. Burada aynı çağrılar aynı sırayla ardışık.
                for client_id, client in enumerate(clients):
                    client.feature_gen_train(task_id, round != 0)

                for client_id, client in enumerate(clients):
                    client.send_feature_gen(task_id)

                server.feature_gen_aggregate()
                server.send_feature_gen()

            server.KD_train(task_id)

        server.update_last_global_model()
        for client_id, client in enumerate(clients):
            client.last_global_model.load_state_dict(server.last_global_model.state_dict())

        for client_id, client in enumerate(clients):
            client.save_validation_acc(task_id)

        # --- FedFST'nin kendi değerlendirmesi (main.py satır 122-132), karşılaştırma için ---
        for eval_task_id in range(0, task_id + 1):
            total_nodes_num = 0
            for client_id in range(args.clients_num):
                nodes_num = clients[client_id].tasks[eval_task_id]["test_mask"].sum()
                if int(nodes_num) == 0:
                    continue
                evaluation = clients[client_id].evaluate(task_id = eval_task_id, global_flag=True)
                native_acc[task_id, eval_task_id] += evaluation["acc"] * nodes_num
                total_nodes_num += nodes_num
            native_acc[task_id, eval_task_id] /= total_nodes_num

        # --- ortak değerlendirme ---
        evaluator.evaluate_after_task(task_id, predict)
        task_seconds.append(_r2(time.time() - t_task))

    native = native_acc.cpu()
    max_diff = float((native - evaluator.acc).abs().max())
    print(f"[harness] ortak değerlendirme ile FedFST'nin kendi değerlendirmesi arasındaki en büyük fark: {max_diff:.2e}")
    return dict(task_seconds=task_seconds,
                acc_matrix_native=[[_r6(v) for v in row] for row in native],
                native_max_abs_diff=max_diff,
                hyperparameters={k: getattr(args, k) for k in (
                    "lr", "weight_decay", "hidden_dim", "dropout", "model", "rounds", "local_epochs",
                    "gen_rounds", "gen_epochs", "gen_lr", "kd_epochs", "kd_lr", "kd_khop", "max_iterations",
                    "tolerance", "gen_num_nodes", "num_samples_per_class", "per_task_class_num")},
                mask_hash=used_hash)


def _r2(x):
    return round(x, 2)


def _r6(v):
    return round(float(v), 6)


In [ ]:
%%writefile fgcl_bench/runner_power.py
"""POWER koşucusu: resmi kod (Zenodo 10.5281/zenodo.15487654, zyl24-FCGL_POWER-78e30b6) + adaptör.

Algoritma kodu (flcore/Ours.py: OursClient, OursServer, GEModel; model.py; utils.py) olduğu gibi
import edilir. Adaptör yalnızca veri arayüzünü değiştirir:
  * POWER'ın veri katmanı (data.py, OpenFGL'in FGLDataset'i ve class_incremental) kullanılmaz;
    istemci grafikleri, görevler ve maskeler ortak cache'ten gelir.
  * OursClient.task_data, görev grafiğini ortak protokoldeki görev alt-grafiği olarak döndürür
    (FedFST'nin görev grafiğiyle aynı kenarlar). Orijinal task_data, edge_masking ile kenarları
    maskeleyip diske yazıyordu (bkz. notebook: edge_masking'in davranışı).
  * flcore/Ours.py'nin OpenFGL'den kullandığı tek fonksiyon (idx_to_mask_tensor) shims/ altında.
Eğitim döngüsü, POWER utils.start()'ın döngüsüdür; değerlendirme ortak Evaluator'la yapılır.
Karşılaştırma için POWER'ın kendi global değerlendirmesi (evaluate(use_global=True)) de
çalıştırılıp "acc_matrix_native" olarak kaydedilir.
"""
import sys
import time
from pathlib import Path

import torch
from torch_geometric.data import Data

from .common import NUM_CLIENTS, set_seed, pick_device, task_view_hash


def load_power_args(power_dir):
    saved = sys.argv
    sys.argv = [saved[0]]          # config.py içe aktarılırken argparse çalıştırıyor -> varsayılanlar
    try:
        from config import args
    finally:
        sys.argv = saved
    return args


def run(cfg, cache, evaluator):
    power_dir = Path(cfg.power_dir)
    sys.path.insert(0, str(power_dir))
    sys.path.insert(1, str(Path(__file__).resolve().parent / "shims"))
    args = load_power_args(power_dir)
    from flcore.Ours import OursClient, OursServer
    import flcore.Ours as power_ours
    assert Path(power_ours.__file__).resolve().is_relative_to(power_dir.resolve()), power_ours.__file__

    meta = cache["meta"]
    args.num_clients = NUM_CLIENTS
    args.num_classes_per_task = meta["per_task"]
    args.dataset = cfg.dataset
    args.seed = cfg.seed
    args.num_rounds_per_task = cfg.rounds
    args.num_epochs = cfg.local_epochs
    args.input_dim = meta["in_dim"]
    args.output_dim = meta["out_dim"]
    args.task_dir = str(cfg.tmp_dir)
    args.global_eval = True   # bilgi amaçlı: ortak değerlendirme global modeli kullanıyor
    if cfg.smoke:
        args.num_it_recon = 3
        args.num_epoch_g = 2
    print("POWER args:", {k: getattr(args, k) for k in sorted(vars(args))})

    set_seed(cfg.seed)
    device = pick_device()
    print(f"Using device: {device}")

    class HarnessPowerClient(OursClient):
        """OursClient'ın yalnızca görev grafiğini kuran task_data yöntemi değiştirildi."""

        def task_data(self, task_id, whole_data, task):
            return Data(x=whole_data.x, edge_index=self.harness_task_edges[task_id].to(self.device), y=whole_data.y)

    fcgl_dataset = {}
    for cid, cl in enumerate(cache["clients"]):
        fcgl_dataset[cid] = {
            "data": Data(x=cl["x"].clone(), edge_index=cl["edge_index"].clone(), y=cl["y"].clone()),
            "task": [{"train_mask": t["train_mask"].clone(), "val_mask": t["valid_mask"].clone(),
                      "test_mask": t["test_mask"].clone()} for t in cl["tasks"]],
        }

    # POWER utils.load_clients_server ile aynı sıra: önce istemciler, sonra sunucu
    message_pool = {}
    clients = [HarnessPowerClient(args, client_id, fcgl_dataset[client_id], message_pool, device)
               for client_id in range(args.num_clients)]
    for cid, client in enumerate(clients):
        client.harness_task_edges = [t["edge_index"].clone() for t in cache["clients"][cid]["tasks"]]
    server = OursServer(args, message_pool, device)

    views = []
    for cid, client in enumerate(clients):
        tv = []
        for t, task in enumerate(client.data["task"]):
            nodes = torch.sort(cache["clients"][cid]["tasks"][t]["nodes"])[0]
            tv.append(dict(train=task["train_mask"].nonzero().view(-1), valid=task["val_mask"].nonzero().view(-1),
                           test=task["test_mask"].nonzero().view(-1), nodes=nodes,
                           edges=client.harness_task_edges[t].contiguous(), y=client.data["data"].y[nodes]))
        views.append(tv)
    used_hash = task_view_hash(views)
    assert used_hash == cache["hash"], "POWER'a verilen görevler ortak cache ile aynı değil!"
    print(f"[harness] görev maskeleri cache ile aynı (hash {used_hash[:12]})")

    def predict(client_id, eval_task_id):
        model = server.global_model
        was_training = model.training
        model.eval()
        whole_data = clients[client_id].data["data"].to(device)
        task = clients[client_id].data["task"][eval_task_id]
        task_data = clients[client_id].task_data(eval_task_id, whole_data, task)
        with torch.no_grad():
            logits, _, _ = model.forward(task_data)
        model.train(was_training)
        pred = logits.max(1)[1]
        return pred[evaluator.test_idx[client_id][eval_task_id].to(pred.device)]

    # ---- POWER utils.start() ----
    print("-" * 50)
    for client_id in range(args.num_clients):
        result_list = []
        for task_id in range(len(clients[client_id].data["task"])):
            label = clients[client_id].data["data"].y
            task = clients[client_id].data["task"][task_id]
            node_mask = task["train_mask"] | task["val_mask"] | task["test_mask"]
            result = torch.unique(label[node_mask]).cpu().tolist()
            result_list.append(str(result))
        print(f"client {client_id} tasks: {'=>'.join(result_list)}")
    print("-" * 50)
    num_tasks = len(fcgl_dataset[0]["task"])
    args.num_tasks = num_tasks
    native_acc = torch.zeros(size=(num_tasks, num_tasks)).to(device)
    task_seconds = []

    for task_id in range(num_tasks):
        t_task = time.time()
        native_acc[task_id, :] = 0
        message_pool["task_id"] = task_id
        for client_id in range(args.num_clients):
            clients[client_id].task_start(task_id)
        server.task_start(task_id)
        for round_id in range(args.num_rounds_per_task):
            message_pool["round_id"] = round_id
            server.send_message()
            for client_id in range(args.num_clients):
                clients[client_id].execute(task_id)
                clients[client_id].send_message(task_id)
            server.execute()

        # POWER'ın kendi global değerlendirmesi (karşılaştırma için)
        for eval_task_id in range(0, task_id + 1):
            total_nodes = 0
            for client_id in range(args.num_clients):
                num_nodes = clients[client_id].data["task"][eval_task_id]["test_mask"].sum()
                if int(num_nodes) == 0:
                    continue
                client_acc = clients[client_id].evaluate(task_id=eval_task_id, use_global=True)
                native_acc[task_id, eval_task_id] += client_acc * num_nodes
                total_nodes += num_nodes
            native_acc[task_id, eval_task_id] /= total_nodes
        # ortak değerlendirme
        evaluator.evaluate_after_task(task_id, predict)

        for client_id in range(args.num_clients):
            clients[client_id].task_done(task_id)
        server.task_done(task_id)
        task_seconds.append(round(time.time() - t_task, 2))
        print("-" * 50)

    native = native_acc.cpu()
    max_diff = float((native - evaluator.acc).abs().max())
    print(f"[harness] ortak değerlendirme ile POWER'ın kendi global değerlendirmesi arasındaki en büyük fark: {max_diff:.2e}")
    return dict(task_seconds=task_seconds,
                acc_matrix_native=[[round(float(v), 6) for v in row] for row in native],
                native_max_abs_diff=max_diff,
                hyperparameters={k: getattr(args, k) for k in (
                    "model", "hid_dim", "dropout", "lr", "optim", "weight_decay", "num_rounds_per_task", "num_epochs",
                    "alpha", "replay", "num_samples_per_class", "lr_g", "LBFGS_init_lr", "num_it_recon", "beta",
                    "num_recon_nodes", "decay", "num_epoch_g")},
                mask_hash=used_hash)


In [ ]:
%%writefile fgcl_bench/fedspa_port.py
"""FedSPA (CVPR 2025; OakleyTan/FedSPA @ d2b3ddb): SFPD + HBDA'nın harness'e port'u.

Orijinal kodun ilgili parçaları buraya kopyalandı; değişen yerler "[port]" ile işaretli.
Kaynak dosyalar (Code/ altında):
  benchmark001/base.py            : Seq, MLP_spa                                   (aynen)
  benchmark001/config.py          : freeze_parameters, normalize_tensor, mp_func_spa,
                                    agg_func_spa, GraphConvolution, ACM, get_spa
  benchmark001/model/default_model.py : Model (get_spa(in, 128, out, 2))
  flgo/utils/fmodule.py           : FModule.has_nan                                (aynen)
  flgo/algorithm/fedbase.py       : calculate_shigh                                (aynen)
  flgo/algorithm/fedspa.py        : fish_cal, calculate_fisher_average, normalize_list,
                                    sum_of_layer_fisher_differences, normalize,
                                    calculate_weights (aynen), Server.aggregate, Client.train
  flgo/experiment/logger/simple_logger.py : adj_low_un / adj_low / adj_high (log_once)
  flgo/benchmark/toolkits/graph/node_classification/__init__.py : GeneralCalculator.compute_loss
Değişiklikler:
  1. DGL: graph.update_all(mp_func_spa, agg_func_spa) -> spa_update_all (index_add_).
     Kenar türü (0: iki uç da etiketli ve aynı sınıf, 1: etiketli ve farklı sınıf,
     2: en az bir uç etiketsiz) mp_func_spa'daki kuralın aynısı; her hedef düğümde
     gelen kenarların kaynak öznitelikleri türe göre toplanır (agg_func_spa).
  2. DGL: g.adjacency_matrix() -> yoğun A[src, dst] = 1 (DGL 2.1'de satır = kaynak).
  3. Sabit 'cuda' / 'cuda:0' -> device parametresi; sabit boyutlar (1703, 5) -> veri setinden.
  4. att_struc_low orijinalde ilklenmemiş bellek (torch.FloatTensor) -> sıfır. İleri geçişte
     kullanılmıyor; yalnızca rastgele bellek içeriğinin (NaN olabilir) has_nan kontrolünü
     tetiklemesini önler. RNG tüketimi değişmez.
  5. Kenarı olmayan grafikte orijinal convert_to_dglgraph boş tensörün max'ını alıp çöküyor;
     burada düğüm sayısı doğrudan x'ten alınır.
"""
import collections
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from torch.nn.modules.module import Module
from torch.nn.parameter import Parameter
from torch_geometric.nn.norm import GraphNorm


# =============================================================== benchmark001/base.py (aynen)
class Seq(nn.Module):
    def __init__(self, modlist):
        super().__init__()
        self.modlist = nn.ModuleList(modlist)

    def forward(self, *args, **kwargs):
        out = self.modlist[0](*args, **kwargs)
        for i in range(1, len(self.modlist)):
            out = self.modlist[i](out)
        return out


class MLP_spa(nn.Module):
    def __init__(self,
                 input_channels: int,
                 hidden_channels: int,
                 output_channels: int,
                 num_layers: int,
                 dropout=0,
                 tail_activation=False,
                 activation=nn.ReLU(inplace=True),
                 gn=False):
        super().__init__()
        modlist = []
        self.seq = None
        if num_layers == 1:
            modlist.append(nn.Linear(input_channels, output_channels))
            if tail_activation:
                if gn:
                    modlist.append(GraphNorm(output_channels))
                if dropout > 0:
                    modlist.append(nn.Dropout(p=dropout, inplace=True))
                modlist.append(activation)
            self.seq = Seq(modlist)
        else:
            modlist.append(nn.Linear(input_channels, hidden_channels))
            for _ in range(num_layers - 2):
                if gn:
                    modlist.append(GraphNorm(hidden_channels))
                if dropout > 0:
                    modlist.append(nn.Dropout(p=dropout, inplace=True))
                modlist.append(activation)
                modlist.append(nn.Linear(hidden_channels, hidden_channels))
            if gn:
                modlist.append(GraphNorm(hidden_channels))
            if dropout > 0:
                modlist.append(nn.Dropout(p=dropout, inplace=True))
            modlist.append(activation)
            modlist.append(nn.Linear(hidden_channels, output_channels))
            if tail_activation:
                if gn:
                    modlist.append(GraphNorm(output_channels))
                if dropout > 0:
                    modlist.append(nn.Dropout(p=dropout, inplace=True))
                modlist.append(activation)
            self.seq = Seq(modlist)

    def forward(self, x):
        return self.seq(x)


# =============================================================== benchmark001/config.py
def freeze_parameters(parameters):
    for param in parameters:
        if isinstance(param, nn.Parameter):
            param.requires_grad = False


def normalize_tensor(mx, eqvar=None):
    rowsum = torch.sum(mx, 1)
    if eqvar:
        r_inv = torch.pow(rowsum, -1 / eqvar).flatten()
        r_inv[torch.isinf(r_inv)] = 0.0
        r_mat_inv = torch.diag(r_inv)
        mx = torch.mm(r_mat_inv, mx)
        return mx

    else:
        r_inv = torch.pow(rowsum, -1).flatten()
        r_inv[torch.isinf(r_inv)] = 0.0
        r_mat_inv = torch.diag(r_inv)
        mx = torch.mm(r_mat_inv, mx)
        return mx


class SPAGraph:
    """[port] ACM.preprocess'teki DGL bloğunun yerine: kenar listesi + label_unk + kenar türü.

    Orijinal: g = dgl.graph((src, dst)); g.ndata['label_unk'] = -1 (eğitim düğümlerinde etiket);
    block = dgl.to_block(g, dst_nodes=arange(N)). Kenar türü mp_func_spa'daki kuralla hesaplanır.
    """

    def __init__(self, edge_index, label_unk, num_nodes):
        self.src = edge_index[0]
        self.dst = edge_index[1]
        self.label_unk = label_unk
        self.num_nodes = num_nodes
        # mp_func_spa (aynı kural)
        src_label = label_unk[self.src]
        dst_label = label_unk[self.dst]
        src_fake_label = torch.zeros_like(src_label)
        src_fake_label[(src_label < 0) | (dst_label < 0)] = 2
        condition = (src_label != dst_label) & (src_label >= 0) & (dst_label >= 0)
        src_fake_label[condition] = 1
        self.src_fake_label = src_fake_label


def spa_update_all(graph, h_low, h_high):
    """[port] graph.update_all(mp_func_spa, agg_func_spa) eşdeğeri.

    mp_func_spa her kenar için kaynağın h_low / h_high'ını mesaj yapar (DGL bunu tüm kenarlar
    için index_select ile toplar). agg_func_spa her hedef düğümde gelen mesajları kenar türü
    maskesiyle çarpıp toplar: neigh_hete_* (tür 1), neigh_homo_* (tür 0), neigh_unk_* (tür 2).
    DGL gelen kenarları kenar numarası sırasıyla dizer; index_add_ de kenarları aynı sırayla
    toplar. Gelen mesajı olmayan düğüm DGL'de sıfır alır; index_add_ de sıfırdan başlar."""
    m_low = torch.index_select(h_low, 0, graph.src)
    m_high = torch.index_select(h_high, 0, graph.src)
    out = {}
    for name, code in (("hete", 1), ("homo", 0), ("unk", 2)):
        mask = (graph.src_fake_label == code).unsqueeze(-1)
        for tag, m in (("low", m_low), ("high", m_high)):
            agg = torch.zeros(graph.num_nodes, m.size(1), dtype=m.dtype, device=m.device)
            out[f"neigh_{name}_{tag}"] = agg.index_add(0, graph.dst, m * mask)
    return out


def get_spa(nfeat,
        nhid,
        nclass,
        nlayers,
        device=None):
    return ACM(nfeat,
        nhid,
        nclass,
        nlayers,
        dropout=0.1,
        model_type='acm_spa',
        variant=False,
        init_layers_X=1,
        device=device)


class GraphConvolution(Module):
    def __init__(
        self,
        in_features,
        out_features,
        model_type,
        output_layer=0,
        variant=False,
        device=None,  # [port] orijinalde modül düzeyinde "cuda:0 if available else cpu"
    ):
        super(GraphConvolution, self).__init__()
        device = device if device is not None else torch.device("cpu")
        self.device = device
        (
            self.in_features,
            self.out_features,
            self.output_layer,
            self.model_type,
            self.variant,
        ) = (
            in_features,
            out_features,
            output_layer,
            model_type,
            variant,
        )
        self.bn_low = nn.BatchNorm1d(out_features)
        self.bn_high = nn.BatchNorm1d(out_features)
        self.bn_mlp = nn.BatchNorm1d(out_features)
        self.att_low, self.att_high, self.att_mlp = 0, 0, 0
        self.weight_low, self.weight_high, self.weight_mlp = (
            Parameter(init.xavier_uniform_(torch.FloatTensor(in_features, out_features).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(in_features, out_features).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(in_features, out_features).to(device))),
        )
        self.att_vec_low, self.att_vec_high, self.att_vec_mlp = (
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
        )
        self.layer_norm_low, self.layer_norm_high, self.layer_norm_mlp = (
            nn.LayerNorm(out_features),
            nn.LayerNorm(out_features),
            nn.LayerNorm(out_features),
        )
        self.layer_norm_struc_low, self.layer_norm_struc_high = nn.LayerNorm(
            out_features
        ), nn.LayerNorm(out_features)
        self.att_struc_low = Parameter(
            torch.zeros(1 * out_features, 1).to(device)  # [port] orijinal: ilklenmemiş torch.FloatTensor
        )
        self.att_vec_3 = init.xavier_uniform_(Parameter(torch.FloatTensor(3, 3).to(device)))
        self.att_vec_low_fr, self.att_vec_high_fr, self.att_vec_low_be, self.att_vec_high_be, self.att_vec_low_unk, self.att_vec_high_unk = (
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
            Parameter(init.xavier_uniform_(torch.FloatTensor(1 * out_features, 1).to(device))),
        )

        self.att_vec = init.xavier_uniform_(Parameter(torch.FloatTensor(7, 7).to(device)))

        self.fc_balance = MLP_spa(out_features, hidden_channels=128, output_channels=1,
                              num_layers=1)
        self.balance_w = nn.Sigmoid()
        self.fr_low_g = None
        self.fr_high_g = None
        self.mlp_g = None

    def attention7(self, neigh_hete_low, neigh_hete_high, neigh_homo_low, neigh_homo_high, neigh_unk_low, neigh_unk_high,
                   output_mlp, balance_1=None, balance_2=None, balance_3=None):
        T = 7
        if balance_1 != None:
            logits = (
                    torch.mm(
                        torch.sigmoid(
                            torch.cat(
                                [
                                    torch.mm(neigh_hete_low, self.att_vec_low_fr),
                                    torch.mm(neigh_hete_high, self.att_vec_high_fr),
                                    torch.mm(neigh_homo_low, self.att_vec_low_be),
                                    torch.mm(neigh_homo_high, self.att_vec_high_be),
                                    torch.mm(neigh_unk_low, self.att_vec_low_fr) * balance_1 + torch.mm(neigh_unk_low,
                                                                                                        self.att_vec_low_be) * (
                                                1 - balance_1),
                                    torch.mm(neigh_unk_high, self.att_vec_high_fr) * balance_2 + torch.mm(neigh_hete_high,
                                                                                                          self.att_vec_high_be) * (
                                                1 - balance_2),
                                    torch.mm(output_mlp, self.att_vec_mlp),
                                ],
                                1,
                            )
                        ),
                        self.att_vec,
                    )
                    / T
            )
        else:
            logits = (
                    torch.mm(
                        torch.sigmoid(
                            torch.cat(
                                [
                                    torch.mm(neigh_hete_low, self.att_vec_low_fr),
                                    torch.mm(neigh_hete_high, self.att_vec_high_fr),
                                    torch.mm(neigh_homo_low, self.att_vec_low_be),
                                    torch.mm(neigh_homo_high, self.att_vec_high_be),
                                    torch.mm(neigh_unk_low, self.att_vec_low_unk),
                                    torch.mm(neigh_unk_high, self.att_vec_high_unk),
                                    torch.mm(output_mlp, self.att_vec_mlp),
                                ],
                                1,
                            )
                        ),
                        self.att_vec,
                    )
                    / T
            )
        att = torch.softmax(logits, 1)
        return (att[:, 0][:, None], att[:, 1][:, None], att[:, 2][:, None],
                att[:, 3][:, None], att[:, 4][:, None], att[:, 5][:, None], att[:, 6][:, None])

    def forward(self, input, adj_low, adj_high, adj_low_unnormalized, graph=None, fr_high_g=None, fr_low_g=None, mlp_g=None):
        device = self.device  # [port]

        output_low = F.relu(torch.spmm(adj_low, (torch.mm(input, self.weight_low))))
        output_high = F.relu(torch.spmm(adj_high, (torch.mm(input, self.weight_high))))
        output_mlp = F.relu(torch.mm(input, self.weight_mlp))

        # (yalnızca model_type == "acm_spa" yolu kullanılıyor)
        dstdata = spa_update_all(graph, output_low, output_high)  # [port] graph.update_all(mp_func_spa, agg_func_spa)

        neigh_hete_low = dstdata["neigh_hete_low"].to(device)
        neigh_hete_high = dstdata["neigh_hete_high"].to(device)
        neigh_homo_low = dstdata["neigh_homo_low"].to(device)
        neigh_homo_high = dstdata["neigh_homo_high"].to(device)
        neigh_unk_low = dstdata["neigh_unk_low"].to(device)
        neigh_unk_high = dstdata["neigh_unk_high"].to(device)

        if fr_high_g != None:
            balance_1 = self.balance_w(self.fc_balance(neigh_unk_low)).to(device)
            balance_2 = self.balance_w(self.fc_balance(neigh_unk_high)).to(device)
            self.att_fr_low, self.att_fr_high, self.att_be_low, self.att_be_high, self.att_unk_low, self.att_unk_high, self.att_mlp = self.attention7(
                neigh_hete_low, neigh_hete_high, neigh_homo_low, neigh_homo_high, neigh_unk_low, neigh_unk_high, output_mlp, balance_1, balance_2
            )

            return 7 * (
                    self.att_fr_low * neigh_hete_low +
                    self.att_fr_high * neigh_hete_high +
                    self.att_be_low * neigh_homo_low +
                    self.att_be_high * neigh_homo_high +
                    self.att_unk_low * neigh_unk_low +
                    self.att_unk_high * neigh_unk_high +
                    self.att_mlp * output_mlp
            )

        self.att_fr_low, self.att_fr_high, self.att_be_low, self.att_be_high, self.att_unk_low, self.att_unk_high, self.att_mlp = self.attention7(
            neigh_hete_low, neigh_hete_high, neigh_homo_low, neigh_homo_high, neigh_unk_low, neigh_unk_high, output_mlp
        )

        return 7 * (
                self.att_fr_low * neigh_hete_low +
                self.att_fr_high * neigh_hete_high +
                self.att_be_low * neigh_homo_low +
                self.att_be_high * neigh_homo_high +
                self.att_unk_low * neigh_unk_low +
                self.att_unk_high * neigh_unk_high +
                self.att_mlp * output_mlp
        )


class ACM(nn.Module):
    def __init__(
        self,
        nfeat,
        nhid,
        nclass,
        nlayers,
        dropout,
        model_type,
        variant=False,
        init_layers_X=1,
        device=None,  # [port]
    ):
        super(ACM, self).__init__()
        device = device if device is not None else torch.device("cpu")
        self.bn = nn.BatchNorm1d(nhid)
        self.preprocessed = False
        self.gcns, self.mlps = nn.ModuleList(), nn.ModuleList()
        self.model_type, self.nlayers = (model_type, nlayers)
        self.gcns.append(
            GraphConvolution(
                nfeat,
                nhid,
                model_type=model_type,
                variant=variant,
                device=device,
            )
        )
        self.gcns.append(
            GraphConvolution(
                1 * nhid,
                nclass,
                model_type=model_type,
                output_layer=1,
                variant=variant,
                device=device,
            )
        )

        self.dropout = dropout
        self.fea_param, self.xX_param = init.xavier_uniform_(Parameter(
            torch.FloatTensor(1, 1).to(device))
        ), init.xavier_uniform_(Parameter(torch.FloatTensor(1, 1).to(device)))

    def preprocess(self, data):
        # [port] convert_to_dglgraph + assign_pseudo_labels_to_dglgraph + create_block_from_dglgraph
        num_nodes = data.x.size(0)
        if data.edge_index.numel() > 0:
            num_nodes = max(num_nodes, int(data.edge_index.max()) + 1)
        train_nid = torch.nonzero(data.train_mask, as_tuple=False).squeeze()
        labels = data.y[train_nid]
        label_unk = (torch.ones(num_nodes, device=data.y.device) * -1).long()
        label_unk[train_nid] = labels.long()
        self.block = SPAGraph(data.edge_index, label_unk, num_nodes)
        self.h = data.x

    def forward(self, data, adj_low_un, adj_low, adj_high, fr_low_g=None, fr_high_g=None, mlp_g=None):
        if fr_low_g !=None:
            freeze_parameters(fr_low_g)
            freeze_parameters(fr_low_g)
            mlp_g
        if self.preprocessed:
            block = self.block
            h = self.h
        else:
            self.preprocess(data)
            block = self.block
        x = data.x
        x = F.dropout(x, self.dropout, training=self.training)

        if fr_high_g != None:
            fea1 = self.gcns[0](x, adj_low, adj_high, adj_low_un, block, fr_high_g[0], fr_low_g[0], mlp_g[0])
        else:
            fea1 = self.gcns[0](x, adj_low, adj_high, adj_low_un, block)

        fea1 = F.relu(fea1)
        fea1 = F.dropout(fea1, self.dropout, training=self.training)
        if fr_high_g != None:
            fea2 = self.gcns[1](fea1, adj_low, adj_high, adj_low_un, block, fr_high_g[1], fr_low_g[1], mlp_g[1])
        else:
            fea2 = self.gcns[1](fea1, adj_low, adj_high, adj_low_un, block)
        return F.log_softmax(fea2, dim=1)


class Model(nn.Module):
    """benchmark001/model/default_model.py: Model (FModule). [port] in_size/out_size veri setinden."""

    def __init__(self, in_size, out_size, hid_size=128, device=None):
        super().__init__()
        self.model = get_spa(in_size, hid_size, out_size, 2, device=device)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)

    def has_nan(self):  # flgo/utils/fmodule.py: FModule.has_nan (aynen)
        for p in self.parameters():
            if torch.any(torch.isnan(p)).item():
                return True
        return False


# =============================================================== simple_logger.py: log_once
def compute_adjacencies(data, device):
    """[port] SimpleLogger.log_once'taki adj_low_un / adj_low / adj_high hesabı (DGL'siz).

    Orijinal:
        g = convert_to_dglgraph(data); adj_low_unnormalized = g.adjacency_matrix()
        adj_low = normalize_tensor(I + adj_low_unnormalized.to_dense())
        adj_high = (I - adj_low).to_sparse()
    DGL 2.1'de adjacency_matrix() satırı kaynak, sütunu hedef olan seyrek matris döndürür
    (A[src, dst] = 1; yerel ortamda DGL 2.1.0 ile doğrulandı)."""
    num_nodes = data.x.size(0)
    if data.edge_index.numel() > 0:
        num_nodes = max(num_nodes, int(data.edge_index.max()) + 1)
    edge_index = data.edge_index.to(device)
    values = torch.ones(edge_index.size(1), device=device)
    adj_low_unnormalized = torch.sparse_coo_tensor(edge_index, values, (num_nodes, num_nodes)).coalesce()
    eye_matrix = torch.eye(num_nodes).to(device)
    adj_low = (normalize_tensor(eye_matrix + adj_low_unnormalized.to_dense())).to(device)
    adj_high = (torch.eye(num_nodes).to(device) - adj_low).to(device).to_sparse()
    return adj_low_unnormalized, adj_low, adj_high


# =============================================================== flgo/algorithm/fedbase.py (aynen)
def calculate_shigh(data, sampling_rate=0.2):
    features = data.x
    num_nodes, num_features = features.shape
    features = features.cpu().numpy()
    edge_index = data.edge_index.cpu().numpy()
    adj_matrix = np.zeros((num_nodes, num_nodes))
    for i, j in zip(edge_index[0], edge_index[1]):
        adj_matrix[i, j] = 1
    degree_matrix = np.diag(np.sum(adj_matrix, axis=1))
    laplacian_matrix = degree_matrix - adj_matrix
    sampled_indices = np.random.choice(num_features, size=int(sampling_rate * num_features), replace=False)
    S_high_values = []
    for idx in sampled_indices:
        x_k = features[:, idx]
        numerator = x_k.T @ laplacian_matrix @ x_k
        denominator = x_k.T @ x_k
        if denominator != 0:
            S_high = numerator / denominator
            if not np.isnan(S_high):
                S_high_values.append(S_high)

    if S_high_values:
        mean_shigh = np.mean(S_high_values)
    else:
        mean_shigh = np.nan

    return mean_shigh


# =============================================================== flgo/algorithm/fedspa.py (aynen)
def sum_of_layer_fisher_differences(fish_dict1, fish_dict2, layer):
    total_difference = 0.0
    for key in fish_dict1:
        if layer in key:
            difference = fish_dict1[key] - fish_dict2[key]
            total_difference += torch.sum(torch.abs(difference))
    return total_difference

def normalize_list(values):
    total = sum(values)
    if total == 0:
        raise ValueError("Sum of the elements is zero, cannot normalize.")
    normalized_values = [value / total for value in values]
    return normalized_values


def calculate_fisher_average(fisher_dicts):
    if not fisher_dicts:
        raise ValueError("The list of Fisher dictionaries is empty.")
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    fisher_avg = {}
    for key in fisher_dicts[0].keys():
        fisher_avg[key] = torch.zeros_like(fisher_dicts[0][key], device=device)

    for fish_dict in fisher_dicts:
        for key in fish_dict:
            fisher_avg[key] += fish_dict[key].to(device)
    num_dicts = len(fisher_dicts)
    for key in fisher_avg:
        fisher_avg[key] /= num_dicts

    return fisher_avg


def normalize(data):
    min_val = min(data)
    max_val = max(data)
    range = max_val - min_val
    normalized_data = [(x - min_val) / range for x in data]
    return normalized_data


def calculate_weights(differences):
    float_diff = [x.item() if isinstance(x, torch.Tensor) else x for x in differences]
    normalized_diff = normalize(float_diff)
    pre_weights = [np.exp(-x) for x in normalized_diff]
    sum_weights = sum(pre_weights)
    weights = [w / sum_weights for w in pre_weights]
    return weights

def fish_cal(net, data, adj_low_un, adj_low, adj_high, high_list, low_list, mlp_list):
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    net.to(device)
    label = data.y
    data, label = data.to(device), label.to(device)
    output = net(data, adj_low_un, adj_low, adj_high, high_list, low_list, mlp_list)
    pre = F.log_softmax(output, dim=1)
    log_likelihoods = pre.gather(1, label.view(-1, 1)).squeeze()
    net.zero_grad()
    log_likelihoods.mean().backward()
    average_grads = {name: param.grad.clone() for name, param in net.named_parameters() if param.grad is not None}
    fish_dict = {name: grad ** 2 for name, grad in average_grads.items()}
    return fish_dict


class _TrainData:
    """flgo TaskDataset'in kullanılan kısmı: .data (grafik, maskelerle)."""

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return int(self.data.train_mask.numel())


class FedSPAClient:
    """flgo/algorithm/fedspa.py: Client'ın eğitimle ilgili kısmı.

    [port] Ortak FGCL ayarı: optimizer Adam (flgo'daki get_optimizer'ın 'adam' dalı: yalnızca
    requires_grad parametreler, lr ve weight_decay), num_steps = yerel epoch sayısı; her adım
    tüm görev grafiği üzerinde (orijinal yükleyici de tek grafiği tek "batch" olarak veriyor)."""

    def __init__(self, cid, learning_rate, weight_decay, num_steps):
        self.id = cid
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.num_steps = num_steps
        self.criterion = torch.nn.NLLLoss()
        self.model_g = None
        self.fr_low_g_list = None
        self.fr_high_g_list = None
        self.mlp_list = None
        self.train_data = None
        self.adj_low_un = self.adj_low = self.adj_high = None
        self.radio_diff = None

    @property
    def datavol(self):
        return len(self.train_data)

    def set_task(self, data, adjs):
        self.train_data = _TrainData(data)
        self.adj_low_un, self.adj_low, self.adj_high = adjs

    def get_optimizer(self, model, lr, weight_decay):
        filter_fn = filter(lambda p: p.requires_grad, model.parameters())
        return torch.optim.Adam(filter_fn, lr=lr, weight_decay=weight_decay)

    def compute_loss(self, model, data, adj_low_un, adj_low, adj_high, fr_low_g_list, fr_high_g_list, mlp_list):
        # GeneralCalculator.compute_loss (aynen)
        tdata = data
        valid_mask = (tdata.y >= 0) & tdata.train_mask
        outputs = model(tdata, adj_low_un, adj_low, adj_high, fr_high_g_list, fr_low_g_list, mlp_list)
        if valid_mask.sum().item() == 0:
            return {'loss': torch.tensor(0.0)}
        loss = self.criterion(outputs[valid_mask], tdata.y[valid_mask])
        return {'loss': loss}

    def train(self, model):
        model.train()

        optimizer = self.get_optimizer(model, lr=self.learning_rate, weight_decay=self.weight_decay)
        if self.model_g is not None:
            for param in self.model_g.parameters():
                param.requires_grad = False

        self.fr_low_g_list = []
        self.fr_high_g_list = []
        self.mlp_list = []

        for iter in range(self.num_steps):
            if iter == 0:
                for gcn_layer in model.model.gcns:
                    fr_low_g = gcn_layer.att_vec_low_unk
                    self.fr_low_g_list.append(fr_low_g)

                    fr_high_g = gcn_layer.att_vec_high_unk
                    self.fr_high_g_list.append(fr_high_g)

                    mlp_g = gcn_layer.att_vec_mlp
                    self.mlp_list.append(mlp_g)
            batch_data = self.train_data.data
            model.zero_grad()
            loss = self.compute_loss(model, batch_data, self.adj_low_un, self.adj_low, self.adj_high, self.fr_low_g_list, self.fr_high_g_list, self.mlp_list)['loss']
            loss.backward()
            optimizer.step()
        return loss.item()


class FedSPAServer:
    """flgo/algorithm/fedspa.py: Server.aggregate (HBDA). [port] logger -> print."""

    def __init__(self, model, clients):
        self.model = model
        self.clients = clients
        self.received_clients = list(range(len(clients)))

    def new_task(self):
        # [port] HBDA'nın homofili ağırlıkları (diffs/weights2) orijinalde eğitimin başında bir kez
        # hesaplanıyor; burada her görevin başında, o görevin alt-grafiğiyle yeniden hesaplanır.
        if hasattr(self, "diffs"):
            del self.diffs
        if hasattr(self, "weights2"):
            del self.weights2

    def aggregate(self, models: list, *args, **kwargs):
        if len(models) == 0: return self.model

        nan_exists = [m.has_nan() for m in models]
        if any(nan_exists):
            if all(nan_exists): raise ValueError("All the received local models have parameters of nan value.")

            print('Warning("There exists nan-value in local models, which will be automatically removed from the aggregation list.")')

            new_models = []
            received_clients = []

            for ni, mi, cid in zip(nan_exists, models, self.received_clients):
                if ni: continue
                new_models.append(mi)
                received_clients.append(cid)
            self.received_clients = received_clients
            models = new_models

        local_data_vols = [c.datavol for c in self.clients]
        total_data_vol = sum(local_data_vols)

        fishs = []
        for model, cid in zip(models, self.received_clients):
            fishs.append(fish_cal(model, self.clients[cid].train_data.data, self.clients[cid].adj_low_un,
                                  self.clients[cid].adj_low, self.clients[cid].adj_high,
                                  self.clients[cid].fr_high_g_list, self.clients[cid].fr_low_g_list,
                                  self.clients[cid].mlp_list))

        fish_g = calculate_fisher_average(fishs)

        aggregated_params = collections.defaultdict(lambda: 0)
        if not hasattr(self, 'diffs'):
            self.diffs = []
            for client in self.clients:
                self.diffs.append(abs(client.radio_diff))
            self.weights2 = calculate_weights(self.diffs)
        fisher_weights = {}
        for layer in ["fr", "be"]:
            layer_fish_diffs = []
            for fish in fishs:
                layer_fish_diffs.append(sum_of_layer_fisher_differences(fish, fish_g, layer))
            layer_fish_diffs = normalize_list(layer_fish_diffs)
            fisher_weights[layer] = calculate_weights(layer_fish_diffs)

        for i, model in enumerate(models):
            for name, param in model.named_parameters():
                if any(layer in name for layer in fisher_weights.keys()):
                    for layer in fisher_weights.keys():
                        if layer in name:
                            aggregated_params[name] += param.data * fisher_weights[layer][i]
                else:
                    aggregated_params[name] += param.data * self.weights2[i]
        with torch.no_grad():
            for name, param in self.model.named_parameters():
                if name in aggregated_params:
                    param.data.copy_(aggregated_params[name])

        self.last_weights = dict(homophily=[float(w) for w in self.weights2],
                                 fr=[float(w) for w in fisher_weights["fr"]],
                                 be=[float(w) for w in fisher_weights["be"]])
        return self.model


In [ ]:
%%writefile fgcl_bench/runner_fedspa.py
"""FedSPA koşucusu: SFPD + HBDA (fedspa_port.py) görev akışında ardışık eğitilir.

FedSPA bir continual learning yöntemi değildir (standart FGL'de homofili heterojenliği için
tasarlandı). Burada, POWER makalesinin FedAvg/FedGTA gibi FGL yöntemlerini kullandığı biçimde,
görevler sırayla geldikçe aynı global model üzerinde eğitilen (fine-tuning) bir FGL baseline'ı
olarak çalıştırılır; hiçbir unutma önleyici mekanizma eklenmez.

Görev t için her istemcide:
  * görev alt-grafiği: cache'teki görev düğümleri (görevin sınıflarındaki tüm düğümler) ve
    aralarındaki kenarlar (FedFST'nin görev grafiğiyle aynı), düğümler yeniden numaralanır;
  * label_unk = görevin eğitim düğümlerinin etiketleri (ACM.preprocess, data.train_mask'ten);
  * adj_low_un / adj_low / adj_high (orijinalde logger'da, eğitimin başında bir kez) ve
    radio_diff = calculate_shigh(...) (orijinalde global_test'te, ilk round'da) görevin
    alt-grafiği için yeniden hesaplanır; HBDA'nın homofili ağırlıkları görev başında yenilenir.
Ortak FGCL bütçesi: görev başına `rounds` round, round başına `local_epochs` tam-grafik adım,
Adam(lr=1e-2, weight_decay=5e-4) (orijinal: SGD lr 0.1, 200 round, num_steps 4).
Her round'da 3 istemcinin hepsi katılır (orijinal betik flgo'nun varsayılan 'md' örneklemesini
kullanıyor; o örnekleme yerine koymalı olduğu için aynı istemci bir round'da birden çok kez seçilebiliyor).
Değerlendirmede global model eval modundadır (orijinal flgo kodu model.eval() çağırmıyor).
"""
import copy
import time

import torch
from torch_geometric.data import Data

from . import fedspa_port as port
from .common import set_seed, pick_device, task_view_hash

LR = 1e-2
WEIGHT_DECAY = 5e-4


def build_task_graph(client_cache, task):
    nodes = torch.sort(task["nodes"])[0]
    n_client = client_cache["x"].size(0)
    remap = torch.full((n_client,), -1, dtype=torch.long)
    remap[nodes] = torch.arange(nodes.numel())
    edge_index = remap[task["edge_index"]]
    assert bool((edge_index >= 0).all()), "görev kenarı görev düğümü dışına çıkıyor"
    data = Data(x=client_cache["x"][nodes].clone(), y=client_cache["y"][nodes].clone(), edge_index=edge_index,
                train_mask=task["train_mask"][nodes].clone(), val_mask=task["valid_mask"][nodes].clone(),
                test_mask=task["test_mask"][nodes].clone())
    return data, nodes


def run(cfg, cache, evaluator):
    meta = cache["meta"]
    set_seed(cfg.seed)
    device = pick_device()
    print(f"Using device: {device}")
    num_tasks = meta["num_tasks"]
    n_clients = len(cache["clients"])

    graphs = [[build_task_graph(cl, t) for t in cl["tasks"]] for cl in cache["clients"]]
    views = []
    for cid in range(n_clients):
        tv = []
        for t in range(num_tasks):
            g, nodes = graphs[cid][t]
            tv.append(dict(train=nodes[g.train_mask], valid=nodes[g.val_mask], test=nodes[g.test_mask], nodes=nodes,
                           edges=nodes[g.edge_index].contiguous(), y=g.y))
        views.append(tv)
    used_hash = task_view_hash(views)
    assert used_hash == cache["hash"], "FedSPA'ya verilen görevler ortak cache ile aynı değil!"
    print(f"[harness] görev maskeleri cache ile aynı (hash {used_hash[:12]})")

    graphs = [[(g.to(device), nodes) for g, nodes in row] for row in graphs]
    adj_cache = {}

    def adjs(cid, t):
        if (cid, t) not in adj_cache:
            adj_cache[(cid, t)] = port.compute_adjacencies(graphs[cid][t][0], device)
        return adj_cache[(cid, t)]

    model = port.Model(meta["in_dim"], meta["out_dim"], device=device).to(device)
    clients = [port.FedSPAClient(cid, LR, WEIGHT_DECAY, cfg.local_epochs) for cid in range(n_clients)]
    server = port.FedSPAServer(model, clients)
    print(f"FedSPA: hidden=128, dropout=0.1, Adam lr={LR} wd={WEIGHT_DECAY}, rounds/task={cfg.rounds}, "
          f"local steps={cfg.local_epochs}, istemci={n_clients}")

    # test sırası: evaluator.test_idx[c][t] istemci-içi numaralar; alt-grafik numarasına çevir
    test_pos = []
    for cid in range(n_clients):
        row = []
        for t in range(num_tasks):
            g, nodes = graphs[cid][t]
            pos = torch.searchsorted(nodes.to(device), evaluator.test_idx[cid][t].to(device))
            row.append(pos)
        test_pos.append(row)

    def predict(client_id, eval_task_id):
        g, _ = graphs[client_id][eval_task_id]
        adj_low_un, adj_low, adj_high = adjs(client_id, eval_task_id)
        c = clients[client_id]
        m = server.model
        was_training = m.training
        m.eval()
        with torch.no_grad():
            # Orijinal Client.test: istemcinin son eğitimdeki fr listeleriyle (denge dalı) çağrı
            out = m(g, adj_low_un, adj_low, adj_high, c.fr_high_g_list, c.fr_low_g_list, c.mlp_list)
        m.train(was_training)
        pred = out.max(1)[1]
        return pred[test_pos[client_id][eval_task_id]]

    task_seconds, hbda_weights, radio = [], [], []
    for task_id in range(num_tasks):
        t_task = time.time()
        print(f"\n========== Task {task_id}/{num_tasks} ==========")
        for cid, c in enumerate(clients):
            c.set_task(graphs[cid][task_id][0], adjs(cid, task_id))
            c.radio_diff = port.calculate_shigh(c.train_data.data)
        radio.append([float(c.radio_diff) for c in clients])
        print("radio_diff (S_high):", radio[-1])
        server.new_task()
        for round_id in range(cfg.rounds):
            models, losses = [], []
            for c in clients:
                local = copy.deepcopy(server.model)       # Server.pack: copy.deepcopy(self.model)
                c.model_g = copy.deepcopy(local)          # Client.reply
                losses.append(c.train(local))
                models.append(local)
            server.received_clients = list(range(n_clients))
            server.aggregate(models)
            print(f"round {round_id}: kayıplar {[round(l, 4) for l in losses]} | HBDA ağırlıkları {server.last_weights}")
        hbda_weights.append(server.last_weights)
        evaluator.evaluate_after_task(task_id, predict)
        task_seconds.append(round(time.time() - t_task, 2))

    return dict(task_seconds=task_seconds, mask_hash=used_hash, hbda_weights=hbda_weights, radio_diff=radio,
                hyperparameters=dict(model="ACM-SPA (2 katman)", hidden=128, dropout=0.1, optimizer="Adam", lr=LR,
                                     weight_decay=WEIGHT_DECAY, rounds=cfg.rounds, local_steps=cfg.local_epochs))


In [ ]:
%%writefile fgcl_bench/run.py
"""Tek koşu (bir yöntem, bir veri seti, bir seed) -> results/<veri>__<yöntem>__seed<s>.json

    python -m fgcl_bench.run --method {fedfst,power,fedspa} --dataset cora --seed 24 \
        --src_dir <kaynak kodlar> --work_dir <Drive klasörü> [--rounds 10 --local_epochs 3] [--smoke]

--smoke: 1 round, 1 yerel epoch; FedFST'de jeneratör/KD epoch'ları 2/3, POWER'da L-BFGS 3 iterasyon.
Sonuçlar results_smoke/ altına yazılır (gerçek sonuçlarla karışmaz).
"""
import argparse
import subprocess
import sys
import tempfile
import time
import types
from pathlib import Path

import torch

from .common import DATASETS, METHODS, run_name, write_json_atomic, env_info, pick_device
from .data import cache_path, load_cache
from .evaluate import Evaluator


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=list(METHODS))
    ap.add_argument("--dataset", required=True, choices=list(DATASETS))
    ap.add_argument("--seed", type=int, default=24)
    ap.add_argument("--rounds", type=int, default=10)
    ap.add_argument("--local_epochs", type=int, default=3)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--src_dir", required=True)
    ap.add_argument("--work_dir", required=True)
    args = ap.parse_args(argv)
    if args.rounds > 15:
        raise SystemExit("Görev başına en fazla 15 communication round (bütçe kuralı).")
    if args.smoke:
        args.rounds, args.local_epochs = 1, 1

    src, work = Path(args.src_dir).resolve(), Path(args.work_dir).resolve()
    tmp_dir = Path(tempfile.mkdtemp(prefix="fgcl_"))
    cfg = types.SimpleNamespace(method=args.method, dataset=args.dataset, seed=args.seed, rounds=args.rounds,
                                local_epochs=args.local_epochs, smoke=args.smoke, src_dir=src, work_dir=work,
                                fedfst_dir=src / "FedFST", power_dir=src / "POWER", fedspa_dir=src / "FedSPA",
                                data_root=work / "data", tmp_dir=tmp_dir)
    device = pick_device()
    print(f"=== {METHODS[args.method]} | {DATASETS[args.dataset]['title']} | seed {args.seed} | "
          f"rounds/görev {args.rounds} | yerel epoch {args.local_epochs} | smoke={args.smoke} ===")
    print("ortam:", env_info(device))

    if not cache_path(work, args.dataset, args.seed).exists():
        # Cache ayrı bir süreçte üretilir: FedFST'nin `utils` modülü POWER'ınkiyle çakışmasın.
        subprocess.run([sys.executable, "-m", "fgcl_bench.data", "--fedfst", str(cfg.fedfst_dir),
                        "--work_dir", str(work), "--dataset", args.dataset, "--seed", str(args.seed)], check=True)
    cache = load_cache(work, args.dataset, args.seed)
    evaluator = Evaluator(cache, cfg.fedfst_dir)

    t0 = time.time()
    if args.method == "fedfst":
        from .runner_fedfst import run
    elif args.method == "power":
        from .runner_power import run
    else:
        from .runner_fedspa import run
    extra = run(cfg, cache, evaluator)
    elapsed = time.time() - t0

    assert extra["mask_hash"] == cache["hash"]
    aa, af = evaluator.final()
    meta = cache["meta"]
    result = dict(method=args.method, method_title=METHODS[args.method], dataset=args.dataset,
                  dataset_title=DATASETS[args.dataset]["title"], seed=args.seed, smoke=args.smoke,
                  rounds_per_task=args.rounds, local_epochs=args.local_epochs, num_clients=meta["num_clients"],
                  per_task_class_num=meta["per_task"], num_tasks=meta["num_tasks"],
                  task_classes=meta["task_classes"], dropped_classes=meta["dropped_classes"],
                  acc_matrix=evaluator.matrix(), AA=round(aa, 4), AF=round(af, 4),
                  time_sec=round(elapsed, 1), env=env_info(device), cache_hash=cache["hash"], **extra)
    out_dir = work / ("results_smoke" if args.smoke else "results")
    out = out_dir / f"{run_name(args.dataset, args.method, args.seed)}.json"
    write_json_atomic(out, result)
    print(f"SONUÇ | {METHODS[args.method]} | {DATASETS[args.dataset]['title']} | AA={aa:.2f} | AF={af:.2f} | "
          f"süre={elapsed:.0f} sn | {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile fgcl_bench/equivalence.py
"""Vektörizasyon eşdeğerlik testi (FedFST yamaları P5a–P5f).

Orijinal fonksiyonlar yamalı repodan değil, git geçmişinden (`git show
d83fac0:<dosya>`) okunur; böylece "orijinal" taraf gerçekten orijinal koddur.
Her karşılaştırmada iki taraf aynı seed'le başlatılır ve şunlar istenir:
  * çıktılar birebir aynı (torch.equal; dtype, şekil ve gerekirse stride dahil),
  * gradyanlar birebir aynı (KL kaybı için),
  * çağrıdan sonra RNG durumu aynı (aynı generator'dan aynı sayıda sayı çekildi).

Kullanım:  python -m fgcl_bench.equivalence --fedfst <yamalı FedFST klasörü>
"""
import argparse
import ast
import random
import subprocess
import sys
import textwrap
import time
import types

import numpy as np
import torch
from torch_geometric.data import Data

COMMIT = "d83fac0"


def git_show(repo, path):
    return subprocess.run(["git", "-C", str(repo), "show", f"{COMMIT}:{path}"],
                          check=True, capture_output=True, text=True).stdout


def module_from_source(name, source):
    mod = types.ModuleType(name)
    exec(compile(source, f"<{COMMIT}:{name}>", "exec"), mod.__dict__)
    return mod


def method_from_class(source, class_name, method_name, namespace):
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == class_name:
            for item in node.body:
                if isinstance(item, ast.FunctionDef) and item.name == method_name:
                    code = textwrap.dedent(ast.get_source_segment(source, item))
                    exec(compile(code, f"<{COMMIT}:{class_name}.{method_name}>", "exec"), namespace)
                    return namespace[method_name]
    raise KeyError(method_name)


def original_kl_block(ours_source):
    """feature_gen_train içindeki orijinal KL döngüsünü fonksiyona sarar."""
    lines = ours_source.split("\n")
    start = next(i for i, l in enumerate(lines) if l.strip() == "kl_losses = []")
    end = next(i for i in range(start, len(lines))
               if l_strip(lines[i]) == "loss_kl = torch.tensor(0.0, device=self.device)")
    block = textwrap.dedent("\n".join(lines[start:end + 1]))
    body = textwrap.indent(block, "    ")
    code = ("def original_kl(all_generated_features, all_generated_labels, subgraph, self):\n"
            f"{body}\n    return loss_kl\n")
    ns = {"torch": torch, "F": torch.nn.functional}
    exec(compile(code, f"<{COMMIT}:feature_gen_train KL block>", "exec"), ns)
    return ns["original_kl"]


def l_strip(s):
    return s.strip()


def rng_snapshot(device):
    snap = {"cpu": torch.get_rng_state().clone()}
    if device.type == "cuda":
        snap["cuda"] = torch.cuda.get_rng_state(device).clone()
    return snap


def same_rng(a, b):
    return all(torch.equal(a[k], b[k]) for k in a)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def same_tensor(a, b, check_stride=True):
    if a.dtype != b.dtype or a.shape != b.shape or a.device != b.device:
        return False
    if check_stride and a.numel() > 1 and a.stride() != b.stride():
        return False
    return torch.equal(a, b)


class Report:
    def __init__(self):
        self.rows = []

    def add(self, name, n_cases, ok, detail=""):
        self.rows.append((name, n_cases, ok, detail))
        print(f"  {'OK ' if ok else 'FARK'}  {name:<46s} {n_cases:>4d} durum  {detail}")

    @property
    def all_ok(self):
        return all(r[2] for r in self.rows)


def random_graph(gen, n_nodes, n_classes, n_edges, device, n_feat=8):
    y = torch.randint(0, n_classes, (n_nodes,), generator=gen)
    x = torch.rand(n_nodes, n_feat, generator=gen)
    edge_index = torch.randint(0, n_nodes, (2, n_edges), generator=gen)
    return Data(x=x.to(device), y=y.to(device), edge_index=edge_index.to(device))


def test_reduce_edges(orig_utils, new_utils, device, report, n_cases=60):
    # GPU'da orijinal döngü her kenarda .item() ile senkronize olur ve yavaştır: GPU'da daha az ve
    # daha küçük durum denenir (CPU'da tam kapsam).
    gpu = device.type == "cuda"
    if gpu:
        n_cases = 9
    gen = torch.Generator().manual_seed(1234)
    for fname in ["reduce_homophilic_edges", "reduce_heterophilic_edges"]:
        ok = True
        for case in range(n_cases):
            n = int(torch.randint(2, 300 if gpu else 900, (1,), generator=gen))
            c = int(torch.randint(1, 5, (1,), generator=gen))
            e = int(torch.randint(0, 2 * n + 1, (1,), generator=gen))
            ratio = [0.05, 0.1, 0.5][case % 3]
            data = random_graph(gen, n, c, e, device)
            outs = []
            for impl in (orig_utils, new_utils):
                seed_all(100 + case)
                ei = data.edge_index
                for _ in range(3):  # ardışık çağrılar (jeneratör eğitimindeki döngü gibi)
                    ei = getattr(impl, fname)(data, ei, reduction_ratio=ratio)
                outs.append((ei, rng_snapshot(device)))
            ok &= same_tensor(outs[0][0], outs[1][0], check_stride=False) and same_rng(outs[0][1], outs[1][1])
        report.add(f"utils.{fname}", n_cases, ok)

    # S_high / homofili ayar döngüsünü taklit eden karışık dizi
    ok = True
    n_seq = 2 if gpu else 20
    for case in range(n_seq):
        data = random_graph(gen, 200 if gpu else 800, 4, 400 if gpu else 1600, device)
        outs = []
        for impl in (orig_utils, new_utils):
            seed_all(500 + case)
            ei = data.edge_index
            for it in range(60):
                fn = impl.reduce_homophilic_edges if (it + case) % 3 else impl.reduce_heterophilic_edges
                ei = fn(data, ei, reduction_ratio=0.05)
            outs.append((ei, rng_snapshot(device)))
        ok &= same_tensor(outs[0][0], outs[1][0], check_stride=False) and same_rng(outs[0][1], outs[1][1])
    report.add("60 iterasyonluk karışık azaltma dizisi", n_seq, ok)


def test_homophily_level(orig_method, new_method, device, report, n_cases=40):
    if device.type == "cuda":
        n_cases = 10
    gen = torch.Generator().manual_seed(77)
    ok = True
    for case in range(n_cases):
        tasks = []
        n_tasks = int(torch.randint(1, 4, (1,), generator=gen))
        for _ in range(n_tasks):
            n = int(torch.randint(5, 400, (1,), generator=gen))
            g = random_graph(gen, n, 3, int(torch.randint(0, 3 * n, (1,), generator=gen)), device)
            train_mask = torch.rand(n, generator=gen) < 0.6
            tasks.append({"local_data": g, "train_mask": train_mask})
        fake = types.SimpleNamespace(tasks=tasks)
        a = orig_method(fake, n_tasks)
        b = new_method(fake, n_tasks)
        ok &= (type(a) is type(b)) and (a == b)
    report.add("OursClient.get_learned_Homophily_Level", n_cases, ok)


def test_kl(orig_kl, new_kl, device, report):
    gen = torch.Generator().manual_seed(4242)
    ok = True
    n_cases = 0
    max_abs = 0.0
    for n_classes in (1, 2, 4):
        for per_class in (3, 200):
            for n_feat in (7, 64, 1433):
                for order in ("block", "shuffled", "missing_class"):
                    n_cases += 1
                    labels = torch.arange(n_classes).repeat_interleave(per_class)
                    if order == "shuffled":
                        labels = labels[torch.randperm(labels.numel(), generator=gen)]
                    feats0 = torch.randn(labels.numel(), n_feat, generator=gen)
                    sub_n = 300
                    sub_y = torch.randint(0, n_classes, (sub_n,), generator=gen)
                    if order == "missing_class" and n_classes > 1:
                        sub_y[sub_y == n_classes - 1] = 0  # bu sınıfın gerçek düğümü yok -> atlanmalı
                    sub = Data(x=torch.rand(sub_n, n_feat, generator=gen).to(device), y=sub_y.to(device))
                    results = []
                    for fn in (orig_kl, new_kl):
                        seed_all(9 + n_cases)
                        feats = feats0.clone().to(device).requires_grad_(True)
                        out = fn(feats, labels.to(device), sub, types.SimpleNamespace(device=device)) \
                            if fn is orig_kl else fn(feats, labels.to(device), sub, device)
                        out.backward()
                        results.append((out.detach(), feats.grad.detach().clone(), rng_snapshot(device)))
                    (l1, g1, r1), (l2, g2, r2) = results
                    if device.type == "cpu":
                        ok &= torch.equal(l1, l2) and torch.equal(g1, g2) and same_rng(r1, r2)
                    else:  # GPU indirgemeleri farklı sıralarla toplayabilir: değerler çok yakın, RNG aynı
                        max_abs = max(max_abs, (l1 - l2).abs().item(), (g1 - g2).abs().max().item())
                        ok &= same_rng(r1, r2) and torch.allclose(l1, l2, rtol=1e-5, atol=1e-6) \
                            and torch.allclose(g1, g2, rtol=1e-4, atol=1e-7)
    detail = "(karşılaştırılan: kayıp, gradyan, RNG durumu)" if device.type == "cpu" \
        else f"(RNG birebir istenir; GPU'da max|fark|={max_abs:.2e})"
    report.add("feature_gen_train KL kaybı (+gradyan)", n_cases, ok, detail)


def test_subgraph(orig_part, new_part, report, n_cases=60):
    gen = torch.Generator().manual_seed(99)
    ok = True
    for case in range(n_cases):
        n = int(torch.randint(1, 500, (1,), generator=gen))
        e = 0 if case % 10 == 0 else int(torch.randint(1, 4 * n, (1,), generator=gen))
        data = random_graph(gen, n, 5, e, torch.device("cpu"))
        k = int(torch.randint(0, n + 1, (1,), generator=gen))
        nodes = torch.randperm(n, generator=gen)[:k]
        if case % 3 == 0:
            node_list = [np.int64(v) for v in nodes.numpy()]       # class_to_task'taki gibi numpy int
        else:
            node_list = sorted(int(v) for v in nodes)              # get_learned_graph'taki gibi
        if case % 15 == 7 and k > 1:
            node_list = node_list + node_list[:1]                  # tekrar eden düğüm -> döngü yolu
        for flag in (True, False):
            try:
                a = orig_part.get_subgraph_by_node(data, list(node_list), flag)
            except Exception as exc:  # orijinal kodun bozuk kenar-durumları
                a = exc
            b = new_part.get_subgraph_by_node(data, list(node_list), flag)
            if isinstance(a, Exception):
                continue
            ok &= same_tensor(a.x, b.x) and same_tensor(a.y, b.y)
            ok &= same_tensor(a.edge_index, b.edge_index, check_stride=True)
            ok &= list(a.global_map) == list(b.global_map)
    report.add("partition.get_subgraph_by_node (flag=True/False)", n_cases, ok)


def test_louvain(orig_part, new_part, data, name, report):
    seed_all(24)
    t0 = time.time()
    a = orig_part.louvain_partitioner(data, 3)
    t_orig = time.time() - t0
    state_a = np.random.get_state()[1].copy()
    seed_all(24)
    t0 = time.time()
    b = new_part.louvain_partitioner(data, 3)
    t_new = time.time() - t0
    state_b = np.random.get_state()[1].copy()
    ok = len(a) == len(b) and np.array_equal(state_a, state_b)
    for ca, cb in zip(a, b):
        ok &= same_tensor(ca.x, cb.x) and same_tensor(ca.y, cb.y) and same_tensor(ca.edge_index, cb.edge_index)
    report.add(f"partition.louvain_partitioner ({name})", 1, ok, f"orijinal {t_orig:.1f} sn, yeni {t_new:.1f} sn")


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--fedfst", required=True, help="yamalı FedFST klasörü (git deposu)")
    ap.add_argument("--data_root", default=None, help="Cora'nın indirildiği kök (louvain testi için)")
    ap.add_argument("--cuda", action="store_true", help="GPU varsa ayrıca GPU'da da test et")
    args = ap.parse_args(argv)

    torch.set_num_threads(1)  # FedFST'nin set_seed'i de tek iş parçacığı kullanıyor
    sys.path.insert(0, args.fedfst)
    import utils as new_utils                       # yamalı
    from datasets import partition as new_part      # yamalı
    from algorithm import Ours as new_ours          # yamalı

    orig_utils = module_from_source("orig_utils", git_show(args.fedfst, "utils.py"))
    orig_part = module_from_source("orig_partition", git_show(args.fedfst, "datasets/partition.py"))
    ours_src = git_show(args.fedfst, "algorithm/Ours.py")
    orig_homo = method_from_class(ours_src, "OursClient", "get_learned_Homophily_Level", {"torch": torch})
    orig_kl = original_kl_block(ours_src)

    report = Report()
    devices = [torch.device("cpu")]
    if args.cuda and torch.cuda.is_available():
        devices.append(torch.device("cuda:0"))
    for device in devices:
        print(f"[{device}]")
        test_reduce_edges(orig_utils, new_utils, device, report)
        test_homophily_level(orig_homo, new_ours.OursClient.get_learned_Homophily_Level, device, report)
        test_kl(orig_kl, new_ours._gen_kl_loss_vectorized, device, report)
    print("[cpu]")
    test_subgraph(orig_part, new_part, report)
    if args.data_root:
        from torch_geometric.datasets import Planetoid
        cora = Planetoid(args.data_root, "cora")[0]
        test_louvain(orig_part, new_part, cora, "Cora", report)
    print("SONUÇ:", "tüm eşdeğerlik testleri geçti" if report.all_ok else "FARK BULUNDU")
    return 0 if report.all_ok else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile fgcl_bench/run_main_sequential.py
"""Sağlama: FedFST'nin (yamalı) orijinal main.py'sini, jeneratör eğitimini yapan
ThreadPoolExecutor tek iş parçacığına (max_workers=1) indirilmiş olarak çalıştırır.
İstemciler 0, 1, 2 sırasıyla, harness'teki sırayla aynı biçimde çalışır; main.py'nin
kodu değiştirilmez.

    python -m fgcl_bench.run_main_sequential <FedFST klasörü> --dataset_name cora --seed 24 --device_id 0 ...
"""
import concurrent.futures
import os
import runpy
import sys


class _SequentialExecutor(concurrent.futures.ThreadPoolExecutor):
    def __init__(self, *args, **kwargs):
        super().__init__(max_workers=1)


def main():
    fedfst_dir = os.path.abspath(sys.argv[1])
    concurrent.futures.ThreadPoolExecutor = _SequentialExecutor
    sys.argv = ["main.py"] + sys.argv[2:]
    sys.path.insert(0, fedfst_dir)
    os.chdir(fedfst_dir)
    runpy.run_path(os.path.join(fedfst_dir, "main.py"), run_name="__main__")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile fgcl_bench/fedspa_dgl_check.py
"""FedSPA port'unun (fedspa_port.py) orijinal DGL'li koda eşdeğerlik testi.

DGL gerektirir (Colab'ın Python/torch sürümü için DGL wheel'i yok); yerelde ayrı bir ortamda
(Python 3.11, torch 2.2.1, DGL 2.1.0) çalıştırıldı. Orijinal kod FedSPA reposundan okunur:
  benchmark001/config.py  (GraphConvolution, ACM, mp_func_spa, agg_func_spa, DGL blok kurulumu)
  benchmark001/base.py    (MLP_spa)
  flgo/experiment/logger/simple_logger.py (adj matrisleri), flgo/algorithm/fedspa.py (fish_cal,
  Server.aggregate ve yardımcıları), flgo/algorithm/fedbase.py (calculate_shigh)
Aynı ağırlıklar (state_dict, strict=True) ve aynı seed'le iki model karşılaştırılır.

    python -m fgcl_bench.fedspa_dgl_check --fedspa <FedSPA klasörü> [--cache <cache.pt>]
"""
import argparse
import ast
import collections
import copy
import sys
import textwrap
import types
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data

import dgl  # noqa: F401  (yoksa test çalışmaz)

from . import fedspa_port as port


def extract(source, names, namespace, class_name=None):
    tree = ast.parse(source)
    body = tree.body
    if class_name:
        body = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == class_name).body
    for node in body:
        if isinstance(node, ast.FunctionDef) and node.name in names:
            exec(compile(textwrap.dedent(ast.get_source_segment(source, node)), "<orig>", "exec"), namespace)
    return namespace


def load_original(fedspa_dir):
    code = Path(fedspa_dir) / "Code"
    base_ns = {}
    exec(compile((code / "benchmark001/base.py").read_text(), "<orig base.py>", "exec"), base_ns)
    cfg_src = (code / "benchmark001/config.py").read_text()
    kept = []
    for line in cfg_src.split("\n"):
        if line.startswith("from .base import") or line.startswith("from load_data.load_data import") \
                or line.startswith("train_data = load_dataset("):
            kept.append("# [test] " + line)
        else:
            kept.append(line)
    cfg = types.ModuleType("orig_config")
    cfg.__dict__["MLP_spa"] = base_ns["MLP_spa"]
    exec(compile("\n".join(kept), "<orig config.py>", "exec"), cfg.__dict__)

    import dgl as _dgl
    log_ns = {"torch": torch, "dgl": _dgl, "np": np}
    extract((code / "flgo/experiment/logger/simple_logger.py").read_text(),
            {"dgl_to_scipy", "compute_low_freq_matrix", "get_unnormalized_low_freq_matrix",
             "normalize_tensor", "convert_to_dglgraph"}, log_ns)
    fed_ns = {"torch": torch, "np": np, "F": F, "collections": collections}
    fedspa_src = (code / "flgo/algorithm/fedspa.py").read_text()
    extract(fedspa_src, {"sum_of_layer_fisher_differences", "normalize_list", "calculate_fisher_average",
                         "normalize", "calculate_weights", "fish_cal"}, fed_ns)
    extract(fedspa_src, {"aggregate"}, fed_ns, class_name="Server")
    extract((code / "flgo/algorithm/fedbase.py").read_text(), {"calculate_shigh"}, fed_ns)
    return cfg, log_ns, fed_ns


def orig_adjacencies(log_ns, data, device):
    # simple_logger.py: SimpleLogger.log_once (current_round == -1 dalı), device = cpu
    g = log_ns["convert_to_dglgraph"](data)
    adj_low_unnormalized = log_ns["get_unnormalized_low_freq_matrix"](g).to(device)
    eye_matrix = torch.eye(g.number_of_nodes()).to(device)
    adj_low = (log_ns["normalize_tensor"](eye_matrix + adj_low_unnormalized.to_dense())).to(device)
    adj_high = (torch.eye(g.number_of_nodes()).to(device) - adj_low).to(device).to_sparse()
    return adj_low_unnormalized, adj_low, adj_high


class OrigWrapper(torch.nn.Module):
    """default_model.Model ile aynı isimlendirme (model.xxx) için.

    Orijinal att_struc_low ilklenmemiş bellek (torch.FloatTensor); yerel denemelerde zaman zaman
    NaN/1e38 gibi değerler içerdi ve has_nan() kontrolü modeli birleştirmeden çıkardı (port'taki
    değişiklik 4). Karşılaştırmanın hesaba odaklanması için burada da sıfırlanır."""

    def __init__(self, acm):
        super().__init__()
        self.model = acm
        with torch.no_grad():
            for g in self.model.gcns:
                g.att_struc_low.zero_()

    def forward(self, *a, **k):
        return self.model(*a, **k)

    def has_nan(self):
        return any(torch.isnan(p).any().item() for p in self.parameters())


def sequential_sum_limit():
    """torch.sum(dim=1)'in sıralı toplamla birebir aynı kaldığı derece sınırı (çeşitli iç boyutlar
    için en küçüğü). DGL'in agg_func_spa'sı posta kutusunu sum(1) ile toplar; PyTorch bu toplamı
    tensör şekline göre farklı sıralarla yapar, index_add_ ise sıralı toplar."""
    gen = torch.Generator().manual_seed(0)
    limits = []
    for d in (1, 2, 3, 4, 5, 8, 16, 64, 128):
        lim = 200
        for deg in range(1, 200):
            t = torch.randn(7, deg, d, generator=gen)
            seq = torch.zeros(7, d)
            for k in range(deg):
                seq = seq + t[:, k]
            if not torch.equal(t.sum(1), seq):
                lim = deg
                break
        limits.append(lim)
    return min(limits)


SEQ_SUM_LIMIT = None


def seed_all(s):
    np.random.seed(s)
    torch.manual_seed(s)


def graphs_for_test(cache_path):
    gen = torch.Generator().manual_seed(5)
    out = []
    for n, e, f, c in [(40, 50, 6, 3), (12, 30, 6, 3), (40, 160, 16, 4), (200, 1500, 32, 2)]:
        edge_index = torch.randint(0, n, (2, e), generator=gen)
        edge_index = torch.unique(edge_index, dim=1)  # tekrar eden kenar yok (gerçek veride de yok)
        x = torch.rand(n, f, generator=gen)
        y = torch.randint(0, c, (n,), generator=gen)
        train_mask = torch.rand(n, generator=gen) < 0.6
        out.append((f"rastgele N={n} E={edge_index.size(1)}", Data(x=x, y=y, edge_index=edge_index, train_mask=train_mask), c))
    if cache_path:
        cache = torch.load(cache_path, weights_only=False)
        cl = cache["clients"][0]
        task = cl["tasks"][0]
        nodes = torch.sort(task["nodes"])[0]
        remap = torch.full((cl["x"].size(0),), -1, dtype=torch.long)
        remap[nodes] = torch.arange(nodes.numel())
        ei = remap[task["edge_index"]]
        d = Data(x=cl["x"][nodes], y=cl["y"][nodes], edge_index=ei, train_mask=task["train_mask"][nodes])
        out.append((f"{cache['meta']['dataset']} istemci0/görev0 N={nodes.numel()} E={ei.size(1)}",
                    d, cache["meta"]["out_dim"]))
    return out


def compare(name, a, b, rows):
    """rows'a (ad, birebir mi, göreli fark) ekler; göreli fark = max|a-b| / max|a|."""
    if isinstance(a, torch.Tensor):
        exact = a.shape == b.shape and torch.equal(a, b)
        if exact:
            diff = 0.0
        elif a.shape != b.shape:
            diff = float("inf")
        else:
            diff = float((a - b).abs().max()) / max(float(a.abs().max()), 1e-30)
    else:
        exact = a == b
        diff = 0.0 if exact else float("inf")
    rows.append((name, exact, diff))


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--fedspa", required=True)
    ap.add_argument("--cache", default=None, help="gerçek bir görev alt-grafiği için cache dosyası")
    args = ap.parse_args(argv)
    torch.set_num_threads(1)
    global SEQ_SUM_LIMIT
    SEQ_SUM_LIMIT = sequential_sum_limit()
    cfg, log_ns, fed_ns = load_original(args.fedspa)
    device = torch.device("cpu")
    print(f"torch {torch.__version__}, dgl {dgl.__version__}; torch.sum(dim=1) derece < {SEQ_SUM_LIMIT} iken sıralı toplamla birebir")
    all_ok, worst = True, 0.0
    for gname, data, nclass in graphs_for_test(args.cache):
        rows = []
        nfeat = data.x.size(1)
        seed_all(0)
        orig = OrigWrapper(cfg.get_spa(nfeat, 128, nclass, 2)).to(device)
        prt = port.Model(nfeat, nclass, device=device).to(device)
        prt.load_state_dict(orig.state_dict(), strict=True)   # aynı parametre adları ve şekilleri
        compare("parametre adları", [n for n, _ in orig.named_parameters()], [n for n, _ in prt.named_parameters()], rows)

        a_un, a_low, a_high = orig_adjacencies(log_ns, data, device)
        p_un, p_low, p_high = port.compute_adjacencies(data, device)
        compare("adj_low_un (yoğun)", a_un.to_dense(), p_un.to_dense(), rows)
        compare("adj_low", a_low, p_low, rows)
        compare("adj_high (yoğun)", a_high.to_dense(), p_high.to_dense(), rows)

        for branch in ("dengesiz (fr listesi yok)", "denge dalı (fr listeleri)"):
            outs = []
            for model, (au, al, ah) in ((orig, (a_un, a_low, a_high)), (prt, (p_un, p_low, p_high))):
                m = copy.deepcopy(model)
                m.train()
                m.zero_grad()
                lists = None
                if branch.startswith("denge"):
                    lists = ([g.att_vec_high_unk for g in m.model.gcns], [g.att_vec_low_unk for g in m.model.gcns],
                             [g.att_vec_mlp for g in m.model.gcns])
                seed_all(11)
                out = m(data, au, al, ah, *lists) if lists else m(data, au, al, ah)
                loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
                loss.backward()
                grads = {n: (p.grad.clone() if p.grad is not None else None) for n, p in m.named_parameters()}
                flags = {n: p.requires_grad for n, p in m.named_parameters()}
                m.eval()
                with torch.no_grad():
                    out_eval = m(data, au, al, ah, *lists) if lists else m(data, au, al, ah)
                outs.append((out.detach(), grads, flags, out_eval))
            compare(f"{branch}: çıktı (train)", outs[0][0], outs[1][0], rows)
            compare(f"{branch}: çıktı (eval)", outs[0][3], outs[1][3], rows)
            compare(f"{branch}: requires_grad bayrakları", outs[0][2], outs[1][2], rows)
            g0, g1 = outs[0][1], outs[1][1]
            same_none = all((g0[k] is None) == (g1[k] is None) for k in g0)
            rows.append((f"{branch}: gradyanı olan parametreler", same_none, 0.0 if same_none else float("inf")))
            for k in g0:
                if g0[k] is not None and g1[k] is not None:
                    compare(f"{branch}: grad {k}", g0[k], g1[k], rows)

        # Fisher (fish_cal) ve HBDA birleştirmesi: 3 istemci, farklı ağırlıklar
        clients_o, clients_p, models_o, models_p = [], [], [], []
        for cid in range(3):
            seed_all(100 + cid)
            mo = OrigWrapper(cfg.get_spa(nfeat, 128, nclass, 2)).to(device)
            mp = port.Model(nfeat, nclass, device=device).to(device)
            mp.load_state_dict(mo.state_dict(), strict=True)
            lists_o = ([g.att_vec_low_unk for g in mo.model.gcns], [g.att_vec_high_unk for g in mo.model.gcns],
                       [g.att_vec_mlp for g in mo.model.gcns])
            lists_p = ([g.att_vec_low_unk for g in mp.model.gcns], [g.att_vec_high_unk for g in mp.model.gcns],
                       [g.att_vec_mlp for g in mp.model.gcns])
            seed_all(200 + cid)
            radio = fed_ns["calculate_shigh"](data)
            seed_all(200 + cid)
            radio_p = port.calculate_shigh(data)
            compare(f"calculate_shigh (istemci {cid})", float(radio), float(radio_p), rows)
            clients_o.append(types.SimpleNamespace(train_data=types.SimpleNamespace(data=data), adj_low_un=a_un, adj_low=a_low,
                                                   adj_high=a_high, fr_low_g_list=lists_o[0], fr_high_g_list=lists_o[1],
                                                   mlp_list=lists_o[2], radio_diff=radio, datavol=100 + cid))
            clients_p.append(types.SimpleNamespace(train_data=types.SimpleNamespace(data=data), adj_low_un=p_un, adj_low=p_low,
                                                   adj_high=p_high, fr_low_g_list=lists_p[0], fr_high_g_list=lists_p[1],
                                                   mlp_list=lists_p[2], radio_diff=radio_p, datavol=100 + cid))
            models_o.append(mo)
            models_p.append(mp)
        seed_all(300)
        fish_o = fed_ns["fish_cal"](models_o[0], data, a_un, a_low, a_high, clients_o[0].fr_high_g_list,
                                    clients_o[0].fr_low_g_list, clients_o[0].mlp_list)
        seed_all(300)
        fish_p = port.fish_cal(models_p[0], data, p_un, p_low, p_high, clients_p[0].fr_high_g_list,
                               clients_p[0].fr_low_g_list, clients_p[0].mlp_list)
        compare("fish_cal: anahtarlar", sorted(fish_o), sorted(fish_p), rows)
        for k in fish_o:
            compare(f"fish_cal: {k}", fish_o[k], fish_p[k], rows)

        seed_all(0)
        server_model_o = OrigWrapper(cfg.get_spa(nfeat, 128, nclass, 2)).to(device)
        server_o = types.SimpleNamespace(model=server_model_o, clients=clients_o, received_clients=[0, 1, 2],
                                         gv=types.SimpleNamespace(logger=types.SimpleNamespace(info=print)))
        server_p = port.FedSPAServer(port.Model(nfeat, nclass, device=device).to(device), clients_p)
        server_p.model.load_state_dict(server_model_o.state_dict(), strict=True)
        seed_all(400)
        agg_o = fed_ns["aggregate"](server_o, models_o)
        seed_all(400)
        agg_p = server_p.aggregate(models_p)
        for (n1, p1), (n2, p2) in zip(agg_o.named_parameters(), agg_p.named_parameters()):
            compare(f"HBDA birleştirme: {n1}", p1.detach(), p2.detach(), rows)

        n_exact = sum(r[1] for r in rows)
        max_diff = max(r[2] for r in rows)
        worst = max(worst, max_diff)
        max_in_deg = int(torch.bincount(data.edge_index[1], minlength=data.num_nodes).max())
        # PyTorch'un sum(1)'i (DGL'in posta kutusu toplamı) derece >= 18'de kademeli toplamaya geçer;
        # index_add_ sıralı toplar. Bu yüzden bit düzeyinde eşitlik yalnızca derece < 18 iken beklenir.
        ok = (n_exact == len(rows)) if max_in_deg < SEQ_SUM_LIMIT else (max_diff < 1e-4)
        all_ok &= ok
        print(f"[{gname}, maks. giriş derecesi {max_in_deg}] {n_exact}/{len(rows)} karşılaştırma birebir aynı; "
              f"en büyük göreli fark {max_diff:.2e} -> {'GEÇTİ' if ok else 'KALDI'}")
        for r in rows:
            if not r[1]:
                print(f"    birebir değil: {r[0]}  (göreli fark {r[2]:.2e})")
    print("SONUÇ:", "port, orijinal DGL'li kodla eşdeğer" if all_ok else "FARK BULUNDU", f"(en büyük göreli fark {worst:.2e})")
    return 0 if all_ok else 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile fgcl_bench/shims/openfgl/__init__.py
# Yalnızca POWER (flcore/Ours.py) için gereken OpenFGL parçası; bkz. utils/basic_utils.py


In [ ]:
%%writefile fgcl_bench/shims/openfgl/utils/__init__.py



In [ ]:
%%writefile fgcl_bench/shims/openfgl/utils/basic_utils.py
# POWER'ın flcore/Ours.py'si OpenFGL'den yalnızca bu fonksiyonu import ediyor.
# Kaynak: https://github.com/xkLi-Allen/OpenFGL (commit 5483768), openfgl/utils/basic_utils.py (aynen kopya).
import torch


def idx_to_mask_tensor(idx_list, length):
    """
    Converts a list of indices to a tensor mask of a specified length.

    Args:
        idx_list (list[int]): List of indices that should be marked as 1 in the mask.
        length (int): Total length of the mask tensor.

    Returns:
        torch.Tensor: A binary mask tensor where positions corresponding to indices in idx_list are set to 1.
    """
    mask = torch.zeros(length)
    mask[idx_list] = 1
    return mask


### 5.4 Değişikliklerin ve protokol kararlarının listesi

| Dosya / yer | Ne değişti | Neden |
|---|---|---|
| FedFST `datasets/`, `algorithm/`, `models/` (P0) | boş `__init__.py` eklendi | Colab'daki Hugging Face `datasets` paketi, FedFST'nin namespace package olan `datasets` klasörünü gölgeliyor |
| FedFST `datasets/dataset_loader.py` (P1) | `torch_sparse` import'u kaldırıldı | ModuleNotFoundError; SparseTensor kullanılmıyor |
| FedFST `algorithm/Ours.py` (P2) | var olmayan iki fonksiyonun import'u kaldırıldı | ImportError; fonksiyonlar kullanılmıyor |
| FedFST `args.py`, `main.py` (P3a/P3b) | `--kd_ce_weight 0.5`, `--gen_ce_weight 1.0` eklendi; tanımsız `S_high_D_weight` print'i düzeltildi | AttributeError; varsayılanlar Ours.py'deki getattr varsayılanlarıyla aynı |
| FedFST `datasets/dataset_loader.py` (P4a/P4b) | Minesweeper ve Squirrel eklendi | yükleyici bu iki veri setini kabul etmiyordu |
| FedFST `utils.py`, `algorithm/Ours.py`, `datasets/partition.py` (P5a–P5f) | Python döngüleri vektörize edildi | hız; çıktı, gradyan ve RNG durumu birebir aynı (Bölüm 6.1) |
| FedFST eğitim döngüsü (`runner_fedfst.py`) | `main.py`'deki döngü aynen; jeneratör eğitimi thread'ler yerine aynı sırayla ardışık; değerlendirme ortak fonksiyonla; kayıp grafikleri çizilmiyor | tekrarlanabilirlik (thread'ler aynı RNG'yi paylaşıyor); ortak değerlendirme |
| POWER `flcore/__init__.py` | boş dosya eklendi | P0 ile aynı önlem |
| POWER veri katmanı (`runner_power.py`) | `data.py` / OpenFGL yerine ortak cache; `task_data` yalnızca görev alt-çizgisini döndürecek şekilde alt sınıfta değiştirildi; `config.py` argümanları varsayılanlarla okunup veri setine göre dolduruldu | ortak protokol: aynı istemciler, görevler, maskeler, aynı görev çizgesi; OpenFGL kurulmuyor |
| POWER değerlendirme | yerel model yerine global model (ortak fonksiyon) | ortak protokol |
| FedSPA (`fedspa_port.py`) | DGL `update_all` → `index_add_`; DGL `adjacency_matrix` → yoğun A[src, dst]; sabit `cuda` ve sabit boyutlar kaldırıldı; `att_struc_low` sıfırla ilklendi | Colab'da DGL kurulamıyor; ilklenmemiş bellek birleştirmeyi rastgele bozuyordu |
| FedSPA eğitim (`runner_fedspa.py`) | görev akışında ardışık ince ayar; her görevde `label_unk`, adj matrisleri, $S_{high}$ ve HBDA homofili ağırlıkları görev alt-çizgisinden yeniden; Adam lr 0.01 / wd 5e-4, görev başına 10 round × 3 tam-çizge adımı (orijinal: SGD lr 0.1, 200 round, 4 adım); her round'da 3 istemci de katılıyor (orijinal flgo 'md' örneklemesi yerine koymalı); değerlendirmede eval modu | FedSPA bir FGL yöntemi; ortak FGCL bütçesi (POWER makalesindeki ortak ayar) |
| Protokol | tüm yöntemler aynı cache'ten okuyor; maske hash'i her koşuda doğrulanıp JSON'a yazılıyor | adil karşılaştırma |

## 6. Eşdeğerlik testleri ve sağlamalar
### 6.1 FedFST vektörizasyonu
Test, orijinal fonksiyonları yamalı repodan değil **git geçmişinden** (`git show d83fac0:<dosya>`) okur ve yamalı sürümlerle aynı seed'le karşılaştırır. Çıktıların (dtype, şekil ve gerekirse bellek düzeni dahil), gradyanların ve çağrıdan sonraki RNG durumunun birebir aynı olması istenir. Böylece rastgele seçimlerin aynı generator'dan aynı sırayla çekildiği doğrulanır. GPU varsa indirgeme sırasından kaynaklanabilecek farklar için KL testi GPU'da ayrıca toleransla çalışır (RNG durumu yine birebir istenir). Testin gerçekten fark yakalayabildiğini yerelde kasıtlı hatalar enjekte ederek de denedim: yanlış RNG kullanımı ve değiştirilmiş oran yakalandı.

In [ ]:
from fgcl_bench.data import prefetch_raw
prefetch_raw(WORK / "data", "cora")            # Louvain testi Cora'yı kullanıyor
cmd = [sys.executable, "-m", "fgcl_bench.equivalence", "--fedfst", str(FEDFST), "--data_root", str(WORK / "data")]
if torch.cuda.is_available():
    cmd.append("--cuda")
res = subprocess.run(cmd, cwd=BASE, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr[-3000:])
    raise RuntimeError("Vektörizasyon eşdeğerlik testi geçmedi.")

### 6.2 FedSPA: DGL → saf PyTorch
Colab'daki Python/torch sürümü için DGL wheel'i yok (PyPI'da Python 3.13 için yalnızca çok eski 0.1.0–0.1.3 sürümleri var). Bu yüzden FedSPA'nın DGL kullanan iki parçası saf PyTorch ile yeniden yazıldı:
* `graph.update_all(mp_func_spa, agg_func_spa)` → `spa_update_all`: mesajlar DGL'deki gibi tüm kenarlar için `index_select` ile toplanır, kenar türü maskesiyle çarpılır ve `index_add_` ile hedef düğümlerde toplanır;
* `g.adjacency_matrix()` → yoğun $A[\mathrm{src}, \mathrm{dst}] = 1$ (DGL 2.1'de satır = kaynak; yerelde doğrulandı).

Port, yerelde ayrı bir ortamda (Python 3.11, torch 2.2.1, DGL 2.1.0) orijinal DGL'li koda karşı test edildi (`fgcl_bench/fedspa_dgl_check.py`). Orijinal sınıflar FedSPA reposundan okundu, aynı ağırlıklar `load_state_dict(strict=True)` ile yüklendi ve aynı seed kullanıldı. Her çizgede 152 karşılaştırma yapıldı: komşuluk matrisleri; iki dalın (fr listesi var/yok) train ve eval çıktıları; tüm gradyanlar; `requires_grad` bayrakları; `fish_cal` Fisher değerleri; 3 istemcili HBDA birleştirmesinin sonucu. Yerel çıktı:

```
torch 2.2.1+cu121, dgl 2.1.0; torch.sum(dim=1) derece < 5 iken sıralı toplamla birebir
[rastgele N=40 E=47, maks. giriş derecesi 3] 152/152 karşılaştırma birebir aynı; en büyük göreli fark 0.00e+00 -> GEÇTİ
[rastgele N=12 E=29, maks. giriş derecesi 4] 152/152 karşılaştırma birebir aynı; en büyük göreli fark 0.00e+00 -> GEÇTİ
[rastgele N=40 E=156, maks. giriş derecesi 8] 152/152 karşılaştırma birebir aynı; en büyük göreli fark 0.00e+00 -> GEÇTİ
[rastgele N=200 E=1475, maks. giriş derecesi 17] 86/152 karşılaştırma birebir aynı; en büyük göreli fark 1.31e-06 -> GEÇTİ
[squirrel istemci0/görev0 N=864 E=25474, maks. giriş derecesi 370] 74/152 karşılaştırma birebir aynı; en büyük göreli fark 1.68e-05 -> GEÇTİ
SONUÇ: port, orijinal DGL'li kodla eşdeğer (en büyük göreli fark 1.68e-05)
```

Düşük dereceli çizgelerde tüm çıktılar bit düzeyinde aynı. Yüksek dereceli düğümlerde kalan fark yalnızca toplama sırasından geliyor. DGL'in `agg_func_spa`'sı her düğümün posta kutusunu `torch.sum(dim=1)` ile topluyor; PyTorch bu toplamı tensör şekline göre farklı sırayla yapıyor: ölçtüğüm iç boyutlarda toplam, derece 5 ya da 18'den itibaren sıralı toplamdan ayrılıyor (ör. iç boyut 2 iken 5'ten, 128 iken 18'den itibaren). `index_add_` ise sıralı topluyor. Test sırasında orijinal kodun ayrıca bir hatası ortaya çıktı: ilklenmemiş `att_struc_low` belleği 150 denemenin 15'inde bir modele NaN soktu ve bunların 8'inde orijinal birleştirme `ZeroDivisionError` ile çöktü. Port bu parametreyi sıfırla ilkliyor; parametre ileri geçişte kullanılmadığı için hesap değişmiyor.

### 6.3 Sağlama: harness'teki FedFST ile orijinal `main.py`
Yerelde (CPU, tek iş parçacığı) yamalı orijinal `main.py`, jeneratör eğitimini yapan `ThreadPoolExecutor` tek iş parçacığına indirilerek (`fgcl_bench/run_main_sequential.py`; `main.py`'nin kodu değişmiyor) Cora'da seed 24 ile çalıştırıldı:

| Koşu | AA | AF | Son doğruluk matrisi |
|---|---|---|---|
| Harness (`fgcl_bench.run --method fedfst`) | 61.7726 | 32.7114 | [[100, 0, 0], [99.15, 94.64, 0], [85.47, 43.75, 56.10]] |
| Orijinal `main.py`, sıralı jeneratör | 61.7726 | 32.7114 | aynı |
| Orijinal `main.py`, thread'li (1. koşu) | 38.9687 | 55.5823 | [[100, 0, 0], [92.31, 86.61, 0], [70.09, 5.36, 41.46]] |
| Orijinal `main.py`, thread'li (2. koşu) | 80.6263 | 12.9273 | [[100, 0, 0], [94.87, 93.75, 0], [99.15, 68.75, 73.98]] |

Harness, orijinal döngüyle bit düzeyinde aynı sonucu veriyor. Thread'li orijinal ise aynı seed'le iki koşuda çok farklı sonuçlar veriyor. Bu, hem tekrarlanamazlığın kaynağını hem de FedFST'nin tek seed'li sonuçlarının RNG akışına ne kadar duyarlı olduğunu gösteriyor. GPU'da GAT ve `index_add` atomik toplamalar kullandığı için Colab'daki koşular bit düzeyinde tekrarlanamaz; aynı karşılaştırma Colab'da isteğe bağlı olarak Bölüm 9.6'da çalıştırılabilir.

### 6.4 Sağlama: POWER
**`edge_masking`** (Bölüm 2.1): aşağıdaki hücre POWER'ın kendi fonksiyonunu küçük bir örnekte çalıştırıyor. İki "görev" var, {0, 1, 2} ve {3, 4, 5}; görevler arası kenarlar 0–5 ve 2–3.

In [ ]:
demo = '''
import sys, torch
sys.path.insert(0, sys.argv[1])
from utils import edge_masking            # POWER/utils.py
edge_index = torch.tensor([[0,1,1,2,2,3,3,4,4,5,0,5],[1,0,2,1,3,2,4,3,5,4,5,0]])
handled = torch.tensor([True, True, True, False, False, False])   # görev 1'in düğümleri (task_data'daki gibi bool maske)
out = edge_masking(edge_index, handled=handled, device=torch.device("cpu"))
inter = sorted({(int(a), int(b)) for a, b in out.t() if (int(a) < 3) != (int(b) < 3)})
print("girdi kenarı:", edge_index.size(1), "| çıktı kenarı:", out.size(1))
print("çıktıda kalan görevler arası kenarlar:", inter)
print("self-loop eklenen düğümler:", sorted({int(a) for a, b in out.t() if a == b}))
'''
print(subprocess.run([sys.executable, "-c", demo, str(POWER_DIR)], capture_output=True, text=True).stdout)

**Orijinal betikle Cora.** POWER yerelde kendi orijinal betik yoluyla bir kez çalıştırıldı (`optuna_search.py`'nin `__main__` bloğu ve `objective()`'i, Optuna olmadan, `config.py` varsayılanlarıyla, tek deneme; OpenFGL veri katmanı, CPU; betik: `tools/power_original_once.py`):

| Koşu (POWER'ın kendi veri hattı, seed 2025) | Görev sıraları | AA | AF |
|---|---|---|---|
| Varsayılan değerlendirme (yerel modeller) | istemciye göre karışık: [4,5]⇒[2,3]⇒[0,1], [4,5]⇒[2,3]⇒[0,1], [2,3]⇒[0,1]⇒[4,5] | 64.07 | 8.14 |
| Global model değerlendirmesi (`global_eval`) | aynı | 57.49 | 2.62 |
| Varsayılan değerlendirme, görev sırası tüm istemcilerde aynı (`tools/power_original_sync_order.py`) | [0,1]⇒[2,3]⇒[4,5] | 30.75 | 1.90 |

Üçüncü koşuda görev 1 ve görev 2'nin doğruluğu, kendi eğitimlerinin hemen ardından %0 (doğruluk matrisi: [[96.05, 0, 0], [91.67, 0, 0], [18.86, 73.39, 0]]). Yani β = 0.01 iken tüm istemciler aynı anda yeni sınıflara geçince yeni görev 10 round içinde öğrenilemiyor. Ortak protokolde de aynı durum görülüyor (Bölüm 9 ve 10). Bu, adaptörden değil yöntemin varsayılan ayarı ile görev senaryosunun etkileşiminden kaynaklanıyor.

## 7. Veri setleri
Veri setleri PyTorch Geometric ile yükleniyor. Ham dosyalar, PyG'nin kullandığı GitHub depolarından (`raw.githubusercontent.com`) Drive'daki `data/` klasörüne indiriliyor; Squirrel için PyG'nin sabitlediği geom-gcn commit'i kullanılıyor. Bölümleme ve görevler, **FedFST'nin kendi fonksiyonlarıyla** ve `main.py`'deki sırayla üretiliyor:

`set_seed` → `load_dataset` → `louvain_partitioner` (python-louvain, 3 istemci) → sınıf sırasının seed'le karıştırılması (`load_server_clients`) → her istemcide `class_to_task` (sınıf başına 0.6/0.2/0.2 eğitim/doğrulama/test).

Sonuç her (veri seti, seed) için bir kez `cache/<veri>_seed<s>.pt` dosyasına yazılıyor ve üç yöntem de buradan okuyor. Görev çizgesi, FedFST'deki gibi yalnızca görevin sınıflarındaki düğümler arasındaki kenarları içeriyor (görevler arası kenar yok). Artan sınıf atılıyor: `out_dim = (C // k)·k`. Minesweeper 2 sınıflı olduğu için görev başına 1 sınıf kullanılıyor (2 görev, her görevde tek sınıf). Bu yapay ve zor bir senaryo: her görevde model "her düğüm bu sınıftandır" demeyi öğrenebiliyor ve sınıf-artımlı değerlendirmede AA kolayca 50 civarına kilitleniyor (Bölüm 10.1).

In [ ]:
ALL_DATASETS = ["cora", "citeseer", "actor", "squirrel", "minesweeper"]
DATA_SEED = 24      # FedFST varsayılanı; CONFIG'de başka seed'ler seçilirse cache'leri koşucu ayrıca üretir
for ds in ALL_DATASETS:
    r = subprocess.run([sys.executable, "-m", "fgcl_bench.data", "--fedfst", str(FEDFST), "--work_dir", str(WORK),
                        "--dataset", ds, "--seed", str(DATA_SEED)], cwd=BASE, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError(f"{ds} hazırlanamadı")
    print(r.stdout.strip().splitlines()[-1])

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from fgcl_bench.data import load_cache
from fgcl_bench.common import DATASETS as DS_INFO

rows, task_rows = [], []
for ds in ALL_DATASETS:
    m = load_cache(WORK, ds, DATA_SEED)["meta"]
    st = m["stats"]
    rows.append({"Veri seti": DS_INFO[ds]["title"], "Düğüm": st["nodes"], "Kenar": st["edges"],
                 "Öznitelik": st["features"], "Sınıf": st["classes"], "Kenar homofilisi": round(st["edge_homophily"], 3),
                 "Sınıf/görev": m["per_task"], "Görev": m["num_tasks"],
                 "Görev sınıfları (orijinal id)": " ⇒ ".join(str(c) for c in m["task_classes"]),
                 "Atılan sınıf": ", ".join(map(str, m["dropped_classes"])) or "—",
                 "İstemci düğümleri": " / ".join(str(c["nodes"]) for c in st["clients"])})
    for cid, c in enumerate(st["clients"]):
        for t, task in enumerate(c["tasks"]):
            task_rows.append({"Veri seti": DS_INFO[ds]["title"], "İstemci": cid, "Görev": t + 1,
                              "Görev düğümü": task["nodes"], "Eğitim": task["train"], "Doğrulama": task["valid"],
                              "Test": task["test"], "Görev içi kenar": task["edges"]})
display(Markdown(f"**Veri setleri ve görevler (seed {DATA_SEED})**"))
display(pd.DataFrame(rows).set_index("Veri seti"))
display(Markdown("**İstemci × görev düğüm sayıları**"))
task_df = pd.DataFrame(task_rows)
task_df["Veri seti"] = pd.Categorical(task_df["Veri seti"], [DS_INFO[d]["title"] for d in ALL_DATASETS], ordered=True)
display(task_df.set_index(["Veri seti", "İstemci", "Görev"]).unstack("Görev").astype("Int64"))

## 8. Deneyler
### 8.1 CONFIG
Koşu sırası: önce Cora'da üç yöntem (elde hemen tam bir karşılaştırma olsun diye), sonra CiteSeer, Actor, Squirrel ve Minesweeper. Ortak bütçe görev başına 10 communication round ve round başına 3 yerel epoch (tam-çizge). Bir round, her istemcinin yerel epoch'ları ile sunucudaki birleştirmeden oluşur. Görev başına 15 round'un üstü kabul edilmiyor.

In [ ]:
# ============================== CONFIG ==============================
DATASETS = ["cora", "citeseer", "actor", "squirrel", "minesweeper"]   # çalıştırma sırası
METHODS = ["fedfst", "power", "fedspa"]
ROUNDS = 10            # görev başına communication round (en fazla 15)
LOCAL_EPOCHS = 3       # round başına yerel epoch
SEEDS = [24]           # FedFST varsayılanı
SMOKE = False          # True: 1 round, 1 epoch, FedFST jeneratör/KD 2/3 epoch, POWER L-BFGS 3 iterasyon, yalnızca Cora
RUN_TIMEOUT_MIN = 90   # tek bir koşu için üst sınır (dakika)
# ====================================================================

In [ ]:
assert ROUNDS <= 15, "Görev başına en fazla 15 communication round."
if SMOKE:
    DATASETS = ["cora"]
RESULTS_DIR = WORK / ("results_smoke" if SMOKE else "results")
LOG_DIR = WORK / ("logs_smoke" if SMOKE else "logs")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
RUNS = [(ds, m, s) for s in SEEDS for ds in DATASETS for m in METHODS]
print(f"{len(RUNS)} koşu | SMOKE={SMOKE} | sonuçlar: {RESULTS_DIR} | log'lar: {LOG_DIR}")

### 8.2 Koşucu
Her koşu ayrı bir süreç. Çıktısının tamamı Drive'daki log dosyasına gidiyor; burada yalnızca bir özet satırı ve log'un son satırları görünüyor. Biten koşunun JSON'u Drive'a yazıldığı için hücre yeniden çalıştırılınca o koşu atlanıyor. Her koşudan sonra, o ana kadarki sürelerden kalan süre tahmin ediliyor.

In [ ]:
def run_name(ds, m, s):
    return f"{ds}__{m}__seed{s}"

def estimate_remaining(done_times, remaining):
    if not done_times:
        return None
    overall = sum(done_times.values()) / len(done_times)
    total = 0.0
    for ds, m, s in remaining:
        same = [t for (d2, m2, _), t in done_times.items() if m2 == m]
        total += sum(same) / len(same) if same else overall
    return total

done_times, failed = {}, []
for i, (ds, m, s) in enumerate(RUNS, 1):
    name = run_name(ds, m, s)
    out = RESULTS_DIR / f"{name}.json"
    if out.exists():
        r = json.loads(out.read_text())
        done_times[(ds, m, s)] = r["time_sec"]
        print(f"[{i}/{len(RUNS)}] {name}: daha önce bitmiş (AA={r['AA']:.2f}, AF={r['AF']:.2f}) -> atlandı")
        continue
    cmd = [sys.executable, "-u", "-m", "fgcl_bench.run", "--method", m, "--dataset", ds, "--seed", str(s),
           "--rounds", str(ROUNDS), "--local_epochs", str(LOCAL_EPOCHS), "--src_dir", str(SRC), "--work_dir", str(WORK)]
    if SMOKE:
        cmd.append("--smoke")
    log = LOG_DIR / f"{name}.log"
    t0 = time.time()
    with open(log, "w") as f:
        try:
            rc = subprocess.run(cmd, cwd=BASE, stdout=f, stderr=subprocess.STDOUT, timeout=RUN_TIMEOUT_MIN * 60).returncode
        except subprocess.TimeoutExpired:
            rc = "zaman aşımı"
    elapsed = time.time() - t0
    tail = log.read_text(errors="replace").strip().splitlines()[-3:]
    if rc == 0 and out.exists():
        r = json.loads(out.read_text())
        done_times[(ds, m, s)] = elapsed
        print(f"[{i}/{len(RUNS)}] {name}: AA={r['AA']:.2f} AF={r['AF']:.2f} | {elapsed/60:.1f} dk | {r['env']['device']}")
    else:
        failed.append(name)
        print(f"[{i}/{len(RUNS)}] {name}: HATA ({rc}) | log: {log}")
    for line in tail:
        print("    │ " + line[:150])
    rest = estimate_remaining(done_times, [x for x in RUNS[i:] if not (RESULTS_DIR / f"{run_name(*x)}.json").exists()])
    if rest:
        print(f"    tahmini kalan süre: ~{rest/60:.0f} dk")
print("\nBiten:", len(done_times), "| hatalı:", failed or "yok")

In [ ]:
# İlerleme tablosu (✓: sonuç var)
status = pd.DataFrame([{"Veri seti": DS_INFO[ds]["title"], "Seed": s, "Yöntem": m,
                        "Durum": "✓" if (RESULTS_DIR / f"{run_name(ds, m, s)}.json").exists() else "—"}
                       for ds, m, s in RUNS]).pivot_table(index=["Veri seti", "Seed"], columns="Yöntem",
                                                           values="Durum", aggfunc="first")
display(status)

## 9. Sonuçlar
### 9.1 Sonuçların yüklenmesi ve maske kontrolü
Her JSON, koşunun beslediği görev maskelerinin hash'ini taşıyor. Aşağıdaki hücre, aynı (veri seti, seed) için üç yöntemin hash'lerinin aynı olduğunu ve cache'teki hash'le eşleştiğini doğruluyor.

In [ ]:
from fgcl_bench.common import METHODS as METHOD_TITLES

results = {}
for ds, m, s in RUNS:
    p = RESULTS_DIR / f"{run_name(ds, m, s)}.json"
    if p.exists():
        results[(ds, m, s)] = json.loads(p.read_text())
print(f"{len(results)}/{len(RUNS)} koşunun sonucu bulundu.")

for ds in DATASETS:
    for s in SEEDS:
        hashes = {m: results[(ds, m, s)]["mask_hash"] for m in METHODS if (ds, m, s) in results}
        if not hashes:
            continue
        cache_hash = load_cache(WORK, ds, s)["hash"]
        assert all(h == cache_hash for h in hashes.values()), f"{ds}/seed{s}: maske hash'leri farklı! {hashes}"
        print(f"{DS_INFO[ds]['title']:12s} seed {s}: {len(hashes)} yöntem aynı maskeleri kullandı (hash {cache_hash[:12]})")

### 9.2 AA / AF tablosu
Her veri setinde en iyi değer kalın: AA için en yüksek, AF için en düşük. Birden fazla seed varsa ortalama ± standart sapma gösteriliyor. AF tek başına okunmamalı. Negatif AF, bir görevin doğruluğunun sonradan arttığını gösterir. Düşük AF ile birlikte görev sonu doğruluğu (köşegen) da düşükse bu "az unutma" değil, yeni görevin hiç öğrenilmediği anlamına gelir. Bölüm 9.5'teki tablo bu ayrımı gösteriyor.

In [ ]:
import numpy as np

def agg(ds, m, key):
    vals = [results[(ds, m, s)][key] for s in SEEDS if (ds, m, s) in results]
    return (float(np.mean(vals)), float(np.std(vals)), len(vals)) if vals else None

def cell(v, best):
    if v is None:
        return "—"
    txt = f"{v[0]:.2f}" + (f" ± {v[1]:.2f}" if v[2] > 1 else "")
    return f"**{txt}**" if best else txt

header = "| Veri seti | " + " | ".join(f"AA ↑ {METHOD_TITLES[m]}" for m in METHODS) + " | " + \
         " | ".join(f"AF ↓ {METHOD_TITLES[m]}" for m in METHODS) + " |"
lines = [header, "|" + "---|" * (1 + 2 * len(METHODS))]
for ds in DATASETS:
    aa = {m: agg(ds, m, "AA") for m in METHODS}
    af = {m: agg(ds, m, "AF") for m in METHODS}
    best_aa = max((v[0] for v in aa.values() if v), default=None)
    best_af = min((v[0] for v in af.values() if v), default=None)
    lines.append(f"| {DS_INFO[ds]['title']} | " +
                 " | ".join(cell(aa[m], aa[m] is not None and aa[m][0] == best_aa) for m in METHODS) + " | " +
                 " | ".join(cell(af[m], af[m] is not None and af[m][0] == best_af) for m in METHODS) + " |")
display(Markdown("\n".join(lines)))

### 9.3 Doğruluk matrisleri
Satır: eğitimi biten görev; sütun: değerlendirilen görev. Hücreler test doğruluğunu (%) gösteriyor. Köşegen, bir görevin hemen ardından ölçülen doğruluk; son satır, akışın sonundaki durum.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

INK, INK2, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7", "#fcfcfb"
METHOD_COLORS = {"fedfst": "#2a78d6", "power": "#eb6834", "fedspa": "#1baf7a"}
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                                                     "#256abf", "#184f95", "#0d366b"])
SEQ.set_bad("#f0efec")
plt.rcParams.update({"font.size": 9, "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.titlecolor": INK, "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE})

seed0 = SEEDS[0]
ds_present = [ds for ds in DATASETS if any((ds, m, seed0) in results for m in METHODS)]
if ds_present:
    fig, axes = plt.subplots(len(ds_present), len(METHODS), figsize=(3.1 * len(METHODS), 2.7 * len(ds_present)),
                             squeeze=False, constrained_layout=True)
    im = None
    for r_i, ds in enumerate(ds_present):
        for c_i, m in enumerate(METHODS):
            ax = axes[r_i][c_i]
            res_ = results.get((ds, m, seed0))
            if res_ is None:
                ax.set_axis_off()
                continue
            A = np.array(res_["acc_matrix"], dtype=float)
            T = A.shape[0]
            A_masked = np.where(np.tril(np.ones_like(A)) > 0, A, np.nan)
            im = ax.imshow(A_masked, cmap=SEQ, vmin=0, vmax=100)
            for i in range(T):
                for j in range(i + 1):
                    ax.text(j, i, f"{A[i, j]:.0f}", ha="center", va="center", fontsize=9,
                            color="white" if A[i, j] >= 55 else INK)
            ax.set_xticks(range(T), [f"G{j + 1}" for j in range(T)])
            ax.set_yticks(range(T), [f"G{j + 1}" for j in range(T)])
            ax.set_title(f"{DS_INFO[ds]['title']} · {METHOD_TITLES[m]}\nAA {res_['AA']:.1f} · AF {res_['AF']:.1f}",
                         fontsize=9)
            if r_i == len(ds_present) - 1:
                ax.set_xlabel("değerlendirilen görev")
            if c_i == 0:
                ax.set_ylabel("eğitimi biten görev")
            for side in ax.spines.values():
                side.set_visible(False)
    if im is not None:
        fig.colorbar(im, ax=axes, shrink=0.6, label="test doğruluğu (%)")
    fig.savefig(WORK / ("acc_matrices_smoke.png" if SMOKE else "acc_matrices.png"), dpi=150)
    plt.show()

### 9.4 AA ve AF karşılaştırması
İki ölçü farklı yönlerde iyidir (AA ↑, AF ↓), bu yüzden ayrı panellerde gösteriliyor.

In [ ]:
if ds_present:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
    x = np.arange(len(ds_present))
    width = 0.26
    for ax, key, label in [(axes[0], "AA", "AA (%) ↑"), (axes[1], "AF", "AF (%) ↓")]:
        for k, m in enumerate(METHODS):
            vals = [agg(ds, m, key)[0] if agg(ds, m, key) else np.nan for ds in ds_present]
            ax.bar(x + (k - 1) * width, vals, width, color=METHOD_COLORS[m], edgecolor=SURFACE, linewidth=1.5,
                   label=METHOD_TITLES[m], zorder=3)
        ax.axhline(0, color=AXIS, linewidth=1, zorder=4)
        ax.set_xticks(x, [DS_INFO[ds]["title"] for ds in ds_present])
        ax.set_ylabel(label)
        ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
    axes[0].legend(ncol=3, frameon=False, loc="upper left", bbox_to_anchor=(0, 1.15))
    fig.savefig(WORK / ("aa_af_smoke.png" if SMOKE else "aa_af.png"), dpi=150)
    plt.show()

### 9.5 Ayrıntılı göstergeler ve süreler
* **Görev sonu doğruluğu (ort. köşegen):** her görevin, kendi eğitimi biter bitmez ölçülen doğruluğu. Yeni bilgiyi öğrenebilme (plastisite) göstergesi.
* **Eski görevler (son):** akış bitince önceki görevlerin ortalama doğruluğu. Kalıcılık (stabilite) göstergesi.
* **Son görev:** son görevin doğruluğu.

In [ ]:
detail = []
for (ds, m, s), r in sorted(results.items(), key=lambda kv: (DATASETS.index(kv[0][0]), METHODS.index(kv[0][1]), kv[0][2])):
    A = np.array(r["acc_matrix"], dtype=float)
    T = A.shape[0]
    detail.append({"Veri seti": DS_INFO[ds]["title"], "Yöntem": METHOD_TITLES[m], "Seed": s,
                   "AA": round(r["AA"], 2), "AF": round(r["AF"], 2),
                   "Görev sonu doğruluğu (ort. köşegen)": round(float(np.mean(np.diag(A))), 2),
                   "Eski görevler (son)": round(float(np.mean(A[T - 1, :T - 1])), 2) if T > 1 else np.nan,
                   "Son görev": round(float(A[T - 1, T - 1]), 2),
                   "Süre (dk)": round(r["time_sec"] / 60, 2),
                   "Görev süreleri (sn)": " / ".join(f"{t:.0f}" for t in r.get("task_seconds", [])),
                   "Cihaz": r["env"]["device"]})
detail_df = pd.DataFrame(detail).set_index(["Veri seti", "Yöntem", "Seed"])
display(detail_df)
if detail:
    total_min = sum(r["time_sec"] for r in results.values()) / 60
    display(Markdown(f"**Toplam deney süresi:** {total_min:.1f} dk ({len(results)} koşu)"))
    print("\nOtomatik özet (Bölüm 10'daki yorumlar bu sayılarla birlikte okunmalı):")
    for ds in ds_present:
        aa = {m: agg(ds, m, "AA")[0] for m in METHODS if agg(ds, m, "AA")}
        af = {m: agg(ds, m, "AF")[0] for m in METHODS if agg(ds, m, "AF")}
        best_aa = max(aa, key=aa.get)
        best_af = min(af, key=af.get)
        print(f"  {DS_INFO[ds]['title']:12s} en yüksek AA: {METHOD_TITLES[best_aa]} ({aa[best_aa]:.2f}) | "
              f"en düşük AF: {METHOD_TITLES[best_af]} ({af[best_af]:.2f})")

### 9.6 İsteğe bağlı sağlama: Colab'da harness ile `main.py`
`RUN_MAIN_PY_CHECK = True` yapılırsa yamalı orijinal `main.py`, jeneratör eğitimi tek iş parçacığına indirilerek Cora'da (seed 24, varsayılan ayarlar) çalıştırılır ve sonucu harness'in sonucuyla karşılaştırılır. Süre, FedFST'nin Cora koşusu kadardır. Bölüm 6.3'teki yerel karşılaştırma CPU'da birebir aynı sonucu verdi. GPU'da atomik toplamalar nedeniyle küçük farklar beklenebilir.

In [ ]:
RUN_MAIN_PY_CHECK = False
if RUN_MAIN_PY_CHECK and not SMOKE:
    log = WORK / "logs" / "sanity_main_py_cora.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    with open(log, "w") as f:
        subprocess.run([sys.executable, "-m", "fgcl_bench.run_main_sequential", str(FEDFST), "--dataset_name", "cora",
                        "--seed", "24", "--device_id", "0", "--dataset_dir", str(WORK / "data"),
                        "--save_dir", str(BASE / "main_py_plots")], cwd=BASE, stdout=f, stderr=subprocess.STDOUT, check=True)
    txt = log.read_text()
    main_aa = float(re.search(r"Final AA: ([-\d.]+)", txt).group(1))
    main_af = float(re.search(r"Final AF: ([-\d.]+)", txt).group(1))
    print(f"main.py (sıralı jeneratör): AA={main_aa:.4f} AF={main_af:.4f}")
    if ("cora", "fedfst", 24) in results:
        r = results[("cora", "fedfst", 24)]
        print(f"harness:                    AA={r['AA']:.4f} AF={r['AF']:.4f}")
else:
    print("Atlandı (RUN_MAIN_PY_CHECK = False). Yerel CPU karşılaştırması Bölüm 6.3'te.")

## 10. Tartışma
Aşağıdaki yorumlar Bölüm 9'daki tablolara ve kodu okurken gördüklerime dayanıyor. Bölüm 9.5'teki otomatik özet bu koşunun sayılarını veriyor; metindeki bir genelleme bu sayılarla çelişirse sayılar esas alınmalı. Yerel CPU testlerinde (aynı kod, aynı seed) gördüğüm değerleri "yerel testte" diye belirttim. Colab'daki GPU koşuları, atomik toplamalar ve FedFST'nin RNG duyarlılığı nedeniyle bunlardan farklı çıkabilir.

### 10.1 Homofilik ve heterofilik veri setleri
* **Cora ve CiteSeer** yüksek homofililidir: kenar homofilisi sırasıyla ≈0.81 ve ≈0.74 (Bölüm 7). GAT'ın komşu toplaması burada işe yarıyor ve görev sonu doğrulukları yüksek. Yerel testte FedFST bu iki veri setinde en yüksek AA'yı verdi. FedSPA'ya karşı farkı eski görevleri korumasından (son satırdaki eski görev doğrulukları), POWER'a karşı farkı ise yeni görevleri öğrenebilmesinden (köşegen) geliyordu. Bölüm 9.5'teki "Görev sonu doğruluğu" ve "Eski görevler (son)" sütunları bu ayrımı Colab sonuçları için gösteriyor.
* **Actor ve Squirrel** heterofiliktir (≈0.22): komşular çoğunlukla farklı sınıftan. Mesaj geçirme ayırt edici bilgiyi taşımakta zorlanıyor ve görev sonu doğrulukları da düşük. FedFST'nin $S_{high}$ eşleştirmesi tam bu yüksek frekans bilgisini sentetik tekrar çizgesinde korumaya yönelik. FedSPA'nın homofilik/heterofilik/bilinmeyen kanal ayrımı da heterofiliye yönelik, ama unutmayı önlemiyor.
* **Minesweeper**'ın kenar homofilisi ≈0.68 görünüyor. Ancak sınıflar %80 / %20 dengesiz olduğu için rastgele kenarlarda bile beklenen kenar homofilisi $0.8^2+0.2^2=0.68$'dir. Ölçülen değer bu taban çizgisine çok yakın; yani etiketler komşulukla neredeyse ilişkisiz ve veri seti pratikte heterofilik sayılmalı. Görev başına tek sınıf olduğu için sınıf-artımlı değerlendirmede iki uç durum var. Model yalnızca son sınıfı tahmin ederse AA = 50 ve AF = 100 olur (tam unutma). Yalnızca ilk sınıfı tahmin etmeye devam ederse yine AA = 50, ama AF = 0 olur (yeni görev hiç öğrenilmemiş). Bu nedenle Minesweeper'da AA tek başına yanıltıcıdır; doğruluk matrisiyle birlikte okunmalı.

### 10.2 Unutma analizi
* **FedSPA** unutmaya karşı bir mekanizma içermiyor. Her görev yalnızca yeni sınıfların etiketleriyle eğitildiği için çıkış katmanı yeni sınıflara kayıyor ve eski görevlerin doğruluğu çöküyor. Yerel testte Cora'da köşegen %89–99 iken son satırdaki eski görev doğrulukları 0'dı (AF ≈ 96). HBDA, istemciler arasındaki homofili farkına göre ağırlık veriyor; zamandaki (görevler arası) kaymayı ele almıyor. FedSPA'nın yüksek AF'si, POWER makalesindeki "FL/FGL fine-tuning" yöntemlerinden beklenen davranış.
* **POWER**, varsayılan β = 0.01 ile mevcut görevin kaybına yalnızca 0.01 ağırlık veriyor. Ortak protokolde tüm istemciler aynı anda yeni sınıflara geçiyor ve yerel testte yeni görevler 10 round içinde neredeyse hiç öğrenilemedi (görev sonu doğrulukları ≈ 0). Bu durumda düşük AF "az unutma" anlamına gelmiyor; yeni bilgi hiç gelmemiş oluyor. Cora'da AF negatif bile çıktı: görev 2'nin doğruluğu görev 3 sırasında 0'dan yükseldi. O sırada görev 2'nin sınıfları yalnızca tampondaki tekrar düğümleri ve sunucudaki sözde prototipler üzerinden eğitiliyordu. Bunun adaptörden değil, β ile görev senaryosunun etkileşiminden kaynaklandığını POWER'ın kendi betiğiyle doğruladım (Bölüm 6.4). Orijinal veri hattında istemcilerin görev sıraları karışıkken Cora'da AA 64.07 (yerel modeller) / 57.49 (global model) çıktı; sıra tüm istemcilerde aynı yapılınca yeni görev doğrulukları 0'a düştü.
* **FedFST**, sentetik tekrar ve düşük frekanslı damıtmayla eski görevleri koruyor. KD aşamasındaki doğrulama tabanlı model seçimi, eski ve yeni görev doğrulukları arasında bir denge noktası seçiyor. Bununla birlikte iki noktaya dikkat etmek gerek. Birincisi, istemciler geçmiş görevlerin gerçek verisini kullanıyor (Bölüm 1.5). İkincisi, sonuçlar RNG'ye çok duyarlı: yerel testte thread'li orijinal `main.py` aynı seed'le iki kez çalıştırıldığında Cora'da AA 38.97 ve 80.63 verdi (Bölüm 6.3). Tek seed'li karşılaştırmada FedFST'nin AA'sı bu belirsizlikle birlikte okunmalı.

### 10.3 POWER makalesindeki sonuçlarla kıyas
POWER makalesinde ([arXiv:2411.18919](https://arxiv.org/abs/2411.18919)) raporlanan AM/FM değerleri: Cora 65.74 / 28.10, CiteSeer 54.47 / 28.94, Squirrel 40.27 / 35.24. Bu değerler bizim protokolümüzle birebir karşılaştırılamaz:
* **Bölme:** POWER'ın veri katmanı Cora ve CiteSeer'de 0.2/0.4/0.4, Squirrel'de 0.48/0.32/0.2 eğitim/doğrulama/test oranı kullanıyor (`data.py`). Biz FedFST'nin 0.6/0.2/0.2 bölmesini kullandık.
* **İstemciler ve bölümleme:** makalede Squirrel deneyleri 10 istemciyle yapılmış ve Louvain bölümlemesi OpenFGL'in (scikit-network) uygulamasıyla yapılıyor. Biz her veri setinde FedFST'nin `louvain_partitioner`'ı (python-louvain) ile 3 istemci kullandık.
* **Senaryo ve değerlendirme:** POWER'da görev sırası istemciden istemciye karışık ve değerlendirme varsayılan olarak yerel modelle yapılıyor. Makaledeki değerler 10 koşunun ortalaması (Ek A.3) ve keşif hiperparametreleri Optuna ile aranmış.

Yerel sağlama koşusunda POWER'ın orijinal betiği (tek koşu, `config.py` varsayılanları, CPU) Cora'da AA 64.07 / AF 8.14 verdi. AA, makaledeki 65.74'e yakın; AF ise farklı.

### 10.4 Kısıtlar
* **Tek seed ve görev başına 10 round.** FedFST'nin RNG duyarlılığı düşünüldüğünde, yöntemler arasındaki farklar istatistiksel olarak sınanmadı. CONFIG'deki `SEEDS` listesi genişletilerek deney tekrarlanabilir; süre seed sayısıyla doğrusal artar.
* **Minesweeper'da görev başına tek sınıf.** Yapay ve zor bir senaryo (10.1).
* **FedSPA'nın uyarlanması.** FedSPA bir FGL yöntemi; FGCL'ye ince ayarla uyarlandı. Ortak bütçe için optimizer değiştirildi: SGD lr 0.1 ve 200 round yerine Adam lr 0.01, wd 5e-4, görev başına 10 round × 3 adım. Her görevde komşuluk matrisleri, $S_{high}$ ve HBDA'nın homofili ağırlıkları görev alt-çizgisinden yeniden hesaplandı. FedSPA kendi omurgasını (ACM-SPA, 128 gizli birim, dropout 0.1) kullanıyor; FedFST ve POWER ise 2 katmanlı GAT (64 gizli birim, dropout 0.5) kullanıyor.
* **DGL → PyTorch.** Düşük dereceli çizgelerde bit düzeyinde aynı sonuç; yüksek dereceli düğümlerde toplama sırasından kaynaklanan, göreli olarak en çok ≈2·10⁻⁵'lik fark (Bölüm 6.2).
* **Senaryo farkları.** POWER ve FedFST'nin kendi senaryoları (görev sırası, bölme, değerlendirilen model) ortak protokol için değiştirildi. Yöntemler kendi en iyi koşullarında değil, aynı koşullarda karşılaştırıldı.
* **Tekrarlanabilirlik.** GPU'da GAT ve `index_add` atomik toplamalar kullandığı için Colab koşuları bit düzeyinde tekrarlanamaz; CPU'da tekrarlanabilir.

## 11. Kaynakça
1. H. Guo, Z. Tan, W. Huang, B. Yang, M. Ye. *FedFST: Mitigating Spectral Catastrophic Forgetting in Federated Graph Continual Learning.* SIGKDD 2026. Kod: https://github.com/YunQi572/FedFST (commit `d83fac0`).
2. Y. Zhu, M. Hu, D. Wu. *Federated Continual Graph Learning.* KDD 2025. https://arxiv.org/abs/2411.18919 ; DOI 10.1145/3711896.3736956. Kod ve teknik rapor: https://zenodo.org/records/15487655 (DOI 10.5281/zenodo.15487654).
3. Z. Tan, G. Wan, W. Huang, H. Li, G. Zhang, C. Yang, M. Ye. *FedSPA: Generalizable Federated Graph Learning under Homophily Heterogeneity.* CVPR 2025, s. 15464–15475. Kod: https://github.com/OakleyTan/FedSPA (commit `d2b3ddb`).
4. P. Veličković, G. Cucurull, A. Casanova, A. Romero, P. Liò, Y. Bengio. *Graph Attention Networks.* ICLR 2018.
5. M. Fey, J. E. Lenssen. *Fast Graph Representation Learning with PyTorch Geometric.* ICLR Workshop on Representation Learning on Graphs and Manifolds, 2019.
6. V. D. Blondel, J.-L. Guillaume, R. Lambiotte, E. Lefebvre. *Fast unfolding of communities in large networks.* J. Stat. Mech., 2008. Uygulama: python-louvain, https://github.com/taynaud/python-louvain.
7. B. McMahan, E. Moore, D. Ramage, S. Hampson, B. Agüera y Arcas. *Communication-Efficient Learning of Deep Networks from Decentralized Data.* AISTATS 2017 (FedAvg).
8. P. Sen, G. Namata, M. Bilgic, L. Getoor, B. Gallagher, T. Eliassi-Rad. *Collective Classification in Network Data.* AI Magazine, 2008 (Cora, CiteSeer).
9. Z. Yang, W. Cohen, R. Salakhutdinov. *Revisiting Semi-Supervised Learning with Graph Embeddings.* ICML 2016 (Planetoid sürümü).
10. H. Pei, B. Wei, K. C.-C. Chang, Y. Lei, B. Yang. *Geom-GCN: Geometric Graph Convolutional Networks.* ICLR 2020 (Actor ve Squirrel'in kullanılan ön işlenmiş sürümleri).
11. B. Rozemberczki, C. Allen, R. Sarkar. *Multi-scale Attributed Node Embedding.* Journal of Complex Networks, 2021 (Squirrel).
12. O. Platonov, D. Kuznedelev, M. Diskin, A. Babenko, L. Prokhorenkova. *A critical look at the evaluation of GNNs under heterophily: Are we really making progress?* ICLR 2023 (Minesweeper).
13. S. Luan, C. Hua, Q. Lu, J. Zhu, M. Zhao, S. Zhang, X.-W. Chang, D. Precup. *Revisiting Heterophily For Graph Neural Networks.* NeurIPS 2022 (ACM-GCN).
14. L. Zhu, Z. Liu, S. Han. *Deep Leakage from Gradients.* NeurIPS 2019 (gradyan ters çevirme).
15. F. Zhou, C. Cao. *Overcoming Catastrophic Forgetting in Graph Neural Networks with Experience Replay.* AAAI 2021 (ER-GNN, kapsama maksimizasyonu).
16. X. Li vd. *OpenFGL: A Comprehensive Benchmark for Federated Graph Learning.* arXiv:2408.16288. Kod: https://github.com/xkLi-Allen/OpenFGL (POWER'ın veri katmanı).
17. M. Wang vd. *Deep Graph Library: A Graph-Centric, Highly-Performant Package for Graph Neural Networks.* arXiv:1909.01315 (DGL).